# Chapter 11 Singular Value Decomposition and Its Applications

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asj252/linear-algebra-with-python/blob/main/en/EN_Chap11_Singular_Value_Decomposition.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/asj252/linear-algebra-with-python/HEAD?labpath=en%2FEN_Chap11_Singular_Value_Decomposition.ipynb)

The history of the singular value decomposition (SVD) shows how theory, computation, and applications push one another forward.

In 1873 the Italian mathematician Eugenio Beltrami studied real bilinear forms; in 1874 the French mathematician Camille Jordan obtained related results. Erhard Schmidt studied integral operators and approximation problems in 1907, Carl Eckart and Gale Young established the result on the best low-rank approximation of a matrix in 1936, and Mirsky extended the approximation theory to unitarily invariant norms in 1960. These results dealt separately with the existence of the decomposition, with operators, and with best approximation, and together they form a theory that developed step by step; without historical evidence, one should not claim that these scholars were unaware of each other or that their results were repeatedly forgotten. [History and original sources](https://epubs.siam.org/doi/10.1137/1035134)

Beyond the theory, computational methods are just as essential. The SVD can be built by finding the eigenvalues of $\mathbf{A}^{*}\mathbf{A}$, but forming this Gram matrix directly may lose relative accuracy in the small singular values. For a matrix of full column rank, $\kappa_2(\mathbf{A}^{*}\mathbf{A})=\kappa_2(\mathbf{A})^2$; this is a matter of numerical conditioning, and it does not mean that every input error is amplified quadratically.

Algorithms did not first appear in 1970. Jacobi-type methods already existed in the 1950s; Golub and Kahan developed the bidiagonalization approach in 1965, and Golub and Reinsch went on in 1970 to give algorithms for the SVD and for least squares. These methods avoid making the explicit formation of the Gram matrix a required step, and they laid an important foundation for reliable numerical computation. [Hestenes 1958](https://epubs.siam.org/doi/10.1137/0106005), [Golub–Kahan 1965](https://epubs.siam.org/doi/10.1137/0702016)

The teaching power of the SVD comes from a clear geometric picture: once orthonormal bases are chosen for the input and the output, $\mathbf{A}\mathbf{v}_i=\sigma_i\mathbf{u}_i$ ($i<r$), and the remaining input basis vectors fall into the null space. The SVD provides compatible orthogonal coordinates for the four fundamental subspaces; the nonzero singular values give the stretching factors—the SVD does not turn every vector into another unit vector.

Applications have also carried these ideas beyond the classroom. The Netflix Prize spurred regularized low-rank factor models for ratings with missing entries; these models share the low-rank viewpoint of the SVD, but they do not fill in the missing values arbitrarily and then run an ordinary SVD, nor are they the same as exact nuclear-norm completion. PCA, the pseudoinverse, and data compression have made singular values a common language of data analysis.

This line of development reminds us that **theoretical correctness, computational feasibility, and the needs of applications** together determine how much power a mathematical tool can exert.

### Chapter Structure and Learning Objectives

This chapter follows the path of theory, computation, and applications, breaking the SVD into material that can be understood section by section.

**§11.1** first answers the most fundamental question: what exactly does this theorem say, and why does it hold for every matrix? We rigorously prove the existence of the decomposition $\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{*}$ and examine its meaning again and again from three angles: algebra, subspaces, and geometry. $\mathbf{A}^{*}\mathbf{A}$ is the bridge connecting the SVD with the spectral theorem of Chapter 9; under the SVD, each of the four fundamental subspaces receives an orthonormal basis; and the geometric picture of "rotation—stretch—rotation" makes the action of a matrix clear at a glance. By the end of this section you will hold a key that opens the door to every application that follows.

In **§11.2** this key faces its first test: in what sense is the low-rank approximation given by the truncated SVD "the best"? This is precisely the question Eckart and Young studied in 1936. Three matrix norms—the spectral norm, the Frobenius norm, and the nuclear norm—are all completely determined by the singular values, and the theorem's answer is clean and decisive: among all matrices of the same rank bound, the truncated SVD has the smallest error, and the discarded singular values tell exactly how much is lost. This is the theoretical peak of the chapter, and the three application sections that follow are all its echoes.

**§11.3** brings this optimality into the setting of data science. The "directions that explain the most variation in the data" sought by principal component analysis (PCA) are, in the language of the SVD, the right singular vectors; numerically, performing the SVD directly on the data matrix rather than on the covariance matrix avoids the very problem Golub worked to solve: the condition number being quietly squared during the computation. A complete demonstration on the Iris dataset takes this procedure from formulas into practice.

**§11.4** changes the question from "finding directions" to "solving equations": rectangular or singular matrices have no inverse, but the Moore–Penrose pseudoinverse, through the SVD, gracefully extends the meaning of "inversion" and provides the geometrically most natural solution to least-squares problems. As you read this section, heed the warning that "the reciprocals of small singular values blow up"—it leads to ridge regression regularization, and it also gives you your first reliable tool for handling any ill-conditioned linear problem in the future.

**§11.5** then carries the SVD to an unexpected shore. Quantum entanglement—what Einstein called "spooky action at a distance"—is, mathematically, exactly a singular value problem for the coefficient matrix of a bipartite pure state. The decomposition Schmidt built for integral operators in 1907 became, after his death, a core language of quantum information theory, which is probably one of the most surprising "late applications" in the history of mathematics. This section is a mathematical deepening of the quantum mechanics framework of Chapter 10, and also the book's final statement of the theme of "the deep unity of linear algebra and the physical world."

Having read the whole chapter, you will find that singular values are not isolated numbers but a ranking of the "signal strength" of a matrix in each direction—truncate them, and you are compressing; invert them, and you are solving; measure them, and you are quantifying entanglement. The same set of numbers ties together three worlds—mathematics, data, and physics—and also the long development of theory and applications from 1873 to the present day.

---

## 11.1 Definition and Proof of the Singular Value Decomposition

### From Diagonalization to the SVD: Breaking the Square-Matrix Barrier

In Chapter 8 we witnessed the power of diagonalization: for a diagonalizable square matrix $\mathbf{A} \in \mathbb{K}^{n \times n}$, there exists an invertible matrix $\mathbf{P}$ such that

$$\mathbf{A} = \mathbf{P} \boldsymbol{\Lambda} \mathbf{P}^{-1}$$

where $\boldsymbol{\Lambda}$ is a diagonal matrix whose diagonal entries are the eigenvalues of $\mathbf{A}$. This decomposition reveals the **essential structure** of the matrix—in a suitable basis, the linear transformation is nothing more than independent stretching along each direction.

Chapter 9 went further: for a **symmetric matrix** $\mathbf{A} = \mathbf{A}^{\top}$ (real field) or a **Hermitian matrix** $\mathbf{A} = \mathbf{A}^{*}$ (complex field), the situation is even better—we can choose an **orthogonal matrix** $\mathbf{Q}$ (or a **unitary matrix** $\mathbf{U}$) to diagonalize it:

$$\mathbf{A} = \mathbf{Q} \boldsymbol{\Lambda} \mathbf{Q}^{\top} \quad \text{or} \quad \mathbf{A} = \mathbf{U} \boldsymbol{\Lambda} \mathbf{U}^{*}$$

This is the **spectral theorem**—the **orthogonal diagonalization** of symmetric/Hermitian matrices. The beauty of orthogonal diagonalization is that the transformations $\mathbf{Q}$ and $\mathbf{Q}^{\top}$ are both geometric **rotations** (they preserve lengths and angles), so the whole linear transformation can be understood as "rotate—stretch—rotate back."

**But the real world presents a harsh fact: the vast majority of important matrices are neither square nor symmetric.**

Consider the following practical scenarios:
- **Image matrices**: for a $1920 \times 1080$ picture, the matrix of pixel values is **rectangular**
- **Data matrices**: recording the ratings of $1000$ customers on $50$ products gives a $1000 \times 50$ matrix, far from square
- **Document-term matrices**: $10000$ documents containing $5000$ terms give a $10000 \times 5000$ matrix

Clearly these matrices cannot use the eigendecomposition directly. We need a **more general decomposition** that can handle an arbitrary $m \times n$ matrix.

### The Core Idea of the SVD: Starting from $\mathbf{A}^{*}\mathbf{A}$

The key observation is this: even if $\mathbf{A}$ is not square, $\mathbf{A}^{*}\mathbf{A}$ and $\mathbf{A}\mathbf{A}^{*}$ **are symmetric/Hermitian square matrices**!

Let $\mathbf{A} \in \mathbb{C}^{m \times n}$. Then:
- $\mathbf{A}^{*}\mathbf{A} \in \mathbb{C}^{n \times n}$ and $(\mathbf{A}^{*}\mathbf{A})^{*} = \mathbf{A}^{*}\mathbf{A}$ (a Hermitian matrix)
- $\mathbf{A}\mathbf{A}^{*} \in \mathbb{C}^{m \times m}$ and $(\mathbf{A}\mathbf{A}^{*})^{*} = \mathbf{A}\mathbf{A}^{*}$ (a Hermitian matrix)

More importantly, these two matrices are also **positive semidefinite** (recall Chapter 9): for any vector $\mathbf{x}$,

$$⟨  \mathbf{x}| \mathbf{A}^{*}\mathbf{A} | \mathbf{x} ⟩ = ⟨  \mathbf{A}\mathbf{x}| \mathbf{A}\mathbf{x} ⟩ = \|\mathbf{A}\mathbf{x}\|^2 \geq 0$$

By the spectral theorem of Chapter 9, $\mathbf{A}^{*}\mathbf{A}$ and $\mathbf{A}\mathbf{A}^{*}$ can both be **orthogonally diagonalized**, and **all their eigenvalues are nonnegative real numbers**. This is the theoretical foundation of the SVD.

**Historical note.** The work on bilinear forms by the Italian mathematician Beltrami in 1873 and the French mathematician Camille Jordan in 1874 is the early source of the SVD. Schmidt studied integral operators and approximation theory in 1907; Eckart and Young gave the result on the best low-rank approximation of a matrix in 1936, and Mirsky extended it to unitarily invariant norms in 1960. These works developed, respectively, the theory of the decomposition, of operators, and of approximation; they should not be lumped together as a single theorem that was repeatedly forgotten.

---

### 11.1.1 Definition of the SVD: The Real and Complex Cases

:::{prf:theorem} Existence of the Singular Value Decomposition
:label: thm-svd-existence

Let $\mathbf{A} \in \mathbb{C}^{m \times n}$ have rank $r = \text{rank}(\mathbf{A})$. Then there exist **unitary matrices** $\mathbf{U} \in \mathbb{C}^{m \times m}$ and $\mathbf{V} \in \mathbb{C}^{n \times n}$, and a **nonnegative real diagonal matrix** $\boldsymbol{\Sigma} \in \mathbb{R}^{m \times n}$, such that

$$\mathbf{A} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^{*}$$

where the diagonal entries of $\boldsymbol{\Sigma}$ satisfy

$$\sigma_0 \geq \sigma_1 \geq \cdots \geq \sigma_{r-1} > 0, \quad \sigma_r = \sigma_{r+1} = \cdots = 0$$

We call $\sigma_0, \sigma_1, \ldots, \sigma_{r-1}$ the **singular values** of $\mathbf{A}$, the columns $\mathbf{u}_0, \mathbf{u}_1, \ldots, \mathbf{u}_{m-1}$ of $\mathbf{U}$ the **left singular vectors**, and the columns $\mathbf{v}_0, \mathbf{v}_1, \ldots, \mathbf{v}_{n-1}$ of $\mathbf{V}$ the **right singular vectors**.

**The real case.** If $\mathbf{A} \in \mathbb{R}^{m \times n}$, then $\mathbf{U} \in \mathbb{R}^{m \times m}$ and $\mathbf{V} \in \mathbb{R}^{n \times n}$ are both **orthogonal matrices**, that is, $\mathbf{U}^{\top} = \mathbf{U}^{*}$ and $\mathbf{V}^{\top} = \mathbf{V}^{*}$, and the SVD can be written as

$$\mathbf{A} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^{\top}$$
:::


:::{prf:proof} A Constructive Proof of Existence
:label: prf-svd-existence


We prove the existence of the SVD by explicit construction. The proof proceeds in four steps:

1. **Spectral decomposition of $\mathbf{A}^{*}\mathbf{A}$**

Since $\mathbf{A}^{*}\mathbf{A} \in \mathbb{C}^{n \times n}$ is a Hermitian positive semidefinite matrix, by the spectral theorem of Chapter 9 there exist a unitary matrix $\mathbf{V} \in \mathbb{C}^{n \times n}$ and a nonnegative real diagonal matrix $\boldsymbol{\Lambda} = \text{diag}(\lambda_0, \lambda_1, \ldots, \lambda_{n-1})$ (with $\lambda_0 \geq \lambda_1 \geq \cdots \geq \lambda_{n-1} \geq 0$) such that

$$\mathbf{A}^{*}\mathbf{A} = \mathbf{V} \boldsymbol{\Lambda} \mathbf{V}^{*}$$

We define the **singular values** as the square roots of these eigenvalues:

$$\sigma_i := \sqrt{\lambda_i}, \quad i = 0, 1, \ldots, n-1$$

Let $\mathbf{A}$ have rank $r$. Then the first $r$ eigenvalues $\lambda_0, \ldots, \lambda_{r-1}$ are positive and the rest are zero. Hence the first $r$ singular values satisfy $\sigma_0, \ldots, \sigma_{r-1} > 0$, and the rest are $\sigma_r = \cdots = \sigma_{n-1} = 0$.

2. **Constructing the left singular vectors**

Let the columns of $\mathbf{V}$ be $\mathbf{v}_0, \mathbf{v}_1, \ldots, \mathbf{v}_{n-1}$ (that is, $\mathbf{V} = [\mathbf{v}_0\ \mathbf{v}_1\ \cdots\ \mathbf{v}_{n-1}]$). Then $\{\mathbf{v}_i\}$ is an orthonormal basis of $\mathbb{C}^n$, and

$$\mathbf{A}^{*}\mathbf{A} \mathbf{v}_i = \lambda_i \mathbf{v}_i = \sigma_i^2 \mathbf{v}_i$$

For the right singular vectors $\mathbf{v}_0, \ldots, \mathbf{v}_{r-1}$ corresponding to the first $r$ nonzero singular values, define

$$\mathbf{u}_i := \frac{1}{\sigma_i} \mathbf{A} \mathbf{v}_i, \quad i = 0, 1, \ldots, r-1$$

**Key verification.** The vectors $\mathbf{u}_0, \ldots, \mathbf{u}_{r-1}$ form an **orthonormal system** in $\mathbb{C}^m$:

$$⟨  \mathbf{u}_i | \mathbf{u}_j ⟩ = ⟨  \frac{\mathbf{A}\mathbf{v}_i}{\sigma_i} \bigg| \frac{\mathbf{A}\mathbf{v}_j}{\sigma_j} ⟩ = \frac{1}{\sigma_i \sigma_j} ⟨  \mathbf{v}_i | \mathbf{A}^{*}\mathbf{A} | \mathbf{v}_j ⟩ = \frac{\sigma_j^2}{\sigma_i \sigma_j} ⟨  \mathbf{v}_i | \mathbf{v}_j ⟩ = \delta_{ij}$$

where $\delta_{ij}$ is the Kronecker delta.

3. **Extending the left singular vectors to an orthonormal basis of $\mathbb{C}^m$**

We now have $r$ orthonormal vectors $\mathbf{u}_0, \ldots, \mathbf{u}_{r-1}$. Using Gram-Schmidt orthogonalization (Chapter 9), we can extend them to a full orthonormal basis $\mathbf{u}_0, \mathbf{u}_1, \ldots, \mathbf{u}_{m-1}$ of $\mathbb{C}^m$.

Define the unitary matrix

$$\mathbf{U} := [\mathbf{u}_0 \; \mathbf{u}_1 \; \cdots \; \mathbf{u}_{m-1}]\in \mathbb{C}^{m \times m}$$

4. **Verifying the SVD**

We now verify $\mathbf{A} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^{*}$. Note that

$$\mathbf{A} \mathbf{v}_i = \begin{cases}
\sigma_i \mathbf{u}_i, & i = 0, 1, \ldots, r-1 \\
\mathbf{0}, & i = r, r+1, \ldots, n-1
\end{cases}$$

This corresponds exactly to the definition of $\mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^{*}$:

$$(\mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^{*}) \mathbf{v}_i = \mathbf{U} \boldsymbol{\Sigma} \mathbf{e}_i = \begin{cases}\sigma_i\mathbf{u}_i,&0\leq i<r\\\mathbf{0}\in\mathbb{C}^m,&r\leq i<n\end{cases}=\mathbf{A}\mathbf{v}_i$$

Since $\{\mathbf{v}_i\}$ is a basis of $\mathbb{C}^n$, it follows that $\underline{\mathbf{A} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^{*}}$.
:::


```{prf:remark} Two Equivalent Constructions
:label: rmk-svd-from-AtA-or-AAt

1. The proof above starts from $\mathbf{A}^{*}\mathbf{A}$ to obtain the right singular vectors $\mathbf{V}$, and then constructs the left singular vectors $\mathbf{U}$
2. One can also start from $\mathbf{A}\mathbf{A}^{*}$ to obtain the left singular vectors $\mathbf{U}$ first, and then construct the right singular vectors $\mathbf{V}$

The two methods lead to the same result, but their computational efficiency differs:
- If $m \gg n$ (as is usual for data matrices, where the number of samples far exceeds the number of features), computing the eigendecomposition of $\mathbf{A}^{*}\mathbf{A} \in \mathbb{C}^{n \times n}$ is more efficient
- If $n \gg m$, computing the eigendecomposition of $\mathbf{A}\mathbf{A}^{*} \in \mathbb{C}^{m \times m}$ is more efficient

Practical numerical algorithms (such as LAPACK's `DGESDD`) use more efficient methods: rather than computing $\mathbf{A}^{*}\mathbf{A}$ directly, they use **bidiagonalization** to avoid worsening the condition number.
```


**Uniqueness theorem.** For a given matrix $\mathbf{A}$, its **singular values** are uniquely determined (counting multiplicity). The **singular vectors, however, are not unique**.

:::{prf:theorem} Uniqueness of the SVD
:label: thm-svd-uniqueness

Let $\mathbf{A} \in \mathbb{C}^{m \times n}$ have rank $r$. Then:
1. **The singular values are unique**: $\sigma_0 \geq \sigma_1 \geq \cdots \geq \sigma_{r-1} > 0$ are uniquely determined
2. **The singular vectors are not unique**:
   - If $\sigma_i$ has multiplicity $k$ (that is, $\sigma_i = \sigma_{i+1} = \cdots = \sigma_{i+k-1}$), then the subspace spanned by the corresponding $k$ right singular vectors $\mathbf{v}_i, \mathbf{v}_{i+1}, \ldots, \mathbf{v}_{i+k-1}$ is unique, but the choice of the individual vectors is not (any unitary transformation may be applied)
   - Likewise, the subspace spanned by the corresponding left singular vectors $\mathbf{u}_i, \mathbf{u}_{i+1}, \ldots, \mathbf{u}_{i+k-1}$ is unique, but the individual vectors are not
3. **The signs are not unique**: even when a singular value is simple (multiplicity 1), we can still change the signs (or phase factors $e^{i\theta}$) of $\mathbf{u}_i$ and $\mathbf{v}_i$ simultaneously and obtain a different SVD

:::

```{prf:proof}
1. **Uniqueness of the singular values**: by the construction, $\sigma_i^2 = \lambda_i$ is an eigenvalue of $\mathbf{A}^{*}\mathbf{A}$. The eigenvalues of a matrix are **invariants** (unchanged under similarity transformations), so the singular values are uniquely determined.

2. **Non-uniqueness of the singular vectors**: consider the eigenspace $E_{\lambda_i} = \ker(\mathbf{A}^{*}\mathbf{A} - \lambda_i \mathbf{I})$ corresponding to the eigenvalue $\lambda_i$. The spectral theorem guarantees that we can choose an orthonormal basis in $E_{\lambda_i}$, but this choice is not unique—any unitary transformation on $E_{\lambda_i}$ gives a new orthonormal basis.

3. **Non-uniqueness of signs/phases**: we may take diagonal unitary matrices $\mathbf{D}_U\in\mathbb{C}^{m\times m}$ and $\mathbf{D}_V\in\mathbb{C}^{n\times n}$ that use the same phases in the first $r$ positions, so that $\mathbf{D}_U\boldsymbol{\Sigma}\mathbf{D}_V^*=\boldsymbol{\Sigma}$. Then $(\mathbf{U}\mathbf{D}_U,\boldsymbol{\Sigma},\mathbf{V}\mathbf{D}_V)$ is still an SVD; the phases in the null-space part can be chosen independently.

Indexing convention for singular values: for $m,n\geq1$, the full list has $q=\min(m,n)$ values $\sigma_0\geq\cdots\geq\sigma_{q-1}\geq0$; when the rank is $r$, only the first $r$ are positive and the rest are padded with zeros. All singular values of the zero matrix are zero, and an empty sum over positive singular values is taken to be zero.
```


:::{prf:remark} A Sketch of the SVD Construction
:label: rmk-svd-construction

The proof of existence of the SVD has three steps, each with a clear geometric meaning:

1. **Building the right singular vectors**: $\mathbf{A}^{*}\mathbf{A}$ is a positive semidefinite Hermitian matrix (because $\langle \mathbf{x}, \mathbf{A}^{*}\mathbf{A}\mathbf{x} \rangle = \|\mathbf{A}\mathbf{x}\|^2 \geq 0$),
   so by the spectral theorem of Chapter 9 it can be orthogonally diagonalized: $\mathbf{A}^{*}\mathbf{A} = \mathbf{V}\boldsymbol{\Lambda}\mathbf{V}^{*}$,
   with eigenvalues $\lambda_i \geq 0$. Let $\sigma_i := \sqrt{\lambda_i}$;
   then $\sigma_i \geq 0$ and $\|\mathbf{A}\mathbf{v}_i\|^2 = \mathbf{v}_i^{*}\mathbf{A}^{*}\mathbf{A}\mathbf{v}_i = \sigma_i^2$.

2. **Building the left singular vectors (the range part)**: for each $\sigma_i > 0$, let
   $\mathbf{u}_i := \dfrac{\mathbf{A}\mathbf{v}_i}{\sigma_i}$;
   one can verify that $\{\mathbf{u}_i\}$ is an orthonormal set (since $\langle \mathbf{u}_i, \mathbf{u}_j \rangle = \frac{1}{\sigma_i\sigma_j}\langle \mathbf{A}\mathbf{v}_i, \mathbf{A}\mathbf{v}_j \rangle = \frac{\sigma_j}{\sigma_i}\delta_{ij}$).
   These $r$ vectors span $\text{Col}(\mathbf{A})$.

3. **Extending to a full unitary basis**: $\{\mathbf{u}_0, \ldots, \mathbf{u}_{r-1}\}$ has only $r$ vectors;
   the Gram-Schmidt process (§9.1.3) extends it to a full orthonormal basis of $\mathbb{C}^m$,
   and the added vectors $\mathbf{u}_r, \ldots, \mathbf{u}_{m-1}$ span $\ker(\mathbf{A}^{*})$.
   Then $\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{*}$ can be verified directly.
:::

#### Three Forms of the SVD

Depending on practical needs, the SVD comes in three different forms:

**1. Full SVD**

This is the most standard form, and the direct statement of the theorem above:

$$\mathbf{A} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^{*}$$

where $\mathbf{U} \in \mathbb{C}^{m \times m}$, $\boldsymbol{\Sigma} \in \mathbb{R}^{m \times n}$, and $\mathbf{V} \in \mathbb{C}^{n \times n}$ are all full matrices.

**Matrix dimensions** (for $m=4, n=3$):

$$\underbrace{\mathbf{A}}_{4 \times 3} = \underbrace{\mathbf{U}}_{4 \times 4} \underbrace{\boldsymbol{\Sigma}}_{4 \times 3} \underbrace{\mathbf{V}^{*}}_{3 \times 3}$$

$$\boldsymbol{\Sigma} = \begin{bmatrix}
\sigma_0 & 0 & 0 \\
0 & \sigma_1 & 0 \\
0 & 0 & \sigma_2 \\
0 & 0 & 0
\end{bmatrix}$$

**2. Reduced (Thin) SVD**

When $m > n$, the lower part of $\boldsymbol{\Sigma}$ has many all-zero rows, and the corresponding last $m-n$ columns of $\mathbf{U}$ do not actually take part in the matrix multiplication. We can omit these redundant parts:

$$\mathbf{A} = \mathbf{U}_r \boldsymbol{\Sigma}_r \mathbf{V}^{*}$$

where:
- $\mathbf{U}_r \in \mathbb{C}^{m \times n}$ keeps only the first $n$ columns
- $\boldsymbol{\Sigma}_r \in \mathbb{R}^{n \times n}$ is a **square matrix**
- $\mathbf{V} \in \mathbb{C}^{n \times n}$ stays the same

**Matrix dimensions** (for $m=4, n=3$):

$$\underbrace{\mathbf{A}}_{4 \times 3} = \underbrace{\mathbf{U}_r}_{4 \times 3} \underbrace{\boldsymbol{\Sigma}_r}_{3 \times 3} \underbrace{\mathbf{V}^{*}}_{3 \times 3}$$

$$\mathbf{U}_r =  [\mathbf{u}_0 \; \mathbf{u}_1 \;  \mathbf{u}_{2}], \quad \boldsymbol{\Sigma}_r = \begin{bmatrix}
\sigma_0 & 0 & 0 \\
0 & \sigma_1 & 0 \\
0 & 0 & \sigma_2
\end{bmatrix}$$

```{note}
The reduced SVD is a common form for **saving storage**. For example, when handling a $1000 \times 100$ data matrix:
- the full SVD must store $\mathbf{U} \in \mathbb{R}^{1000 \times 1000}$ (about 1 million numbers)
- the reduced SVD only needs to store $\mathbf{U}_r \in \mathbb{R}^{1000 \times 100}$ (only 100,000 numbers)
```

**3. Truncated SVD**

When the rank $r$ of the matrix is much smaller than $\min(m,n)$, or when we care only about the $k$ largest singular values, we can truncate further:

$$\mathbf{A}_k := \mathbf{U}_k \boldsymbol{\Sigma}_k \mathbf{V}_k^{*} = \sum_{i=0}^{k-1} \sigma_i \mathbf{u}_i \mathbf{v}_i^{*}$$

where:
- $\mathbf{U}_k \in \mathbb{C}^{m \times k}$ contains only the first $k$ left singular vectors
- $\boldsymbol{\Sigma}_k \in \mathbb{R}^{k \times k}$ contains only the first $k$ singular values
- $\mathbf{V}_k \in \mathbb{C}^{n \times k}$ contains only the first $k$ right singular vectors

**Matrix dimensions** (for $m=4, n=3, k=2$):

$$\underbrace{\mathbf{A}_2}_{4 \times 3} = \underbrace{\mathbf{U}_2}_{4 \times 2} \underbrace{\boldsymbol{\Sigma}_2}_{2 \times 2} \underbrace{\mathbf{V}_2^{*}}_{2 \times 3}$$

This gives a **rank-$k$ approximation** of $\mathbf{A}$, which is the basis of the Eckart–Young theorem (§11.2).

```{important}
:label: note-truncated-svd-applications

**Key applications of the truncated SVD**:
1. **Image compression**: keeping the first $k$ terms greatly reduces storage while preserving image quality; this idea is widely used in image processing
2. **Denoising**: image content usually corresponds to large singular values and noise to small ones, so truncation can remove noise
3. **Latent semantic analysis** (LSA): in text mining, the truncated SVD can uncover hidden semantic structure
4. **Recommender systems**: many methods in collaborative filtering systems, such as those from the Netflix Prize competition, are based on low-rank approximation of matrices
```

#### Outer-Product Expansion (Dyadic Decomposition)

Rewriting the truncated SVD as a **sum of outer products** gives a very illuminating decomposition of the matrix:

$$\mathbf{A} = \sum_{i=0}^{r-1} \sigma_i \mathbf{u}_i \mathbf{v}_i^{*} = \sum_{i=0}^{r-1} \sigma_i | \mathbf{u}_i ⟩ ⟨  \mathbf{v}_i | $$

where $\mathbf{u}_i \mathbf{v}_i^{*}$ is a **rank-one matrix** (also called a dyad or an outer product).

**Geometric interpretation.** Every matrix $\mathbf{A}$ can be decomposed into a weighted sum of $r$ rank-one matrices, with the singular values $\sigma_i$ as weights. These rank-one components are arranged in decreasing order of importance (size of the singular value), so:
- $\sigma_0 \mathbf{u}_0 \mathbf{v}_0^{*}$ is the "most important" component of $\mathbf{A}$
- $\sigma_1 \mathbf{u}_1 \mathbf{v}_1^{*}$ is the next most important component
- and so on

:::{prf:example} SVD of a $2 \times 2$ Matrix

Consider the matrix

$$\mathbf{A} = \begin{bmatrix} 3 & 0 \\ 4 & 5 \end{bmatrix}$$

**Step 1.** Compute $\mathbf{A}^{\top}\mathbf{A}$:

$$\mathbf{A}^{\top}\mathbf{A} = \begin{bmatrix} 3 & 4 \\ 0 & 5 \end{bmatrix} \begin{bmatrix} 3 & 0 \\ 4 & 5 \end{bmatrix} = \begin{bmatrix} 25 & 20 \\ 20 & 25 \end{bmatrix}$$

**Step 2.** Find the eigenvalues of $\mathbf{A}^{\top}\mathbf{A}$:

The characteristic polynomial is

$$\det(\mathbf{A}^{\top}\mathbf{A} - \lambda \mathbf{I}) = \det\begin{bmatrix} 25-\lambda & 20 \\ 20 & 25-\lambda \end{bmatrix} = (25-\lambda)^2 - 400 = \lambda^2 - 50\lambda + 225$$

Solving gives $\lambda_0 = 45$ and $\lambda_1 = 5$, so the singular values are

$$\sigma_0 = \sqrt{45} = 3\sqrt{5}, \quad \sigma_1 = \sqrt{5}$$

**Step 3.** Find the corresponding eigenvectors (the right singular vectors):

For $\lambda_0 = 45$:

$$(\mathbf{A}^{\top}\mathbf{A} - 45\mathbf{I})\mathbf{v}_0 = \begin{bmatrix} -20 & 20 \\ 20 & -20 \end{bmatrix}\mathbf{v}_0 = \mathbf{0} \implies \mathbf{v}_0 = \frac{1}{\sqrt{2}}\begin{bmatrix} 1 \\ 1 \end{bmatrix}$$

For $\lambda_1 = 5$:

$$(\mathbf{A}^{\top}\mathbf{A} - 5\mathbf{I})\mathbf{v}_1 = \begin{bmatrix} 20 & 20 \\ 20 & 20 \end{bmatrix}\mathbf{v}_1 = \mathbf{0} \implies \mathbf{v}_1 = \frac{1}{\sqrt{2}}\begin{bmatrix} 1 \\ -1 \end{bmatrix}$$

Therefore

$$\mathbf{V} = \frac{1}{\sqrt{2}}\begin{bmatrix} 1 & 1 \\ 1 & -1 \end{bmatrix}$$

**Step 4.** Compute the left singular vectors:

$$\mathbf{u}_0 = \frac{1}{\sigma_0}\mathbf{A}\mathbf{v}_0 = \frac{1}{3\sqrt{5}}\begin{bmatrix} 3 & 0 \\ 4 & 5 \end{bmatrix} \frac{1}{\sqrt{2}}\begin{bmatrix} 1 \\ 1 \end{bmatrix} = \frac{1}{3\sqrt{10}}\begin{bmatrix} 3 \\ 9 \end{bmatrix} = \frac{1}{\sqrt{10}}\begin{bmatrix} 1 \\ 3 \end{bmatrix}$$

$$\mathbf{u}_1 = \frac{1}{\sigma_1}\mathbf{A}\mathbf{v}_1 = \frac{1}{\sqrt{5}}\begin{bmatrix} 3 & 0 \\ 4 & 5 \end{bmatrix} \frac{1}{\sqrt{2}}\begin{bmatrix} 1 \\ -1 \end{bmatrix} = \frac{1}{\sqrt{10}}\begin{bmatrix} 3 \\ -1 \end{bmatrix}$$

Therefore

$$\mathbf{U} = \frac{1}{\sqrt{10}}\begin{bmatrix} 1 & 3 \\ 3 & -1 \end{bmatrix}$$

**Step 5.** Verify the SVD:

$$\mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{\top} = \frac{1}{\sqrt{10}}\begin{bmatrix} 1 & 3 \\ 3 & -1 \end{bmatrix} \begin{bmatrix} 3\sqrt{5} & 0 \\ 0 & \sqrt{5} \end{bmatrix} \frac{1}{\sqrt{2}}\begin{bmatrix} 1 & 1 \\ 1 & -1 \end{bmatrix}$$

$$= \frac{1}{\sqrt{10}}\begin{bmatrix} 3\sqrt{5} & 3\sqrt{5} \\ 9\sqrt{5} & -\sqrt{5} \end{bmatrix} \frac{1}{\sqrt{2}}\begin{bmatrix} 1 & 1 \\ 1 & -1 \end{bmatrix} = \begin{bmatrix} 3 & 0 \\ 4 & 5 \end{bmatrix} = \mathbf{A} \quad \checkmark$$

**Outer-product expansion**:

$$\mathbf{A} = \sigma_0 \mathbf{u}_0 \mathbf{v}_0^{\top} + \sigma_1 \mathbf{u}_1 \mathbf{v}_1^{\top}$$

$$= 3\sqrt{5} \cdot \frac{1}{\sqrt{10}}\begin{bmatrix} 1 \\ 3 \end{bmatrix} \frac{1}{\sqrt{2}}\begin{bmatrix} 1 & 1 \end{bmatrix} + \sqrt{5} \cdot \frac{1}{\sqrt{10}}\begin{bmatrix} 3 \\ -1 \end{bmatrix} \frac{1}{\sqrt{2}}\begin{bmatrix} 1 & -1 \end{bmatrix}$$

$$= \frac{3}{2}\begin{bmatrix} 1 & 1 \\ 3 & 3 \end{bmatrix} + \frac{1}{2}\begin{bmatrix} 3 & -3 \\ -1 & 1 \end{bmatrix} = \begin{bmatrix} 3 & 0 \\ 4 & 5 \end{bmatrix}$$
:::

:::{prf:example} A Rank-One Matrix
Consider the rank-one matrix

$$\mathbf{A} = \begin{bmatrix} 2 & 4 \\ 1 & 2 \\ 3 & 6 \end{bmatrix} = \begin{bmatrix} 2 \\ 1 \\ 3 \end{bmatrix} \begin{bmatrix} 1 & 2 \end{bmatrix}$$

This matrix clearly has rank 1 (column 1 is a multiple of column 0). Let us compute its SVD:

**Step 1.** Compute $\mathbf{A}^{\top}\mathbf{A}$:

$$\mathbf{A}^{\top}\mathbf{A} = \begin{bmatrix} 1 & 2 \end{bmatrix}^{\top} \begin{bmatrix} 2 \\ 1 \\ 3 \end{bmatrix}^{\top} \begin{bmatrix} 2 \\ 1 \\ 3 \end{bmatrix} \begin{bmatrix} 1 & 2 \end{bmatrix} = \begin{bmatrix} 1 & 2 \end{bmatrix}^{\top} (4+1+9) \begin{bmatrix} 1 & 2 \end{bmatrix}$$

$$= 14 \begin{bmatrix} 1 & 2 \\ 2 & 4 \end{bmatrix}$$

The eigenvalues are $\lambda_0 = 14 \cdot 5 = 70$ and $\lambda_1 = 0$, and the singular values are $\sigma_0 = \sqrt{70}$ and $\sigma_1 = 0$.

The corresponding right singular vector:

$$\mathbf{v}_0 = \frac{1}{\sqrt{5}}\begin{bmatrix} 1 \\ 2 \end{bmatrix}$$

The left singular vector:

$$\mathbf{u}_0 = \frac{1}{\sigma_0}\mathbf{A}\mathbf{v}_0 = \frac{1}{\sqrt{70}} \begin{bmatrix} 2 & 4 \\ 1 & 2 \\ 3 & 6 \end{bmatrix} \frac{1}{\sqrt{5}}\begin{bmatrix} 1 \\ 2 \end{bmatrix} = \frac{1}{\sqrt{14}}\begin{bmatrix} 2 \\ 1 \\ 3 \end{bmatrix}$$

Hence the SVD is

$$\mathbf{A} = \sigma_0 \mathbf{u}_0 \mathbf{v}_0^{\top} = \sqrt{70} \cdot \frac{1}{\sqrt{14}}\begin{bmatrix} 2 \\ 1 \\ 3 \end{bmatrix} \cdot \frac{1}{\sqrt{5}}\begin{bmatrix} 1 & 2 \end{bmatrix} = \begin{bmatrix} 2 \\ 1 \\ 3 \end{bmatrix} \begin{bmatrix} 1 & 2 \end{bmatrix}$$

This confirms that the SVD of a rank-one matrix has only one nonzero singular value.
:::

---

In [ ]:
import numpy as np

# NumPy print format: 4 decimal places, suppress scientific notation for tiny values
np.set_printoptions(precision=4, suppress=True, linewidth=100)

def print_header(title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")

def print_step(step, desc):
    print(f"\n▶ Step {step}: {desc}")
    print("-" * 30)

def compare_print(label, actual, expected_desc):
    """Show the computed result next to the expected description for readability"""
    print(f"[{label} computed]:")
    print(actual)
    print(f"[{label} expected]: {expected_desc}")

# ============================================================
# Example: SVD of a 2×2 matrix
# ============================================================
print_header("Example: Verifying the SVD of a 2×2 Matrix")

A = np.array([[3, 0],
              [4, 5]], dtype=float)
print(f"Original matrix A:\n{A}")

# --- Step 1 ---
print_step(1, "Compute $A^T A$")
ATA = A.T @ A
compare_print("A^T A", ATA, "[[25, 20], [20, 25]]")

# --- Step 2 ---
print_step(2, "Solve for the eigenvalues and singular values")
eigenvalues, eigenvectors = np.linalg.eigh(ATA)
# sort: largest to smallest
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
V = eigenvectors[:, idx]

sigma = np.sqrt(eigenvalues)
print(f"Eigenvalues (λ): {eigenvalues}")
print(f"Singular values (σ): {sigma}")
print(f"Expected σ: [3√5, √5] ≈ [6.7082, 2.2361]")

# --- Step 3 ---
print_step(3, "Matrix of right singular vectors V")
compare_print("V", V, "1/√2 * [[1, 1], [1, -1]]")

# --- Step 4 ---
print_step(4, "Compute the left singular vectors u_i = (1/σ_i) A v_i")
u0 = A @ V[:, 0] / sigma[0]
u1 = A @ V[:, 1] / sigma[1]
U = np.column_stack([u0, u1])

print(f"u0: {u0} (expected 1/√10 * [1, 3])")
print(f"u1: {u1} (expected 1/√10 * [3, -1])")
print(f"\nMatrix of left singular vectors U:\n{U}")

# --- Step 5 ---
print_step(5, "Verify the reconstruction A = U Σ V^T")
Sigma = np.diag(sigma)
A_reconstructed = U @ Sigma @ V.T
print(f"Reconstruction:\n{A_reconstructed}")
print(f"Matches A: {np.allclose(A_reconstructed, A)}")

# --- Step 6 ---
print_step(6, "Outer-product expansion")
A_outer = (sigma[0] * np.outer(u0, V[:, 0]) + 
           sigma[1] * np.outer(u1, V[:, 1]))
print(f"Sum of outer products:\n{A_outer}")

In [ ]:
# ============================================================
print_header("Example: SVD of a Rank-One Matrix")
# ============================================================

A2 = np.array([[2, 4],
               [1, 2],
               [3, 6]], dtype=float)

# --- Step 1: display the matrix and observe its linear dependence ---
print_step(1, "Observe the matrix structure: the geometric meaning of rank one")
print("A =\n", A2)
print(f"  col_0(A) = {A2[:, 0]}")
print(f"  col_1(A) = {A2[:, 1]}")
print(f"  col_1 = 2 × col_0? → {np.allclose(A2[:, 1], 2 * A2[:, 0])}")
print("  ∴ rank(A) = 1; the two columns are collinear, so we expect only 1 nonzero singular value.")

# --- Step 2: compute A^T A ---
print_step(2, "Compute $A^T A$")
ATA2 = A2.T @ A2
print("A^T A =\n", ATA2)
compare_print(
    label   = "A^T A",
    actual  = ATA2,
    expected_desc = "14 × [[1,2],[2,4]]"
)
print("Expected =\n", 14 * np.array([[1, 2], [2, 4]]))

# --- Step 3: eigenvalues → singular values σᵢ = √λᵢ ---
print_step(3, "Find the eigenvalues and singular values $\\sigma_i = \\sqrt{\\lambda_i}$")
eigenvalues2, eigenvectors2 = np.linalg.eigh(ATA2)
idx2 = np.argsort(eigenvalues2)[::-1]          # descending order
eigenvalues2  = eigenvalues2[idx2]
eigenvectors2 = eigenvectors2[:, idx2]

print(f"  λ₀ = 14 × (1² + 2²) = 14 × 5 = {14 * 5}")
print(f"  λ₁ = 0  (a rank-one matrix must have n-1 zero eigenvalues)")
compare_print(
    label        = "eigenvalues [λ₀, λ₁]",
    actual       = np.round(eigenvalues2, 10),
    expected_desc= "[70, 0]"
)

sigma2 = np.sqrt(eigenvalues2[0])
print(f"  σ₀ = √70 ≈ {sigma2:.6f}")

# --- Step 4: right singular vector v₀ ---
print_step(4, "Right singular vector $\\mathbf{v}_0$ (unit eigenvector of $A^T A$)")
v0 = eigenvectors2[:, 0]
compare_print(
    label        = "v₀",
    actual       = v0,
    expected_desc= "1/√5 × [1, 2]"
)
print(f"  Expected: {np.array([1, 2]) / np.sqrt(5)}")

# --- Step 5: left singular vector u₀ = (1/σ₀) A v₀ ---
print_step(5, "Left singular vector $\\mathbf{u}_0 = \\frac{1}{\\sigma_0} A \\mathbf{v}_0$")
u0_2 = A2 @ v0 / sigma2
compare_print(
    label        = "u₀",
    actual       = u0_2,
    expected_desc= "1/√14 × [2, 1, 3]"
)
print(f"  Expected: {np.array([2, 1, 3]) / np.sqrt(14)}")

# --- Step 6: verify the SVD: A = σ₀ u₀ v₀^T (rank-one expansion) ---
print_step(6, "Verify the SVD reconstruction: $A = \\sigma_0 \\, \\mathbf{u}_0 \\mathbf{v}_0^T$")
A2_reconstructed = sigma2 * np.outer(u0_2, v0)
print("Reconstruction =\n", np.round(A2_reconstructed, 10))
print("Original A =\n", A2)
print(f"  Reconstruction correct? → {np.allclose(A2_reconstructed, A2)}")

coeff_check = np.sqrt(70) / np.sqrt(14) / np.sqrt(5)
print(f"\n  Coefficient check: the factor in front of the outer product σ₀/(√14·√5) = √70 / √14 / √5 = {coeff_check:.6f}  (expected 1.0)")

# --- Step 7: cross-check with NumPy SVD ---
print_step(7, "Cross-check with NumPy SVD")
U_np2, s_np2, Vt_np2 = np.linalg.svd(A2, full_matrices=False)
compare_print(
    label        = "singular values s",
    actual       = s_np2,
    expected_desc= "[√70, 0]"
)
print(f"  Expected: [{np.sqrt(70):.6f}, 0.000000]")
print(f"  NumPy reconstruction correct? → {np.allclose(U_np2 @ np.diag(s_np2) @ Vt_np2, A2)}")

### 11.1.2 The SVD and the Four Fundamental Subspaces

Recall from Chapter 6 that every matrix $\mathbf{A} \in \mathbb{C}^{m \times n}$ defines four fundamental subspaces:

1. **Row space**: $\text{Row}(\mathbf{A}) = \text{span}\{\text{row}_0(\mathbf{A}), \ldots, \text{row}_{m-1}(\mathbf{A})\} \subseteq \mathbb{C}^n$
2. **Null space**: $\ker(\mathbf{A}) = \{\mathbf{x} \in \mathbb{C}^n : \mathbf{A}\mathbf{x} = \mathbf{0}\} \subseteq \mathbb{C}^n$
3. **Image** (column space): $\text{Col}(\mathbf{A}) = \text{span}\{\text{col}_0(\mathbf{A}), \ldots, \text{col}_{n-1}(\mathbf{A})\} \subseteq \mathbb{C}^m$
4. **Left null space**: $\ker(\mathbf{A}^{*}) = \{\mathbf{y} \in \mathbb{C}^m : \mathbf{A}^{*}\mathbf{y} = \mathbf{0}\} \subseteq \mathbb{C}^m$

Here $\text{Row}(\mathbf{A})$ is spanned by row vectors written horizontally; to place it in the input space of upright (column) vectors we must take the **conjugate transpose**, obtaining $\text{Col}(\mathbf{A}^{*})$—an ordinary transpose is not enough. The corresponding two **orthogonal decompositions** are

$$\mathbb{C}^n = \text{Col}(\mathbf{A}^{*}) \oplus \ker(\mathbf{A})$$
$$\mathbb{C}^m = \text{Col}(\mathbf{A}) \oplus \ker(\mathbf{A}^{*})$$

**What makes the SVD so convenient is that it provides explicit orthonormal bases for these four fundamental subspaces!**

#### The Orthogonal Bases Provided by the SVD

Let $\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{*}$ be the SVD of a matrix of rank $r$, and let $\mathbf{v}_i$ be the $i$-th column of $\mathbf{V}$. Then:

:::{prf:theorem} The SVD and the Four Fundamental Subspaces
:label: thm-svd-four-subspaces

Let $\mathbf{A} \in \mathbb{C}^{m \times n}$ have SVD $\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{*}$, rank $r$, and singular values $\sigma_0 \geq \cdots \geq \sigma_{r-1} > 0$. Then:

1. **Row space**: $\text{Row}(\mathbf{A}) = \text{span}\{\mathbf{v}_0^*, \mathbf{v}_1^*, \ldots, \mathbf{v}_{r-1}^*\}$
   - $\{\mathbf{v}_0^*, \ldots, \mathbf{v}_{r-1}^*\}$ is an **orthonormal basis** of $\text{Row}(\mathbf{A})$ (in horizontal form; its conjugate transpose $\{\mathbf{v}_0,\ldots,\mathbf{v}_{r-1}\}$ is an orthonormal basis of $\text{Col}(\mathbf{A}^{*})$)

2. **Null space**: $\ker(\mathbf{A}) = \text{span}\{\mathbf{v}_r, \mathbf{v}_{r+1}, \ldots, \mathbf{v}_{n-1}\}$
   - $\{\mathbf{v}_r, \ldots, \mathbf{v}_{n-1}\}$ is an **orthonormal basis** of $\ker(\mathbf{A})$

3. **Image**: $\text{Col}(\mathbf{A}) = \text{span}\{\mathbf{u}_0, \mathbf{u}_1, \ldots, \mathbf{u}_{r-1}\}$
   - $\{\mathbf{u}_0, \ldots, \mathbf{u}_{r-1}\}$ is an **orthonormal basis** of $\text{Col}(\mathbf{A})$

4. **Left null space**: $\ker(\mathbf{A}^{*}) = \text{span}\{\mathbf{u}_r, \mathbf{u}_{r+1}, \ldots, \mathbf{u}_{m-1}\}$
   - $\{\mathbf{u}_r, \ldots, \mathbf{u}_{m-1}\}$ is an **orthonormal basis** of $\ker(\mathbf{A}^{*})$


:::

```{prf:proof}
We verify the four statements one by one:

**1. Row space**: 

We can decompose the matrix product as $\mathbf{A} = \sum_{i=0}^{r-1} \sigma_i \mathbf{u}_i \mathbf{v}_i^*$. Here we sum only up to $r-1$ because the singular values satisfy $\sigma_i = 0$ when $i \geq r$. Every row of the matrix $\mathbf{A}$ is in fact a linear combination of $\mathbf{v}_0^*, \mathbf{v}_1^*, \ldots, \mathbf{v}_{r-1}^*$. Specifically, $\text{Row}_j(\mathbf{A}) = \sum_{i=0}^{r-1} (\sigma_i u_{ji}) \mathbf{v}_i^*$; in other words, $\text{Row}(\mathbf{A}) \subseteq \text{span}\{\mathbf{v}_0^*, \mathbf{v}_1^*, \ldots, \mathbf{v}_{r-1}^*\}$. Since $\text{dim}(\text{Row}(\mathbf{A})) = r$ and $\{\mathbf{v}_0^*, \ldots, \mathbf{v}_{r-1}^*\}$ are $r$ linearly independent vectors, they span $\text{Row}(\mathbf{A})$.

**2. Null space**: for any $i = r, \ldots, n-1$: $\mathbf{A}\mathbf{v}_i = \mathbf{U}\boldsymbol{\Sigma}\mathbf{e}_i = \mathbf{U} \cdot \mathbf{0} = \mathbf{0}$

Hence $\mathbf{v}_i \in \ker(\mathbf{A})$. Since $\text{dim}(\ker(\mathbf{A})) = n - r$ (the rank-nullity theorem) and $\{\mathbf{v}_r, \ldots, \mathbf{v}_{n-1}\}$ has exactly $n-r$ vectors, they span $\ker(\mathbf{A})$.

**3. Image**: by construction, $\mathbf{u}_i = \frac{1}{\sigma_i}\mathbf{A}\mathbf{v}_i$ holds for $i = 0, \ldots, r-1$, so $\mathbf{u}_i \in \text{Col}(\mathbf{A})$. Since $\text{dim}(\text{Col}(\mathbf{A})) = r$, $\{\mathbf{u}_0, \ldots, \mathbf{u}_{r-1}\}$ spans $\text{Col}(\mathbf{A})$.

**4. Left null space**: we need to show that $\mathbf{A}^{*}\mathbf{u}_i = \mathbf{0}$ for $i = r, \ldots, m-1$. From $\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{*}$ we get $\mathbf{A}^{*} = \mathbf{V}\boldsymbol{\Sigma}^{*}\mathbf{U}^{*}$, so

$\mathbf{A}^{*}\mathbf{u}_i = \mathbf{V}\boldsymbol{\Sigma}^{*}\mathbf{U}^{*}\mathbf{u}_i = \mathbf{V}\boldsymbol{\Sigma}^{*}\mathbf{e}_i = \mathbf{V} \cdot \mathbf{0} = \mathbf{0}$ (because column $i$ of $\boldsymbol{\Sigma}^{*}$ is entirely zero when $i \geq r$).
```

:::{prf:example} The Four Fundamental Subspaces
:label: ex-four-fundamental-subspaces

Consider the matrix

$$\mathbf{A} = \begin{bmatrix} 1 & 1 & 0 \\ 0 & 1 & 1 \end{bmatrix}$$

1. **Compute the rank**

Row reduction shows that $\operatorname{rank}(\mathbf{A}) = 2$ (the two rows are linearly independent).

2. **Find bases for the four fundamental subspaces**

    - **Row space** $\operatorname{Row}(\mathbf{A}) \subseteq \mathbb{C}^3$, $\dim = 2$, with basis
  $\{\begin{bmatrix}1\\1\\0\end{bmatrix},\ \begin{bmatrix}0\\1\\1\end{bmatrix}\}$.

    - **Null space** $\ker(\mathbf{A}) \subseteq \mathbb{C}^3$, $\dim = 1$; solving $\mathbf{A}\mathbf{x} = \mathbf{0}$ gives the basis
  $\{\begin{bmatrix}1\\-1\\1\end{bmatrix}\}$.

    - **Image** $\operatorname{Col}(\mathbf{A}) \subseteq \mathbb{C}^2$, $\dim = 2$; taking the first two columns gives the basis
  $\{\begin{bmatrix}1\\0\end{bmatrix},\ \begin{bmatrix}1\\1\end{bmatrix}\}$.

    - **Left null space** $\ker(\mathbf{A}^{\top}) \subseteq \mathbb{C}^2$, $\dim = 0$ (since $\operatorname{rank}(\mathbf{A}^{\top}) = 2$, all of $\mathbb{C}^2$ is the image); its basis is the empty set.

3. **Compute the SVD and orthogonalize the basis vectors**

Compute $\mathbf{A}^{\top}\mathbf{A}$:

$$\mathbf{A}^{\top}\mathbf{A}
= \begin{bmatrix} 1 & 0 \\ 1 & 1 \\ 0 & 1 \end{bmatrix}
  \begin{bmatrix} 1 & 1 & 0 \\ 0 & 1 & 1 \end{bmatrix}
= \begin{bmatrix} 1 & 1 & 0 \\ 1 & 2 & 1 \\ 0 & 1 & 1 \end{bmatrix}$$

Characteristic polynomial:

$$\det(\mathbf{A}^{\top}\mathbf{A} - \lambda\mathbf{I})
= \det\begin{bmatrix} 1-\lambda & 1 & 0 \\ 1 & 2-\lambda & 1 \\ 0 & 1 & 1-\lambda \end{bmatrix}
= -\lambda(\lambda-1)(\lambda-3) = 0$$

The eigenvalues are $\lambda_0 = 3$, $\lambda_1 = 1$, $\lambda_2 = 0$, with corresponding singular values $\sigma_0 = \sqrt{3}$ and $\sigma_1 = 1$.

The orthogonalized eigenvectors are

$$\mathbf{v}_0 = \frac{1}{\sqrt{6}}\begin{bmatrix}1\\2\\1\end{bmatrix}, \qquad
  \mathbf{v}_1 = \frac{1}{\sqrt{2}}\begin{bmatrix}1\\0\\-1\end{bmatrix}, \qquad
  \mathbf{v}_2 = \frac{1}{\sqrt{3}}\begin{bmatrix}1\\-1\\1\end{bmatrix}$$

**Verifying orthogonality**

- $\operatorname{Row}(\mathbf{A}) = \operatorname{span}\{\mathbf{v}_0,\, \mathbf{v}_1\}$ (the first two right singular vectors)
- $\ker(\mathbf{A}) = \operatorname{span}\{\mathbf{v}_2\}$ (the right singular vector for the zero singular value)
- $\mathbf{v}_0 \perp \mathbf{v}_2$ and $\mathbf{v}_1 \perp \mathbf{v}_2$ ✓

This example shows perfectly how the SVD automatically provides **orthonormal bases** for the four fundamental subspaces.
:::

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True, linewidth=100)

# ── Helper functions ────────────────────────────────────────────────────────────────
def print_header(title):
    print("\n" + "=" * 60)
    print(f"  {title}")
    print("=" * 60)

def print_step(step, desc):
    print(f"\n▶ Step {step}: {desc}")
    print("-" * 40)

def compare_print(label, actual, expected_desc):
    print(f"  {label}")
    print(f"    computed : {actual}")
    print(f"    expected : {expected_desc}")

# ============================================================
print_header("Example: Verifying the Four Fundamental Subspaces with the SVD")
# ============================================================

A = np.array([[1, 1, 0],
              [0, 1, 1]], dtype=float)

# --- Step 1 ---
print_step(1, "Define the matrix A and compute rank(A)")
print(f"  A =\n{A}")
compare_print("rank(A)", np.linalg.matrix_rank(A), "2")

# --- Step 2 ---
print_step(2, "Null space Null(A): solve Ax = 0")
null_vec = np.array([1, -1, 1]) / np.sqrt(3)
print(f"  Null-space basis v₂ = [1, -1, 1] / √3 = {null_vec}")
compare_print("A @ v₂", np.round(A @ null_vec, 10), "0  (should be the zero vector)")

# --- Step 3 ---
print_step(3, "Compute A^T A and compare with the formula")
ATA = A.T @ A
print(f"  A^T A =\n{ATA}")
print(f"  Expected:\n  [[1,1,0],[1,2,1],[0,1,1]]")

# --- Step 4 ---
print_step(4, "Eigendecomposition of A^T A (descending order)")
eigenvalues, eigenvectors = np.linalg.eigh(ATA)   # eigh returns ascending order
eigenvalues  = eigenvalues[::-1]                   # switch to descending λ₀ ≥ λ₁ ≥ λ₂
eigenvectors = eigenvectors[:, ::-1]
compare_print("eigenvalues [λ₀, λ₁, λ₂]",
              np.round(eigenvalues, 10),
              "[3, 1, 0]")

sigma = np.sqrt(eigenvalues[:2])                   # nonzero singular values σᵢ = √λᵢ
compare_print("singular values [σ₀, σ₁]",
              np.round(sigma, 6),
              "[√3, 1] ≈ [1.7321, 1.0000]")

# --- Step 5 ---
print_step(5, "Compare with the right singular vectors vᵢ in the text (bases of Row(A) and ker(A))")
v0_doc = np.array([ 1,  2,  1]) / np.sqrt(6)
v1_doc = np.array([ 1,  0, -1]) / np.sqrt(2)
v2_doc = np.array([ 1, -1,  1]) / np.sqrt(3)

V = eigenvectors   # computed [v₀ | v₁ | v₂]

for i, (v_doc, label) in enumerate([(v0_doc, "v₀"), (v1_doc, "v₁"), (v2_doc, "v₂")]):
    v_calc = V[:, i]
    consistent = np.isclose(abs(np.vdot(v_calc, v_doc)) / (np.linalg.norm(v_calc) * np.linalg.norm(v_doc)), 1.0)
    print(f"\n  {label}  text     = {v_doc}")
    print(f"      computed = {v_calc}")
    print(f"      same direction: {consistent}")

print(f"\n  Check A @ v₂ ≈ 0 : {np.allclose(A @ v2_doc, 0)}")

# --- Step 6 ---
print_step(6, "Compute the left singular vectors uᵢ = (1/σᵢ) A vᵢ (basis of Col(A))")
u0 = A @ V[:, 0] / sigma[0]
u1 = A @ V[:, 1] / sigma[1]
U  = np.column_stack([u0, u1])
print(f"  U = [u₀ | u₁] =\n{np.round(U, 6)}")

# --- Step 7 ---
print_step(7, "Verify the reconstruction: A = U Σ V^T")
Sigma = np.diag(sigma)
A_rec = U @ Sigma @ V[:, :2].T
print(f"  U Σ V^T =\n{np.round(A_rec, 10)}")
compare_print("reconstruction correct", np.allclose(A_rec, A), "True")

# --- Step 8 ---
print_step(8, "Cross-check with NumPy SVD")
U_np, s_np, Vt_np = np.linalg.svd(A, full_matrices=True)
compare_print("NumPy singular values [σ₀, σ₁]",
              np.round(s_np, 6),
              f"[√3, 1] ≈ {np.round(sigma, 6)}")
A_np_rec = U_np @ np.diag([s_np[0], s_np[1]]) @ Vt_np[:2]
compare_print("NumPy reconstruction correct", np.allclose(A_np_rec, A), "True")

### 11.1.3 Geometric Intuition: From the Unit Sphere to an Ellipsoid

The most beautiful geometric interpretation of the SVD is this: **every linear transformation maps the unit sphere to an ellipsoid, and the SVD describes precisely the directions and lengths of the principal axes of this ellipsoid.**

#### The Geometric Effect of a Linear Transformation

Consider a linear transformation $\mathbf{A}: \mathbb{R}^n \to \mathbb{R}^m$. The **unit sphere** in the domain is

$$S^{n-1} := \{\mathbf{x} \in \mathbb{R}^n : \|\mathbf{x}\| = 1\}$$

and its image under $\mathbf{A}$ is

$$\mathbf{A}[S^{n-1}] = \{\mathbf{A}\mathbf{x} : \|\mathbf{x}\| = 1\}$$

**Question.** What is the geometric shape of this image set?

**Answer.** It is an **ellipsoid** in $\mathbb{R}^m$! More precisely, it is an **$r$-dimensional ellipsoid** (where $r = \text{rank}(\mathbf{A})$), embedded in an $r$-dimensional subspace of the $m$-dimensional space (namely the image $\text{Col}(\mathbf{A})$).

#### The Geometric Interpretation of the SVD

Let $\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{\top}$ be an SVD. Then for any unit vector $\mathbf{x} \in S^{n-1}$,

$$\mathbf{A}\mathbf{x} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{\top}\mathbf{x}$$

**We understand this in three steps**:

**Step 1. Rotation ($\mathbf{V}^{\top}$)**

$\mathbf{V}^{\top}$ is an orthogonal matrix, corresponding to a **rigid rotation** (or reflection). It rotates $\mathbf{x}$ from the original coordinate system into the **coordinate system of the right singular vectors** $\{\mathbf{v}_0, \mathbf{v}_1, \ldots, \mathbf{v}_{n-1}\}$:

$$\mathbf{y} := \mathbf{V}^{\top}\mathbf{x} = \begin{bmatrix}⟨  \mathbf{v}_0 | \mathbf{x} ⟩ \\ ⟨  \mathbf{v}_1 | \mathbf{x} ⟩ \\ \vdots \\ ⟨  \mathbf{v}_{n-1} | \mathbf{x} ⟩\end{bmatrix}$$

Since $\mathbf{V}$ is an orthogonal matrix, $\|\mathbf{y}\| = \|\mathbf{x}\| = 1$, so $\mathbf{y}$ is still on the unit sphere.

**Step 2. Anisotropic stretching ($\boldsymbol{\Sigma}$)**

$\boldsymbol{\Sigma}$ is a diagonal matrix that performs **independent stretching** along each coordinate direction:

$$\mathbf{z} := \boldsymbol{\Sigma}\mathbf{y} = \begin{bmatrix}\sigma_0 y_0 \\ \sigma_1 y_1 \\ \vdots \\ \sigma_{r-1} y_{r-1} \\ 0 \\ \vdots \\ 0\end{bmatrix}$$

This operation **stretches** the unit sphere into an ellipsoid:
- stretching by a factor of $\sigma_0$ in the direction $\mathbf{e}_0$ (the largest singular value, the longest principal axis)
- stretching by a factor of $\sigma_1$ in the direction $\mathbf{e}_1$
- ...
- stretching by a factor of $\sigma_{r-1}$ in the direction $\mathbf{e}_{r-1}$ (the smallest nonzero singular value)
- compressing to $0$ in the directions $\mathbf{e}_r, \ldots, \mathbf{e}_{n-1}$ (corresponding to the null space)

**Step 3. Rotation ($\mathbf{U}$)**

$\mathbf{U}$ rotates the ellipsoid from the standard coordinate system into the **coordinate system of the left singular vectors** $\{\mathbf{u}_0, \mathbf{u}_1, \ldots, \mathbf{u}_{m-1}\}$:

$$\mathbf{A}\mathbf{x} = \mathbf{U}\mathbf{z}$$

In the end, the **directions of the principal axes** of the ellipsoid are the left singular vectors $\mathbf{u}_0, \mathbf{u}_1, \ldots, \mathbf{u}_{r-1}$, and the **lengths of the principal axes** are the corresponding singular values $\sigma_0, \sigma_1, \ldots, \sigma_{r-1}$.

#### The Variational Viewpoint: Directions of Maximal Stretching

Another way to understand the SVD is through **optimization**: the singular vectors are the directions in which $\mathbf{A}$ produces the **largest/smallest stretching**.

:::{prf:theorem} Extremal Characterization of Singular Values
:label: thm-singular-value-extremal

Let $\mathbf{A} \in \mathbb{R}^{m \times n}$ have SVD $\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{\top}$. Then

$$\sigma_0 = \max_{\|\mathbf{x}\|=1} \|\mathbf{A}\mathbf{x}\|$$

and the maximum is attained at $\mathbf{x} = \mathbf{v}_0$. More generally, for $i = 0, 1, \ldots, r-1$,

$$\sigma_i = \max_{\mathbf{x}} \|\mathbf{A}\mathbf{x}\|
\quad \text{s.t.} \quad
\|\mathbf{x}\| = 1,\; \mathbf{x} \perp \mathbf{v}_0, \ldots, \mathbf{v}_{i-1}$$

and the maximum is attained at $\mathbf{x} = \mathbf{v}_i$.
:::

:::{prf:proof}
Write any unit vector $\mathbf{x}$ as a linear combination of the right singular vectors:

$$\mathbf{x} = \sum_{j=0}^{n-1} c_j \mathbf{v}_j, \qquad \sum_{j=0}^{n-1} |c_j|^2 = 1$$

By the orthogonality in the SVD,

$$\mathbf{A}\mathbf{x} = \sum_{j=0}^{r-1} c_j \sigma_j \mathbf{u}_j$$

Therefore

$$\|\mathbf{A}\mathbf{x}\|^2 = \sum_{j=0}^{r-1} \sigma_j^2 |c_j|^2 \leq \sigma_0^2 \sum_{j=0}^{r-1} |c_j|^2 \leq \sigma_0^2$$

For nonzero $\mathbf{A}$, equality holds if and only if the unit vector $\mathbf{x}$ lies in the right singular subspace corresponding to the largest singular value. When the largest singular value is simple, this means $\pm\mathbf{v}_0$ in the real case and $e^{i\theta}\mathbf{v}_0$ in the complex case. If $\mathbf{A}=0$, equality holds for every unit vector.

$$\underline{\sigma_0 = \max_{\|\mathbf{x}\|=1} \|\mathbf{A}\mathbf{x}\|, \text{ with the maximum attained at } \mathbf{x} = \mathbf{v}_0}$$
:::

**Geometric meaning**: 
- $\mathbf{v}_0$ is the direction that $\mathbf{A}$ stretches the **most**
- when $r>0$, among the unit vectors in $\ker(\mathbf{A})^\perp$, $\mathbf{v}_{r-1}$ attains the smallest stretching $\sigma_{r-1}$
- $\mathbf{v}_r, \ldots, \mathbf{v}_{n-1}$ are compressed to zero by $\mathbf{A}$

This variational viewpoint plays a central role in principal component analysis (PCA, §11.3)—PCA is essentially a search for the directions of greatest variation in the data, and these directions are exactly the right singular vectors of the data matrix!

:::{prf:example} 2D Visualization: The Unit Circle Mapped to an Ellipse
:label: ex-svd-2d-visualization

Consider the symmetric matrix

$$\mathbf{A} = \begin{bmatrix} 3 & 1 \\ 1 & 3 \end{bmatrix}$$

**Computing the SVD**

$\mathbf{A}$ is a symmetric positive definite matrix, so its SVD coincides with its eigendecomposition (see Chapter 9). The eigenvalues are $\lambda_0 = 4$ and $\lambda_1 = 2$, with orthonormal eigenvectors

$$\mathbf{v}_0 = \frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix}, \qquad \mathbf{v}_1 = \frac{1}{\sqrt{2}}\begin{bmatrix}1\\-1\end{bmatrix}$$

Since $\mathbf{A}$ is symmetric positive definite, $\mathbf{U} = \mathbf{V}$, and the singular values are $\sigma_0 = 4$ and $\sigma_1 = 2$.

**Transforming the unit circle**

Under $\mathbf{A}$, the unit circle $\{\mathbf{x} \in \mathbb{R}^2 : \|\mathbf{x}\| = 1\}$ is mapped to the ellipse $\{\mathbf{A}\mathbf{x} : \|\mathbf{x}\| = 1\}$, whose principal axes are:

- **Major axis**: direction $\mathbf{v}_0 = \tfrac{1}{\sqrt{2}}[1,\ 1]^{\top}$ ($45°$), length $\sigma_0 = 4$
- **Minor axis**: direction $\mathbf{v}_1 = \tfrac{1}{\sqrt{2}}[1,\ {-1}]^{\top}$ ($-45°$), length $\sigma_1 = 2$

**Verifying the direction of maximal stretching**

Take $\mathbf{x} = \mathbf{v}_0$:

$$\mathbf{A}\mathbf{v}_0 = \begin{bmatrix} 3 & 1 \\ 1 & 3 \end{bmatrix} \frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix} = \frac{1}{\sqrt{2}}\begin{bmatrix}4\\4\end{bmatrix} = 4\mathbf{v}_0$$

Hence $\|\mathbf{A}\mathbf{v}_0\| = 4 = \sigma_0$, confirming that $\mathbf{v}_0$ is the direction of maximal stretching. Similarly,

$$\mathbf{A}\mathbf{v}_1 = 2\mathbf{v}_1, \qquad \|\mathbf{A}\mathbf{v}_1\| = 2 = \sigma_1$$

This example clearly shows the geometric essence of the SVD: **the singular vectors are the directions of the principal axes of the ellipsoid, and the singular values are the lengths of the principal axes**.
:::

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Color scheme: NYU Official ---
C_BG, C_GRID, C_AXIS = "#F8F8F8", "#D6D6D6", "#000000"
C_V1, C_V2, C_T1, C_WARN = "#57068C", "#006385", "#2AD2C9", "#FF5D47"
# Fonts: the English edition needs no CJK font, so nothing is downloaded when _lang == 'en';
# the Chinese editions use this same block to fetch Noto Sans TC/SC where no CJK font is installed
import os, urllib.request
import matplotlib.font_manager as fm
_lang = 'en'
_cjk = ['Microsoft JhengHei', 'PingFang TC', 'Noto Sans CJK TC', 'Noto Sans TC']
_have = {f.name for f in fm.fontManager.ttflist}
if _lang != 'en' and not _have & set(_cjk):
    _font = os.path.join(os.path.expanduser('~'), '.cache', 'fonts', 'NotoSansTC.ttf')
    try:
        if not os.path.exists(_font):
            os.makedirs(os.path.dirname(_font), exist_ok=True)
            urllib.request.urlretrieve('https://github.com/google/fonts/raw/main/ofl/notosanstc/NotoSansTC%5Bwght%5D.ttf', _font + '.part')
            os.replace(_font + '.part', _font)
        fm.fontManager.addfont(_font)
        _have.add('Noto Sans TC')
    except OSError as err:
        print('Could not download the CJK font; Chinese text in figures may not display:', err)
plt.rcParams['font.family'] = [f for f in _cjk if f in _have] + ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# the symmetric matrix of the example: σ0 = 4, σ1 = 2
A = np.array([[3., 1.],
              [1., 3.]])
U, sigma, Vt = np.linalg.svd(A)

t = np.linspace(0, 2 * np.pi, 400)
circle = np.vstack([np.cos(t), np.sin(t)])   # unit circle
ellipse = A @ circle                          # image: an ellipse

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 5), subplot_kw=dict(aspect='equal'))
for ax in (ax0, ax1):
    ax.set_facecolor(C_BG)
    ax.grid(True, color=C_GRID, linewidth=0.8, alpha=0.7)
    ax.axhline(0, color=C_GRID, linewidth=0.8)
    ax.axvline(0, color=C_GRID, linewidth=0.8)
    ax.set_xlim(-4.8, 4.8)
    ax.set_ylim(-4.8, 4.8)

# left panel: the unit circle and right singular vectors v_i on the input side
ax0.plot(circle[0], circle[1], color=C_T1, linewidth=2)
for i, c in enumerate([C_V1, C_V2]):
    v = Vt[i]
    ax0.annotate('', xy=v, xytext=(0, 0),
                 arrowprops=dict(arrowstyle='-|>', color=c, lw=2.5))
    ax0.text(*(v * 1.3), f'$\\mathbf{{v}}_{i}$', color=c, fontsize=13, ha='center')
ax0.set_title('Input: unit circle and right singular vectors vᵢ', color=C_AXIS)

# right panel: the ellipse on the output side, principal axes = σᵢ uᵢ
ax1.plot(ellipse[0], ellipse[1], color=C_WARN, linewidth=2)
for i, c in enumerate([C_V1, C_V2]):
    u_scaled = sigma[i] * U[:, i]
    ax1.annotate('', xy=u_scaled, xytext=(0, 0),
                 arrowprops=dict(arrowstyle='-|>', color=c, lw=2.5))
    ax1.text(*(u_scaled * 1.15), f'$\\sigma_{i}\\mathbf{{u}}_{i}$', color=c, fontsize=13, ha='center')
ax1.set_title('Output: ellipse with axis directions uᵢ and lengths σᵢ', color=C_AXIS)

fig.suptitle('A = [[3, 1], [1, 3]]: unit circle → ellipse (σ0 = 4, σ1 = 2)', color=C_AXIS)
plt.tight_layout()
plt.show()

print(f"Singular values σ = {np.round(sigma, 4)} (theoretical values [4, 2])")


:::{admonition} Three Ways to Read the Geometric Picture
:class: note

From this figure we can read off the three core quantities of the SVD directly:

- **Singular value $\sigma_i$** = the **length** of the $i$-th semi-axis of the ellipse
- **Left singular vector $\mathbf{u}_i$** = the **direction** of the $i$-th semi-axis of the ellipse (in the output space)
- **Right singular vector $\mathbf{v}_i$** = the corresponding **principal axis direction** in the input space, before the mapping

This reading runs through the next two sections: the spectral norm $\|\mathbf{A}\|_2 = \sigma_0$ of §11.2
is the length of the longest semi-axis, and the principal component directions of PCA in §11.3 are the principal axis directions $\mathbf{v}_i$ of the data ellipsoid.
:::


---

:::{admonition} §11.1 Summary
:class: tip

This section answered a fundamental question: how can the spectral theorem for symmetric matrices be generalized to matrices of arbitrary shape? The answer is the singular value decomposition $\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{*}$—through the bridge $\mathbf{A}^{*}\mathbf{A}$, every matrix has two orthonormal bases and a set of nonnegative real numbers that decompose the linear transformation exactly into three geometric actions: "rotation—stretch—rotation."

This decomposition can be read in three equivalent ways. Algebraically, the singular values are the nonnegative square roots of the eigenvalues of $\mathbf{A}^{*}\mathbf{A}$, and the left and right singular vectors form orthonormal bases of $\mathbb{C}^m$ and $\mathbb{C}^n$, respectively. In terms of subspaces, the columns of $\mathbf{V}$ span $\text{Col}(\mathbf{A}^{*})$ and the null space, the columns of $\mathbf{U}$ split into the range and the left null space, and each of the four fundamental subspaces finds its place under the SVD. Geometrically, $\mathbf{A}$ maps the unit sphere to an ellipsoid; the singular values are precisely the lengths of the principal axes, the right singular vectors are the principal axis directions on the input side, and the left singular vectors are the principal axis directions on the output side.

The uniqueness of the singular values guarantees the objective meaning of the decomposition, while the freedom of the singular vectors under repeated singular values reflects that the subspaces are the true invariants. These three viewpoints take turns on stage in the sections that follow: §11.2 asks "what optimality do these singular values have in the sense of norms?", which leads to the crown of the chapter's theory—the Eckart–Young theorem.
:::

## 11.2 Singular Values and Matrix Norms

In §11.1 we established the basic theory of the SVD and saw how the singular values describe the geometric structure by which a matrix maps the unit sphere to an ellipsoid. This section explores the **relationship between singular values and matrix norms** in depth and proves one of the most important applications of the SVD—the **Eckart–Young theorem**: the truncated SVD gives the best low-rank approximation of a matrix.

### From Vector Norms to Matrix Norms

In Chapter 2 we became familiar with the concept of the $l_2$ (Euclidean) **norm** of a vector, which abstracts the intuitive notion of length. In fact, $l_2$ is just one special case of a whole family of $l_p$ norms. We introduce them here together, since they will be used later in the analysis of matrix norms.

:::{prf:definition} $l_p$ Norm
:label: def-lp-norm

For an $n$-dimensional real vector $\mathbf{v} = (v_0, v_1, \ldots, v_{n-1})^\top$, the $l_p$ **norm** ($p \geq 1$) is defined as

$$
\|\mathbf{v}\|_p := (\sum_{i=0}^{n-1} |v_i|^p)^{1/p}
$$

Common special cases:

$$
\|\mathbf{v}\|_1 = \sum_{i=0}^{n-1} |v_i|, \quad
\|\mathbf{v}\|_2 = \sqrt{\sum_{i=0}^{n-1} v_i^2}, \quad
\|\mathbf{v}\|_\infty = \lim_{p\to\infty}\|\mathbf{v}\|_p = \max_{0 \leq i < n} |v_i|
$$

Every $l_p$ norm ($p \geq 1$) satisfies the three norm axioms:

1. **Positive definiteness**: $\|\mathbf{v}\|_p \geq 0$, and $\|\mathbf{v}\|_p = 0 ⟺ \mathbf{v} = \mathbf{0}$
2. **Absolute homogeneity**: $\|\lambda \mathbf{v}\|_p = |\lambda|\,\|\mathbf{v}\|_p$
3. **Triangle inequality**: $\|\mathbf{u} + \mathbf{v}\|_p \leq \|\mathbf{u}\|_p + \|\mathbf{v}\|_p$ (the Minkowski inequality)
:::
```{note}
For $p < 1$ the definition above does not satisfy the triangle inequality, so it is not a norm, but it is still used in sparsity analysis (where it is called a "quasinorm"). This book considers only the case $p \geq 1$.
```

Different $l_p$ norms define different "balls" (the set of all vectors with $\|\mathbf{v}\|_p \leq 1$). In the plane, the shapes of these **unit balls** reveal the geometric essence of each norm:

- $p=1$: a diamond (Manhattan distance)
- $p=2$: the standard circle (Euclidean distance)
- $p \to \infty$: a square (maximum-coordinate distance)

The figure drawn by the code below clearly shows the geometric process by which, as $p$ increases, the unit ball gradually "inflates" from a diamond into a circle and finally approaches a square. This intuition is essential for understanding the properties of matrix norms later on.

**Question.** How should we define the "size" of a matrix?

There are two most natural lines of generalization:

1. **Induced norm**: regard the matrix as an **operator** and measure its "maximum amplification factor" on vectors
2. **Entrywise norm**: regard the matrix as a **high-dimensional vector** and define a norm directly on all of its entries

These two lines lead to the two great families of matrix norms. The spectral norm, the Frobenius norm, and the nuclear norm introduced below are all unitarily invariant norms and can be described uniformly by the singular values; not every induced norm or entrywise norm has this property.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Fonts: the English edition needs no CJK font, so nothing is downloaded when _lang == 'en';
# the Chinese editions use this same block to fetch Noto Sans TC/SC where no CJK font is installed
import os, urllib.request
import matplotlib.font_manager as fm
_lang = 'en'
_cjk = ['Microsoft JhengHei', 'PingFang TC', 'Noto Sans CJK TC', 'Noto Sans TC']
_have = {f.name for f in fm.fontManager.ttflist}
if _lang != 'en' and not _have & set(_cjk):
    _font = os.path.join(os.path.expanduser('~'), '.cache', 'fonts', 'NotoSansTC.ttf')
    try:
        if not os.path.exists(_font):
            os.makedirs(os.path.dirname(_font), exist_ok=True)
            urllib.request.urlretrieve('https://github.com/google/fonts/raw/main/ofl/notosanstc/NotoSansTC%5Bwght%5D.ttf', _font + '.part')
            os.replace(_font + '.part', _font)
        fm.fontManager.addfont(_font)
        _have.add('Noto Sans TC')
    except OSError as err:
        print('Could not download the CJK font; Chinese text in figures may not display:', err)
plt.rcParams['font.family'] = [f for f in _cjk if f in _have] + ['DejaVu Sans']

# fix the display of minus signs
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 5, figsize=(14, 3), subplot_kw=dict(aspect='equal'))
p_values = [0.5, 1, 2, 4, np.inf]
titles    = [r'$l_{0.5}$', r'$l_1$', r'$l_2$', r'$l_4$', r'$l_\infty$']
colors    = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6']

theta = np.linspace(0, 2 * np.pi, 1000)

for ax, p, title, color in zip(axes, p_values, titles, colors):
    if np.isinf(p):
        # l_∞ unit ball: a square
        square = plt.Polygon(
            [[-1,-1],[1,-1],[1,1],[-1,1]], closed=True,
            facecolor=color, alpha=0.25, edgecolor=color, linewidth=2
        )
        ax.add_patch(square)
    else:
        # parametrize by angle: in the direction of angle theta, the boundary point of the unit ball is
        # (cos θ, sin θ) / ||(cos θ, sin θ)||_p
        cos_t = np.cos(theta)
        sin_t = np.sin(theta)
        norm_p = (np.abs(cos_t)**p + np.abs(sin_t)**p) ** (1.0 / p)
        x = cos_t / norm_p
        y = sin_t / norm_p
        ax.fill(x, y, alpha=0.25, color=color)
        ax.plot(x, y, color=color, linewidth=2)

    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_title(title, fontsize=14)
    ax.set_xticks([-1, 0, 1])
    ax.set_yticks([-1, 0, 1])
    ax.tick_params(labelsize=9)

fig.suptitle(r'Unit balls $\{\mathbf{v} : \|\mathbf{v}\|_p \leq 1\}$ of the $l_p$ norms in $\mathbb{R}^2$',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('lp_unit_balls.svg', bbox_inches='tight')
plt.show()

### 11.2.1 Characterizing Matrix Norms by Singular Values

#### The Spectral Norm (Operator Norm)

**Definition.** The **spectral norm** (also called the **operator norm** or the **2-norm**) of a matrix $\mathbf{A} \in \mathbb{C}^{m \times n}$ is defined as

$$\|\mathbf{A}\|_2 := \max_{\mathbf{x} \neq \mathbf{0}} \frac{\|\mathbf{A}\mathbf{x}\|_2}{\|\mathbf{x}\|_2} = \max_{\|\mathbf{x}\|_2 = 1} \|\mathbf{A}\mathbf{x}\|_2$$

**Geometric meaning.** The spectral norm measures the **maximum stretching ratio of unit vectors** under the matrix $\mathbf{A}$.

:::{prf:theorem} Characterization of the Spectral Norm by Singular Values
:label: thm-spectral-norm-svd

Let $\mathbf{A} \in \mathbb{C}^{m \times n}$ have singular values $\sigma_0 \geq \sigma_1 \geq \cdots \geq \sigma_{r-1} > 0$ (where $r = \text{rank}(\mathbf{A})$). Then

$$\|\mathbf{A}\|_2 = \sigma_0$$

that is, the spectral norm equals the **largest singular value**.
:::

```{prf:proof}
From the variational result of §11.1.3 we have already proved that

$$\max_{\|\mathbf{x}\|_2=1} \|\mathbf{A}\mathbf{x}\|_2 = \sigma_0, \quad \text{attained at } \mathbf{x} = \mathbf{v}_0$$

Hence $\underline{\|\mathbf{A}\|_2 = \sigma_0}$.
```

**Geometric intuition.** Recall from §11.1.3 that the matrix $\mathbf{A}$ maps the unit sphere to an ellipsoid, and the **length of the longest principal axis** of the ellipsoid is precisely $\sigma_0$. The spectral norm captures exactly this "maximum stretching."

:::{prf:example} Spectral Norm of a Diagonal Matrix
:label: ex-spectral-norm-diagonal

Consider the diagonal matrix

$$\mathbf{D} = \begin{bmatrix} 5 & 0 & 0 \\ 0 & 3 & 0 \\ 0 & 0 & 1 \end{bmatrix}$$

Since $\mathbf{D}$ is already in the diagonal form of an SVD, its singular values are the diagonal entries $\sigma_0 = 5,\ \sigma_1 = 3,\ \sigma_2 = 1$, and therefore

$$\|\mathbf{D}\|_2 = 5$$

Check: take the unit vector $\mathbf{e}_0 = [1, 0, 0]^{\top}$; then

$$\|\mathbf{D}\mathbf{e}_0\|_2 = \|[5, 0, 0]^{\top}\|_2 = 5 = \|\mathbf{D}\|_2 \checkmark$$
:::

:::{prf:example} Spectral Norm of a Rotation Matrix
:label: ex-spectral-norm-rotation

Consider the 2D rotation matrix (recall Chapter 3)

$$\mathbf{R}_\theta = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

A rotation matrix is an **orthogonal matrix**, satisfying $\mathbf{R}_\theta^{\top}\mathbf{R}_\theta = \mathbf{I}$, so for any vector $\mathbf{x}$,

$$\|\mathbf{R}_\theta \mathbf{x}\|_2^2 = \mathbf{x}^{\top}\mathbf{R}_\theta^{\top}\mathbf{R}_\theta \mathbf{x} = \mathbf{x}^{\top}\mathbf{x} = \|\mathbf{x}\|_2^2$$

A rotation matrix preserves length, so its spectral norm is

$$\|\mathbf{R}_\theta\|_2 = 1$$

This conclusion generalizes: all singular values of any orthogonal (or unitary) matrix equal 1, so its spectral norm is always 1.
:::

#### The Frobenius Norm

:::{prf:definition} Frobenius Norm
:label: def-frobenius-norm

The **Frobenius norm** (also called the F-norm) of a matrix $\mathbf{A} \in \mathbb{C}^{m \times n}$ is defined as

$$\|\mathbf{A}\|_F := \sqrt{\sum_{i=0}^{m-1}\sum_{j=0}^{n-1} |a_{ij}|^2}$$

This is the $\ell_2$ norm of the $m \times n$ entries of the matrix regarded as a vector of dimension $mn$. Equivalently, it can be written in terms of the trace:

$$\|\mathbf{A}\|_F = \sqrt{\operatorname{tr}(\mathbf{A}^{*}\mathbf{A})} = \sqrt{\operatorname{tr}(\mathbf{A}\mathbf{A}^{*})}$$
:::

:::{prf:theorem} Characterization of the Frobenius Norm by Singular Values
:label: thm-frobenius-norm-svd

Let $\mathbf{A} \in \mathbb{C}^{m \times n}$ have singular values $\sigma_0, \sigma_1, \ldots, \sigma_{r-1}$ (where $r = \text{rank}(\mathbf{A})$). Then

$$\|\mathbf{A}\|_F^2 = \sum_{i=0}^{r-1} \sigma_i^2 = \sum_{i=0}^{\min(m,n)-1} \sigma_i^2$$

that is, the square of the Frobenius norm equals the **sum of the squares of all singular values**.
:::

```{prf:proof}
Using the SVD $\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{*}$:

$$\mathbf{A}^{*}\mathbf{A} = \mathbf{V}\boldsymbol{\Sigma}^{*}\mathbf{U}^{*}\mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{*} = \mathbf{V}\boldsymbol{\Sigma}^{*}\boldsymbol{\Sigma}\mathbf{V}^{*}$$

(because $\mathbf{U}$ is unitary, $\mathbf{U}^{*}\mathbf{U} = \mathbf{I}$)

Compute the trace:

$$\text{tr}(\mathbf{A}^{*}\mathbf{A}) = \text{tr}(\mathbf{V}\boldsymbol{\Sigma}^{*}\boldsymbol{\Sigma}\mathbf{V}^{*}) = \text{tr}(\boldsymbol{\Sigma}^{*}\boldsymbol{\Sigma}\mathbf{V}^{*}\mathbf{V}) = \text{tr}(\boldsymbol{\Sigma}^{*}\boldsymbol{\Sigma})$$

(using the cyclic property of the trace $\text{tr}(\mathbf{ABC}) = \text{tr}(\mathbf{CAB})$)

Since $\boldsymbol{\Sigma}^{*}\boldsymbol{\Sigma}$ is a diagonal matrix with diagonal entries $\sigma_0^2, \sigma_1^2, \ldots, \sigma_{r-1}^2, 0, \ldots, 0$,

$$\text{tr}(\boldsymbol{\Sigma}^{*}\boldsymbol{\Sigma}) = \sum_{i=0}^{r-1} \sigma_i^2$$

Hence $\underline{\|\mathbf{A}\|_F^2 = \sum_{i=0}^{r-1} \sigma_i^2}$.
```

**Geometric meaning.** The Frobenius norm measures the "total energy" of a matrix—it combines the contributions of all singular values (the square root of the sum of squares). This will play a key role in the low-rank approximation that follows.

:::{prf:example} Computing the Frobenius Norm
:label: ex-frobenius-norm-computation

Consider the matrix

$$\mathbf{A} = \begin{bmatrix} 1 & 2 \\ 3 & 4 \\ 5 & 6 \end{bmatrix}$$

**Method 1: direct computation**

$$\|\mathbf{A}\|_F = \sqrt{1^2 + 2^2 + 3^2 + 4^2 + 5^2 + 6^2} = \sqrt{91}$$

**Method 2: through the singular values**

Compute $\mathbf{A}^{\top}\mathbf{A}$:

$$\mathbf{A}^{\top}\mathbf{A} = \begin{bmatrix}1&3&5\\2&4&6\end{bmatrix}\begin{bmatrix}1&2\\3&4\\5&6\end{bmatrix} = \begin{bmatrix}35&44\\44&56\end{bmatrix}$$

From the characteristic polynomial $\det(\mathbf{A}^{\top}\mathbf{A} - \lambda\mathbf{I}) = \lambda^2 - 91\lambda + 24 = 0$ we get $\lambda_0 \approx 90.74$ and $\lambda_1 \approx 0.26$, so

$$\|\mathbf{A}\|_F = \sqrt{\lambda_0 + \lambda_1} = \sqrt{91} \quad \checkmark$$

**Relation to the spectral norm**

Since $\|\mathbf{A}\|_2 = \sqrt{\lambda_0} \approx 9.53$, we see that $\|\mathbf{A}\|_F \approx 9.54 > \|\mathbf{A}\|_2$, in agreement with the general inequality

$$\|\mathbf{A}\|_2 \leq \|\mathbf{A}\|_F \leq \sqrt{r}\, \|\mathbf{A}\|_2$$

where $r = \operatorname{rank}(\mathbf{A})$. The first equality holds if and only if $\mathbf{A}$ has rank at most 1, and the second holds if and only if all nonzero singular values are equal.
:::

::::{exercise} Orthogonal Projections Shrink the Frobenius Norm
:label: exer-ch11-proj-frobenius-shrink

Let $\mathbf{A} \in \mathbb{C}^{m \times n}$, and let $\mathbf{P} \in \mathbb{C}^{n \times n}$ be an orthogonal projection matrix (that is, $\mathbf{P}^2 = \mathbf{P}$ and $\mathbf{P}^* = \mathbf{P}$).

Using the trace definition of the Frobenius norm
$$
\|\mathbf{A}\|_F^2 = \operatorname{tr}(\mathbf{A}^* \mathbf{A}),
$$
prove that

$$
\|\mathbf{A}\mathbf{P}\|_F \leq \|\mathbf{A}\|_F,
$$

and state the condition for equality.

:::{solution} exer-ch11-proj-frobenius-shrink
:class: dropdown

**Expanding the difference.**

Compute

$$
\|\mathbf{A}\|_F^2 - \|\mathbf{A}\mathbf{P}\|_F^2
= \operatorname{tr}(\mathbf{A}^* \mathbf{A}) - \operatorname{tr}((\mathbf{A}\mathbf{P})^*(\mathbf{A}\mathbf{P}))
= \operatorname{tr}(\mathbf{A}^* \mathbf{A}) - \operatorname{tr}(\mathbf{P}^* \mathbf{A}^* \mathbf{A} \mathbf{P}).
$$

By $\mathbf{P}^* = \mathbf{P}$ and the cyclic property of the trace $\operatorname{tr}(\mathbf{X}\mathbf{Y}\mathbf{Z}) = \operatorname{tr}(\mathbf{Z}\mathbf{X}\mathbf{Y})$,

$$
\operatorname{tr}(\mathbf{P} \mathbf{A}^* \mathbf{A} \mathbf{P}) = \operatorname{tr}(\mathbf{P}^2 \mathbf{A}^* \mathbf{A}) = \operatorname{tr}(\mathbf{P} \mathbf{A}^* \mathbf{A}),
$$

where the last step uses $\mathbf{P}^2 = \mathbf{P}$. Hence

$$
\|\mathbf{A}\|_F^2 - \|\mathbf{A}\mathbf{P}\|_F^2
= \operatorname{tr}(\mathbf{A}^* \mathbf{A}) - \operatorname{tr}(\mathbf{P} \mathbf{A}^* \mathbf{A})
= \operatorname{tr}\!\bigl((\mathbf{I} - \mathbf{P}) \mathbf{A}^* \mathbf{A}\bigr).
$$

**Recognizing a nonnegative structure.**

Using the cyclic property again together with $(\mathbf{I}-\mathbf{P})^2 = \mathbf{I}-\mathbf{P}$,

$$
\operatorname{tr}\!\bigl((\mathbf{I}-\mathbf{P}) \mathbf{A}^* \mathbf{A}\bigr)
= \operatorname{tr}\!\bigl(\mathbf{A} (\mathbf{I}-\mathbf{P}) \mathbf{A}^*\bigr)
= \operatorname{tr}\!\bigl(\mathbf{A} (\mathbf{I}-\mathbf{P})^2 \mathbf{A}^*\bigr)
= \operatorname{tr}\!\Bigl(\mathbf{A}(\mathbf{I}-\mathbf{P})\,\bigl(\mathbf{A}(\mathbf{I}-\mathbf{P})\bigr)^*\Bigr)
= \bigl\|\mathbf{A}(\mathbf{I}-\mathbf{P})\bigr\|_F^2 \geq 0.
$$

Therefore

$$
\|\mathbf{A}\|_F^2 - \|\mathbf{A}\mathbf{P}\|_F^2 = \|\mathbf{A}(\mathbf{I}-\mathbf{P})\|_F^2 \geq 0,
$$

that is, $\underline{\|\mathbf{A}\mathbf{P}\|_F \leq \|\mathbf{A}\|_F}$.

**The condition for equality.**

Equality holds $⟺$ $\|\mathbf{A}(\mathbf{I}-\mathbf{P})\|_F = 0$ $⟺$ $\mathbf{A}(\mathbf{I}-\mathbf{P}) = \mathbf{0}$ $⟺$ $\mathbf{A} = \mathbf{A}\mathbf{P}$, that is, $\ker(\mathbf{P}) \subseteq \ker(\mathbf{A})$; equivalently, the conjugate transpose $\text{row}_i(\mathbf{A})^{*}$ of every row of $\mathbf{A}$ lies in the range $\operatorname{ran}(\mathbf{P})$ of $\mathbf{P}$.

:::
::::

:::{prf:lemma} Frobenius Pythagorean Decomposition for Orthogonal Projections
:label: lem-frobenius-pythagoras

Let $\mathbf{A}\in\mathbb{C}^{m\times n}$, and let $\mathbf{P}\in\mathbb{C}^{n\times n}$ be an orthogonal projection matrix. Then

$$\|\mathbf{A}\|_F^2 = \|\mathbf{A}\mathbf{P}\|_F^2 + \|\mathbf{A}(\mathbf{I}-\mathbf{P})\|_F^2$$

The contraction inequality ({ref}`exer-ch11-proj-frobenius-shrink`) is the corollary obtained by discarding the nonnegative term $\|\mathbf{A}(\mathbf{I}-\mathbf{P})\|_F^2 \geq 0$ on the right.
:::

:::{prf:proof} Frobenius Pythagorean Decomposition for Orthogonal Projections
:label: prf-frobenius-pythagoras

The solution of {ref}`exer-ch11-proj-frobenius-shrink` has already derived $\|\mathbf{A}\|_F^2 - \|\mathbf{A}\mathbf{P}\|_F^2 = \|\mathbf{A}(\mathbf{I}-\mathbf{P})\|_F^2$; moving the term to the other side gives the result. $\square$
:::

::::{exercise} Expanding the Frobenius Norm in an Orthonormal Basis
:label: exer-ch11-frobenius-onb

Let $\mathbf{M} \in \mathbb{C}^{m \times n}$, and let $\{|\mathbf{f}_j⟩\}_{j=0}^{n-1}$ be any orthonormal basis of $\mathbb{C}^n$.

Using the trace definition of the Frobenius norm $\|\mathbf{M}\|_F^2 = \operatorname{tr}(\mathbf{M}^*\mathbf{M})$, prove that

$$
\|\mathbf{M}\|_F^2 = \sum_{j=0}^{n-1} \|\mathbf{M}|\mathbf{f}_j⟩\|^2.
$$

Then explain why this result does not depend on the choice of orthonormal basis.

:::{solution} exer-ch11-frobenius-onb
:class: dropdown

**Rewriting the trace as an expansion in an arbitrary orthonormal basis.**

Let $\mathbf{F} = \begin{bmatrix} |\mathbf{f}_0⟩ & \cdots & |\mathbf{f}_{n-1}⟩ \end{bmatrix}$ be the matrix whose columns are $\{|\mathbf{f}_j⟩\}$. Since the columns form an orthonormal basis, $\mathbf{F}$ is a unitary matrix, that is, $\mathbf{F}^*\mathbf{F} = \mathbf{F}\mathbf{F}^* = \mathbf{I}$.

Using unitary invariance and the cyclic property of the trace:

$$
\|\mathbf{M}\|_F^2 = \operatorname{tr}(\mathbf{M}^*\mathbf{M})
= \operatorname{tr}(\mathbf{F}\mathbf{F}^*\mathbf{M}^*\mathbf{M})
= \operatorname{tr}(\mathbf{F}^*\mathbf{M}^*\mathbf{M}\mathbf{F})
= \operatorname{tr}((\mathbf{M}\mathbf{F})^*(\mathbf{M}\mathbf{F})).
$$

**Column-wise decomposition.**

Column $j$ of $\mathbf{M}\mathbf{F}$ is exactly $\mathbf{M}|\mathbf{f}_j⟩$, so

$$
\operatorname{tr}((\mathbf{M}\mathbf{F})^*(\mathbf{M}\mathbf{F})) = \sum_{j=0}^{n-1} ⟨  \mathbf{f}_j | \mathbf{M}^* \mathbf{M} | \mathbf{f}_j ⟩ = \sum_{j=0}^{n-1} \|\mathbf{M}|\mathbf{f}_j⟩\|^2.
$$

Therefore

$$
\underline{\|\mathbf{M}\|_F^2 = \sum_{j=0}^{n-1} \|\mathbf{M}|\mathbf{f}_j⟩\|^2.}
$$

**Independence of the choice of basis.**

For any two orthonormal bases $\{|\mathbf{f}_j⟩\}$ and $\{|\mathbf{g}_j⟩\}$, the derivation above gives the same quantity $\operatorname{tr}(\mathbf{M}^*\mathbf{M})$, so the result does not depend on the choice of basis. In particular, taking the standard basis $\{|\mathbf{e}_j⟩\}$ recovers the familiar entrywise sum-of-squares formula:

$$
\|\mathbf{M}\|_F^2 = \sum_{j=0}^{n-1}\sum_{i=0}^{m-1} |M_{ij}|^2.
$$

:::
::::

#### The Nuclear Norm (Trace Norm)

:::{prf:definition} Nuclear Norm
:label: def-nuclear-norm

The **nuclear norm** (also called the **trace norm**) of a matrix $\mathbf{A} \in \mathbb{C}^{m \times n}$ is defined as

$$\|\mathbf{A}\|_* := \sum_{i=0}^{r-1} \sigma_i$$

that is, the sum of all singular values.
:::

**Applications.** The nuclear norm plays a central role in **matrix completion** and **low-rank optimization** problems. It is a **convex relaxation** of the rank of a matrix—the rank function $\text{rank}(\mathbf{A})$ is nonconvex, whereas the nuclear norm $\|\mathbf{A}\|_*$ is convex and can serve as a surrogate for the rank in optimization.

The following is an **idealized noise-free matrix completion model**. The Netflix Prize competition spurred regularized low-rank factor models for data with missing entries; its methods should not be equated directly with the exact nuclear-norm completion constraint below:

$$\min_{\mathbf{X}} \|\mathbf{X}\|_* \quad \text{subject to } \mathbf{X}_{ij} = \mathbf{M}_{ij} \text{ for observed entries}$$

**Relations with other norms**:

:::{prf:theorem} Chain of Norm Inequalities
:label: thm-norm-inequalities

Let $\mathbf{A} \in \mathbb{C}^{m \times n}$ have rank $r$. Then

$$\|\mathbf{A}\|_2 \leq \|\mathbf{A}\|_F \leq \|\mathbf{A}\|_* \leq \sqrt{r} \|\mathbf{A}\|_F$$

More precisely,

$$\sigma_0 \leq \sqrt{\sum_{i=0}^{r-1}\sigma_i^2} \leq \sum_{i=0}^{r-1}\sigma_i \leq \sqrt{r}\sqrt{\sum_{i=0}^{r-1}\sigma_i^2}$$
:::

```{prf:proof}
1. **The first inequality** $\|\mathbf{A}\|_2 \leq \|\mathbf{A}\|_F$:

$$\sigma_0^2 \leq \sum_{i=0}^{r-1}\sigma_i^2 $$

2. **The second inequality** $\|\mathbf{A}\|_F \leq \|\mathbf{A}\|_*$:

Since $\sigma_i \geq 0$,

$$\sum_{i=0}^{r-1}\sigma_i^2 \leq  (\sum_{i=0}^{r-1}\sigma_i)^2   $$

3. **The third inequality** $\|\mathbf{A}\|_* \leq \sqrt{r} \|\mathbf{A}\|_F$:

By Cauchy-Schwarz,

$$\sum_{i=0}^{r-1}\sigma_i = \sum_{i=0}^{r-1} 1 \cdot \sigma_i \leq \sqrt{\sum_{i=0}^{r-1}1^2} \sqrt{\sum_{i=0}^{r-1}\sigma_i^2} = \sqrt{r} \|\mathbf{A}\|_F$$

```

:::{prf:example} Three Norms of a Rank-One Matrix
:label: ex-three-norms-rank1

Consider the rank-one matrix

$$\mathbf{A} = \begin{bmatrix} 2 & 4 \\ 1 & 2 \\ 3 & 6 \end{bmatrix} = \begin{bmatrix}2\\1\\3\end{bmatrix}\begin{bmatrix}1&2\end{bmatrix}$$

Compute $\mathbf{A}^{\top}\mathbf{A}$:

$$\mathbf{A}^{\top}\mathbf{A} = \begin{bmatrix}1&2\end{bmatrix}^{\top}(4+1+9)\begin{bmatrix}1&2\end{bmatrix} = 14\begin{bmatrix}1&2\\2&4\end{bmatrix}$$

The eigenvalues are $\lambda_0 = 70$ and $\lambda_1 = 0$, the only nonzero singular value is $\sigma_0 = \sqrt{70}$, and the three norms are

$$\|\mathbf{A}\|_2 = \sqrt{70} \approx 8.37, \qquad \|\mathbf{A}\|_F = \sqrt{70} \approx 8.37, \qquad \|\mathbf{A}\|_* = \sqrt{70} \approx 8.37$$

For a rank-one matrix, since there is only one nonzero singular value, the three norms are always equal:

$$\|\mathbf{A}\|_2 = \|\mathbf{A}\|_F = \|\mathbf{A}\|_* = \sigma_0$$

This is exactly the condition for equality in the first inequality $\|\mathbf{A}\|_2 \leq \|\mathbf{A}\|_F$.
:::

:::{prf:example} The Case of Equal Singular Values
:label: ex-three-norms-scalar-identity

Consider the matrix $\mathbf{A} = c\mathbf{I}_n$ ($c > 0$). Its $n$ singular values are all $\sigma_i = c$, and the three norms are

$$\|\mathbf{A}\|_2 = c, \qquad \|\mathbf{A}\|_F = \sqrt{n}\,c, \qquad \|\mathbf{A}\|_* = nc$$

Checking the chain of inequalities:

$$c \leq \sqrt{n}\,c \leq \sqrt{n} \cdot \sqrt{n}\,c = nc \quad \checkmark$$

This example shows that when all nonzero singular values are equal, the second equality in $\|\mathbf{A}\|_F = \sqrt{r}\,\|\mathbf{A}\|_2$ holds exactly.
:::

#### Summary of the Geometric and Physical Meaning of the Norms

| Norm | Definition | Formula in singular values | Geometric/physical meaning |
|------|------|-----------|--------------|
| **Spectral norm** $\|\mathbf{A}\|_2$ | $\max_{\|\mathbf{x}\|=1}\|\mathbf{A}\mathbf{x}\|$ | $\sigma_0$ | the **longest principal axis** of the ellipsoid; the maximum amplification factor |
| **Frobenius norm** $\|\mathbf{A}\|_F$ | $\sqrt{\sum_{ij}|a_{ij}|^2}$ | $\sqrt{\sum_i \sigma_i^2}$ | the **square root of the total energy** of the matrix; the square root of the sum of the squared lengths of all principal axes |
| **Nuclear norm** $\|\mathbf{A}\|_*$ | $\sum_i \sigma_i$ | $\sum_i \sigma_i$ | a **convex surrogate for the rank** of the matrix; the sum of the lengths of the principal axes |

```{important}
:label: note-norm-applications

**Typical applications of the different norms**:

1. **Spectral norm**:
   - the condition number $\kappa(\mathbf{A}) = \|\mathbf{A}\|_2 \|\mathbf{A}^{-1}\|_2$ in numerical stability analysis
   - the convergence rate of iterative algorithms (power iteration for eigenvalues)
   - the maximum gain in signal processing

2. **Frobenius norm**:
   - measuring matrix approximation error (see the Eckart–Young theorem in the next subsection)
   - the total sum of squares in statistics
   - the mean squared error (MSE) in image processing

3. **Nuclear norm**:
   - matrix completion (recommender systems, collaborative filtering)
   - low-rank matrix recovery (compressed sensing)
   - robust principal component analysis (Robust PCA)
```

---

### 11.2.2 The Eckart–Young Theorem: Best Low-Rank Approximation

One of the most important applications of the SVD is to provide the **best low-rank approximation** of a matrix. This result was proved by Carl Eckart and Gale Young in 1936 and is now called the **Eckart–Young theorem** (or the **Eckart–Young–Mirsky theorem**, which includes Leon Mirsky's 1960 extension to unitarily invariant norms).


Given a matrix $\mathbf{A} \in \mathbb{C}^{m \times n}$ of rank $r$, we wish to find a **matrix $\mathbf{B}$ of rank at most $k$** ($k < r$) such that $\mathbf{B}$ is "closest" to $\mathbf{A}$.

**Mathematical formulation**:

$$\min_{\mathbf{B} \in \mathbb{C}^{m \times n},\; \operatorname{rank}(\mathbf{B}) \leq k} \|\mathbf{A} - \mathbf{B}\|$$

The crux of the problem is:
1. **What does "closest" mean?** That is, which norm should serve as the measure of distance?
2. **What is the optimal solution?** Can it be constructed explicitly?

The Eckart–Young theorem answers both questions perfectly.



#### The Frobenius-Norm Version of the Eckart–Young Theorem

:::{prf:theorem} Eckart–Young Theorem (Frobenius-Norm Version)
:label: thm-eckart-young-frobenius

Let $\mathbf{A} \in \mathbb{C}^{m \times n}$ have SVD

$$\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{*} = \sum_{i=0}^{r-1} \sigma_i |\mathbf{u}_i⟩⟨ \mathbf{v}_i|$$

where $r = \operatorname{rank}(\mathbf{A})$ and $\sigma_0 \geq \sigma_1 \geq \cdots \geq \sigma_{r-1} > 0$. For any $0 \leq k < r$, define the **truncated SVD**

$$\mathbf{A}_k := \sum_{i=0}^{k-1} \sigma_i |\mathbf{u}_i⟩⟨ \mathbf{v}_i|$$

Then $\mathbf{A}_k$ is the best approximation of $\mathbf{A}$ in the **Frobenius norm** among all matrices of rank at most $k$:

$$\mathbf{A}_k \in \arg\min_{\mathbf{B} \in \mathbb{C}^{m \times n},\; \operatorname{rank}(\mathbf{B}) \leq k} \|\mathbf{A} - \mathbf{B}\|_F$$

and the minimum error is

$$\min_{\operatorname{rank}(\mathbf{B}) \leq k} \|\mathbf{A} - \mathbf{B}\|_F = \|\mathbf{A} - \mathbf{A}_k\|_F = \sqrt{\sum_{i=k}^{r-1} \sigma_i^2}$$
:::

A complete proof of the general case requires the von Neumann trace inequality or the Mirsky inequality and is beyond the scope of this book; still, it is worth marking clearly exactly where the gap in the proof lies and which tools this section has already prepared.

:::{prf:remark} A Road Map for the Proof of the Frobenius Version
:label: rmk-eckart-young-frobenius-proof

The three tools of this section have already carried the proof to its last step. The Pythagorean decomposition ({prf:ref}`lem-frobenius-pythagoras`) gives $\|\mathbf{A}-\mathbf{A}\boldsymbol{\Pi}\|_F^2 = \|\mathbf{A}\|_F^2 - \|\mathbf{A}\boldsymbol{\Pi}\|_F^2$, and the orthonormal-basis expansion ({ref}`exer-ch11-frobenius-onb`) writes $\|\mathbf{A}\boldsymbol{\Pi}\|_F^2$ as $\sum_{j=0}^{k-1}\|\mathbf{A}\mathbf{f}_j\|^2$ (where $\{\mathbf{f}_j\}$ is an orthonormal basis of the range of the projection). The "optimality of projection-type approximations" thus reduces to the extremal problem

$$\max_{\{\mathbf{f}_j\} \text{ orthonormal}} \sum_{j=0}^{k-1}\|\mathbf{A}\mathbf{f}_j\|^2 = \sum_{i=0}^{k-1}\sigma_i^2$$

—this is the generalization of the extremal characterization of singular values ({prf:ref}`thm-singular-value-extremal`) from a single direction to a $k$-dimensional subspace (the Ky Fan maximum principle). Supplying this step, and then proving that an optimal rank-$k$ approximation can always be taken in the projection form $\mathbf{A}\boldsymbol{\Pi}$, gives the complete proof; these two details are left for further reading.
:::

#### The Spectral-Norm Version of the Eckart–Young Theorem

:::{prf:theorem} Eckart–Young Theorem (Spectral-Norm Version)
:label: thm-eckart-young-spectral

In the setting of Theorem {prf:ref}`thm-eckart-young-frobenius`, $\mathbf{A}_k$ is also the best rank-$k$ approximation in the **spectral norm**:

$$\mathbf{A}_k \in \arg\min_{\mathbf{B} \in \mathbb{C}^{m \times n},\; \operatorname{rank}(\mathbf{B}) \leq k} \|\mathbf{A} - \mathbf{B}\|_2$$

and the minimum error is

$$\min_{\text{rank}(\mathbf{B}) \leq k} \|\mathbf{A} - \mathbf{B}\|_2 = \|\mathbf{A} - \mathbf{A}_k\|_2 = \sigma_k$$

that is, the $k$-th singular value (0-based indexing, starting from $\sigma_0$).
:::

```{prf:proof}
1. **Upper bound ($\mathbf{A}_k$ attains $\sigma_k$)**: since $\mathbf{A} - \mathbf{A}_k = \sum_{i=k}^{r-1} \sigma_i |\mathbf{u}_i⟩⟨ \mathbf{v}_i|$, its largest singular value is $\sigma_k$, and therefore

$$\|\mathbf{A} - \mathbf{A}_k\|_2 = \sigma_k$$


2. **Proving that $\mathbf{A}_k$ is optimal**:

- Since $\operatorname{rank}(\mathbf{B}) \leq k$, its null space satisfies $\dim(\ker(\mathbf{B})) \geq n - k$.
- Consider the subspace spanned by the first $k+1$ right singular vectors, $W := \operatorname{span}\{|\mathbf{v}_0⟩, \ldots, |\mathbf{v}_k⟩\}$, which has $\dim(W) = k+1$.
- By the dimension theorem, $\dim(W) + \dim(\ker(\mathbf{B})) \geq (k+1)+(n-k) = n+1 > n$, so $W \cap \ker(\mathbf{B}) \neq \{\mathbf{0}\}$, and there must exist a unit vector $|\mathbf{w}⟩ = \sum_{i=0}^{k} c_i \mathbf{v}_i  \in W \cap \ker(\mathbf{B})$.



$$\underline{\|\mathbf{A} - \mathbf{B}\|_2 \geq \|(\mathbf{A}-\mathbf{B})\mathbf{w}\| = \|\mathbf{A}\mathbf{w}\| = \|\sum_{i=0}^{k}c_i\sigma_i\mathbf{u}_i\| \geq \sigma_k}$$

```

**Comparing the two norm versions**:

| Norm | Minimum error | Geometric/statistical meaning |
|------|---------|--------------|
| Frobenius norm | $\sqrt{\sum_{i=k}^{r-1}\sigma_i^2}$ | **the square root of the total squared error**; the square root of the discarded Frobenius energy |
| Spectral norm | $\sigma_k$ | **the maximum error**; the largest of the discarded singular values ($\sigma_k$) |

**An important observation.** The Frobenius norm takes into account **the contributions of all discarded singular values**, while the spectral norm cares only about **the largest discarded singular value**. In practice:
- if the singular values **decay rapidly** (for example $\sigma_i \sim e^{-i}$), both norms give a very good approximation
- if the singular values **decay slowly** (for example $\sigma_i \sim i^{-1}$), more terms may need to be kept

#### The Practical Meaning of Low-Rank Structure

The Eckart–Young theorem reveals the central place of the SVD in **data science**:

**1. Image compression**

An $m \times n$ grayscale image can be regarded as a matrix $\mathbf{A} \in \mathbb{R}^{m \times n}$. Storing it in full requires $mn$ numbers. But if we keep the first $k$ singular values, we only need to store

$$k(m + n + 1) \quad \text{numbers}$$

(namely $k$ vectors $\mathbf{u}_i$ ($km$ numbers in all), $k$ vectors $\mathbf{v}_i$ ($kn$ numbers in all), and $k$ singular values)

**Compression ratio**:

$$\frac{k(m+n+1)}{mn} \approx \frac{k(m+n)}{mn}$$

If $k \ll \min(m,n)$, the compression is substantial. For example, with $m=n=1000$ and $k=50$:

$$\text{compression ratio} = \frac{50 \times 2000}{1000000} = 0.1 = 10\%$$

That is, only 10% of the space of the original data is needed!

**2. Denoising**

Real measured data $\mathbf{A}$ usually contain noise:

$$\mathbf{A} = \mathbf{A}_{\text{signal}} + \mathbf{E}_{\text{noise}}$$

If the signal has low-rank structure, the signal-to-noise ratio is high enough, and the main signal modes are separable from the noise, then the large singular values may be dominated by the signal; small singular values may also contain important signal.

The truncated SVD $\mathbf{A}_k$ is **singular-mode filtering**, which in general is not the same as low-pass filtering. Under the low-rank signal assumption above:
- keep the large singular values (possibly dominated by the signal)
- discard the small singular values (possibly dominated by noise, but weak signals may also be lost)

**3. Latent Semantic Analysis (LSA)**

In text mining, the document-term matrix $\mathbf{A}$ is usually **high-dimensional and sparse** ($m$ documents, $n$ terms; $mn$ is large but most entries are 0).

The truncated SVD maps the high-dimensional term space to a low-dimensional "concept space":
- **Left singular vectors** $\mathbf{u}_i$: the coordinates of documents in concept space
- **Right singular vectors** $\mathbf{v}_i$: the coordinates of terms in concept space
- **Singular values** $\sigma_i$: the "importance" of the $i$-th concept

Keeping the first $k$ concepts (typically $k \sim 100-300$) makes it possible to:
- find semantically similar documents (even if they share no terms)
- reduce dimensionality and improve search efficiency
- remove interference from synonyms and polysemous words

:::{prf:example} Low-Rank Approximation of an Image
:label: ex-low-rank-image-approx

Consider a $4 \times 4$ "image" matrix:

$$\mathbf{A} = \begin{bmatrix}
4 & 0 & 0 & 0 \\
0 & 3 & 0 & 0 \\
0 & 0 & 2 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}$$

This is a diagonal matrix, so its SVD is the matrix itself, with singular values $\sigma_0=4,\ \sigma_1=3,\ \sigma_2=2,\ \sigma_3=1$.

**Rank-one approximation** ($k=1$):

$$\mathbf{A}_1 = \sigma_0 \mathbf{u}_0\mathbf{v}_0^{\top} = 4 \cdot \mathbf{e}_0\mathbf{e}_0^{\top} = \begin{bmatrix}
4 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0
\end{bmatrix}$$

Error:

$$\|\mathbf{A} - \mathbf{A}_1\|_F = \sqrt{3^2+2^2+1^2} = \sqrt{14} \approx 3.74, \qquad \|\mathbf{A} - \mathbf{A}_1\|_2 = \sigma_1 = 3$$

**Rank-two approximation** ($k=2$):

$$\mathbf{A}_2 = \begin{bmatrix}
4 & 0 & 0 & 0 \\
0 & 3 & 0 & 0 \\
0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0
\end{bmatrix}$$

Error:

$$\|\mathbf{A} - \mathbf{A}_2\|_F = \sqrt{2^2+1^2} = \sqrt{5} \approx 2.24, \qquad \|\mathbf{A} - \mathbf{A}_2\|_2 = \sigma_2 = 2$$

**Energy retained.** Define the **relative energy** as

$$\frac{\sum_{i=0}^{k-1}\sigma_i^2}{\sum_{i=0}^{r-1}\sigma_i^2} = \frac{\|\mathbf{A}_k\|_F^2}{\|\mathbf{A}\|_F^2}$$

In this example $\|\mathbf{A}\|_F^2 = 16+9+4+1 = 30$, and the fraction of energy retained at each truncation level is:

| $k$ | Energy retained | Percentage |
|:---:|:-------:|:------:|
| 1 | $16/30$ | $\approx 53.3\%$ |
| 2 | $25/30$ | $\approx 83.3\%$ |
| 3 | $29/30$ | $\approx 96.7\%$ |

If the singular values decay quickly, a few terms suffice to retain most of the Frobenius energy.
:::

For a fixed $m\times n$ image, the Frobenius energy is $\|\mathbf{A}\|_F^2$ and the mean squared error is $\|\mathbf{A}-\mathbf{A}_k\|_F^2/(mn)$. Minimizing the error norm, its square, or the mean squared error gives the same optimal matrix, although the values differ.


::::{exercise} Verifying the Optimality in the Eckart–Young Theorem
:label: exer-ch11-eckart-young-optimality

Consider the $4 \times 4$ diagonal matrix (whose SVD is already diagonal):

$$\mathbf{A} = \begin{bmatrix}
4 & 0 & 0 & 0 \\
0 & 3 & 0 & 0 \\
0 & 0 & 2 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix} = \text{diag}(4, 3, 2, 1)$$

The singular values of this matrix are $\sigma_0=4, \sigma_1=3, \sigma_2=2, \sigma_3=1$, and the singular vectors are all standard basis vectors $\mathbf{e}_i$.

**(a)** Write down the best rank-one approximation $\mathbf{A}_1$ given by the Eckart–Young theorem, and compute $\|\mathbf{A}-\mathbf{A}_1\|_F$.

**(b)** Consider the following rank-one candidates. Compute the Frobenius distance from each of them to $\mathbf{A}$ and compare with $\mathbf{A}_1$:

$$\mathbf{B}_1 = \begin{bmatrix}3.5&0&0&0\\0&0&0&0\\0&0&0&0\\0&0&0&0\end{bmatrix}, \qquad
\mathbf{B}_2 = c\begin{bmatrix}0.5&0.5&0&0\\0.5&0.5&0&0\\0&0&0&0\\0&0&0&0\end{bmatrix}$$

where $\mathbf{B}_2$ corresponds to the direction $\mathbf{w} = \frac{1}{\sqrt{2}}[1,1,0,0]^\top$; optimize over $c$ before comparing.

**(c)** Explain why the following matrix is **not** a rank-one matrix, even though it "seems" to be a good approximation as well:

$$\mathbf{B}_{\text{wrong}} = \begin{bmatrix}
3 & 0 & 0 & 0 \\
0 & 3 & 0 & 0 \\
0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0
\end{bmatrix}$$


:::{solution} exer-ch11-eckart-young-optimality
:class: dropdown

**(a)** The Eckart–Young theorem keeps the largest singular value $\sigma_0 = 4$, and the corresponding best rank-one approximation is

$$\mathbf{A}_1 = \sigma_0 \mathbf{u}_0 \mathbf{v}_0^\top = 4\,\mathbf{e}_0\mathbf{e}_0^\top =
\begin{bmatrix}4&0&0&0\\0&0&0&0\\0&0&0&0\\0&0&0&0\end{bmatrix}$$

The error matrix retains the remaining three singular values:

$$\mathbf{A} - \mathbf{A}_1 = \begin{bmatrix}0&0&0&0\\0&3&0&0\\0&0&2&0\\0&0&0&1\end{bmatrix}$$

$$\|\mathbf{A} - \mathbf{A}_1\|_F = \sqrt{\sigma_1^2 + \sigma_2^2 + \sigma_3^2} = \sqrt{9+4+1} = \sqrt{14} \approx 3.74$$

This also verifies the error formula of the Eckart–Young theorem: $\|\mathbf{A}-\mathbf{A}_k\|_F^2 = \sum_{i=k}^{r-1}\sigma_i^2$.

**(b)** For $\mathbf{B}_1 = 3.5\,\mathbf{e}_0\mathbf{e}_0^\top$:

$$\mathbf{A} - \mathbf{B}_1 = \begin{bmatrix}0.5&0&0&0\\0&3&0&0\\0&0&2&0\\0&0&0&1\end{bmatrix}$$

$$\|\mathbf{A}-\mathbf{B}_1\|_F = \sqrt{0.25 + 9 + 4 + 1} = \sqrt{14.25} > \sqrt{14}$$

For $\mathbf{B}_2 = c\,\mathbf{w}\mathbf{w}^\top$ with $\mathbf{w} = \frac{1}{\sqrt{2}}[1,1,0,0]^\top$, we optimize $c$ by expanding:

$$\|\mathbf{A} - c\,\mathbf{w}\mathbf{w}^\top\|_F^2 = \|\mathbf{A}\|_F^2 - 2c⟨  \mathbf{A}, \mathbf{w}\mathbf{w}^\top⟩ + c^2\|\mathbf{w}\mathbf{w}^\top\|_F^2$$

Computing $⟨  \mathbf{A}, \mathbf{w}\mathbf{w}^\top⟩ = \mathbf{w}^\top \mathbf{A}\,\mathbf{w} = \frac{1}{2}(4+3) = 3.5$ and $\|\mathbf{w}\mathbf{w}^\top\|_F = \|\mathbf{w}\|^2 = 1$, the optimal value is $c^* = 3.5$, and the minimum error is

$$\|\mathbf{A} - \mathbf{B}_2\|_F^2 = \|\mathbf{A}\|_F^2 - (c^*)^2 = 30 - 12.25 = 17.75 > 14$$

Both candidates have errors exceeding $\sqrt{14}$, in agreement with the prediction of the Eckart–Young theorem.

**(c)** $\mathbf{B}_{\text{wrong}}$ has **rank 2**, not 1, because it has two linearly independent nonzero columns ($[3,0,0,0]^\top$ and $[0,3,0,0]^\top$). It can be decomposed as

$$\mathbf{B}_{\text{wrong}} = 3\,\mathbf{e}_0\mathbf{e}_0^\top + 3\,\mathbf{e}_1\mathbf{e}_1^\top$$

This is a sum of two rank-one matrices, hence of rank 2. The comparison in the Eckart–Young theorem must be restricted to the class of matrices of **the same rank**; comparing across ranks is unfair—a rank-two approximation can indeed do better than the best rank-one approximation (for example $\|\mathbf{A}-\mathbf{B}_{\text{wrong}}\|_F = \sqrt{1+4+1} = \sqrt{6} < \sqrt{14}$), but that is exactly what one would expect.

:::
::::

:::{admonition} §11.2 Summary
:class: tip

The question this section answered is: in what sense, exactly, is the low-rank approximation given by the truncated SVD "the best"? To answer it, we first clarified three matrix norms: the spectral norm $\|\mathbf{A}\|_2 = \sigma_0$ captures the direction of maximal stretching and is the longest principal axis of the ellipsoid; the Frobenius norm $\|\mathbf{A}\|_F = \sqrt{\sum_i\sigma_i^2}$ accumulates the energy of all principal axes; and the nuclear norm $\|\mathbf{A}\|_* = \sum_i\sigma_i$ is the convex relaxation of the rank and plays a key role in low-rank matrix recovery. All three norms are completely determined by the singular values, which highlights the status of singular values as the "essential descriptors" of a matrix.

The Eckart–Young theorem is the core of this section: in the spectral norm or the Frobenius norm, among all matrices of rank at most $k$, the one closest to $\mathbf{A}$ is precisely the truncated SVD $\mathbf{A}_k = \sum_{i=0}^{k-1}\sigma_i\mathbf{u}_i\mathbf{v}_i^{*}$, and the approximation error is given exactly by the discarded singular values. The beauty of this conclusion is that the optimal solution requires no optimization search at all—the SVD gives the answer directly.

With the best low-rank approximation as a tool, the core problem of data science in §11.3—how to extract the most important structure from high-dimensional data—acquires a precise mathematical formulation.
:::

## 11.3 Application 1: Principal Component Analysis (PCA)

The Eckart–Young theorem of §11.2 ({prf:ref}`thm-eckart-young-frobenius`) is a purely matrix-optimization result: among all matrices of rank at most $k$, the truncated SVD $\tilde{\mathbf{X}}_k$ is the one nearest to the original matrix. This conclusion by itself carries no statistical meaning. But as soon as the matrix **holds data**—each row a sample, each column a feature—the "best low-rank approximation" immediately takes on a statistical identity: it is precisely **principal component analysis (PCA)**.

So what exactly does PCA do? The best-known answer is "find the directions of greatest variation in the data," which is the viewpoint of Hotelling (1933). This section, however, adopts a more fundamental viewpoint that fits §11.2 more closely—that of Pearson (1901): "the low-dimensional plane that best fits the data." In other words, the essence of PCA is not "finding the directions of greatest variation" but "finding the best low-dimensional projection"; the directions of greatest variation are merely a by-product of this best projection. This order matters, because it determines how we understand PCA; we return in §11.3.2 to discuss "the directions of greatest variation" as an observation.

The measure used throughout this section is **variance**. Experiment 5 (Blocks C and D) has prepared the tool: the total variance $V_{\text{total}} = \operatorname{tr}(\mathbf{S}) = \frac{1}{m-1}\|\tilde{\mathbf{X}}\|_F^2$. With it, the argument of §11.3.1 proceeds in two steps. First, the Frobenius Pythagorean decomposition guarantees that **any** orthogonal projection conserves the total variance—dimensionality reduction neither creates nor erases variance out of nothing; it only divides it between the approximation and the error. Second, the Eckart–Young theorem picks out the **best** one among all projections of the same dimension, namely the one given by the right singular vectors of the SVD. Conservation is the universal background; optimization is the real protagonist of PCA.

This section proceeds in two steps. §11.3.1 establishes the core logic (conservation + best projection); §11.3.2 first points out that the best directions of the SVD are exactly "the directions of greatest variation in the data" (an incidental observation, together with a variational characterization by variance maximization), and then, from this nature of "looking only at linear variance," marks the limits of PCA—it fails on curved manifolds. The rank-by-rank transfer of variance, and the application of PCA to real data to see it reveal cluster structure, are left to the PCA experiment notebook of this chapter for a full demonstration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Color scheme: NYU Official (Accent Mix) ---
C_BG    = "#F8F8F8"
C_GRID  = "#D6D6D6"
C_AXIS  = "#000000"
C_V1    = "#57068C"   # 0-th principal component (v0)
C_V2    = "#006385"   # 1st principal component (v1)
C_T1    = "#2AD2C9"   # best-fit plane
C_AUX   = "#AB82C5"   # data points
C_WARN  = "#FF5D47"   # perpendiculars (residuals)

# Fonts: the English edition needs no CJK font, so nothing is downloaded when _lang == 'en';
# the Chinese editions use this same block to fetch Noto Sans TC/SC where no CJK font is installed
import os, urllib.request
import matplotlib.font_manager as fm
_lang = 'en'
_cjk = ['Microsoft JhengHei', 'PingFang TC', 'Noto Sans CJK TC', 'Noto Sans TC']
_have = {f.name for f in fm.fontManager.ttflist}
if _lang != 'en' and not _have & set(_cjk):
    _font = os.path.join(os.path.expanduser('~'), '.cache', 'fonts', 'NotoSansTC.ttf')
    try:
        if not os.path.exists(_font):
            os.makedirs(os.path.dirname(_font), exist_ok=True)
            urllib.request.urlretrieve('https://github.com/google/fonts/raw/main/ofl/notosanstc/NotoSansTC%5Bwght%5D.ttf', _font + '.part')
            os.replace(_font + '.part', _font)
        fm.fontManager.addfont(_font)
        _have.add('Noto Sans TC')
    except OSError as err:
        print('Could not download the CJK font; Chinese text in figures may not display:', err)
plt.rcParams['font.family'] = [f for f in _cjk if f in _have] + ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.set_printoptions(precision=4, suppress=True, linewidth=100)

def print_header(title):
    print("=" * 60)
    print(f"  {title}")
    print("=" * 60)

def print_step(step, desc):
    print(f"\n▶ Step {step}: {desc}")
    print("-" * 40)

# ============================================================
print_header("3D Ellipsoidal Normal Distribution | PCA Best-Fit Plane")
# ============================================================

print_step(1, "Generate random points from an ellipsoidal normal distribution (off the origin, axes not aligned with the coordinate axes)")
rng = np.random.default_rng(7)

mu = np.array([3.0, 2.0, 1.0])

theta = np.deg2rad(35)
axis = np.array([1.0, 1.0, 1.0]) / np.sqrt(3)
K = np.array([[0, -axis[2], axis[1]],
              [axis[2], 0, -axis[0]],
              [-axis[1], axis[0], 0]])
R = np.eye(3) + np.sin(theta) * K + (1 - np.cos(theta)) * (K @ K)  # Rodrigues rotation
scales = np.array([3.0, 1.6, 0.3])  # standard deviations of the three ellipsoid axes (largest to smallest; axis 2 is very flat)
Sigma = R @ np.diag(scales**2) @ R.T

n = 60   # fewer points, to avoid visual clutter in 3D
X = rng.multivariate_normal(mean=mu, cov=Sigma, size=n)
print(f"  Number of samples n = {n}, center μ = {mu}")
print(f"  Standard deviations of the three ellipsoid axes (largest to smallest) = {np.round(np.sort(scales)[::-1], 3)}")

print_step(2, "Center, take the SVD, and let the first two principal components span the best-fit plane")
x_bar = X.mean(axis=0)
X_tilde = X - x_bar
U, sigma, Vt = np.linalg.svd(X_tilde, full_matrices=False)
v0, v1, v2 = Vt[0], Vt[1], Vt[2]
rho = sigma**2 / (sigma**2).sum()
print(f"  Variance share of each principal component ρ = {np.round(rho, 4)} (the first two explain {rho[0]+rho[1]:.1%} together)")
print(f"  0-th principal component v0 = {v0}")
print(f"  1st principal component v1 = {v1}")

print_step(3, "Compute the foot of the perpendicular from each point to the plane (residual along the 2nd principal component v2)")
c2 = X_tilde @ v2                       # coordinate of each point along v2 (the residual)
X_proj = X - np.outer(c2, v2)           # points projected onto the plane (feet of perpendiculars)
print(f"  Mean perpendicular residual |c2| = {np.mean(np.abs(c2)):.3f} (smaller means the points hug the plane more closely)")

print_step(4, "Plot: original 3D point cloud + perpendiculars + PCA best-fit plane (semi-transparent)")
fig = plt.figure(figsize=(7.5, 7))
ax = fig.add_subplot(111, projection='3d')
ax.set_facecolor(C_BG)

# --- best-fit plane spanned by v0, v1 (semi-transparent surface), drawn first so the perpendiculars lie on top ---
extent = 1.2 * np.max(np.abs(X_tilde @ Vt[:2].T))
s_grid, t_grid = np.meshgrid(np.linspace(-extent, extent, 12),
                              np.linspace(-extent, extent, 12))
plane_pts = (x_bar[:, None, None]
             + v0[:, None, None] * s_grid
             + v1[:, None, None] * t_grid)
ax.plot_surface(plane_pts[0], plane_pts[1], plane_pts[2],
                 color=C_T1, alpha=0.35, edgecolor='none')

# --- perpendiculars: segments from each point to its foot on the plane, to strengthen the 3D sense of depth ---
for i in range(n):
    ax.plot([X[i, 0], X_proj[i, 0]],
            [X[i, 1], X_proj[i, 1]],
            [X[i, 2], X_proj[i, 2]],
            color=C_WARN, linewidth=0.8, alpha=0.55, zorder=1)

# --- feet of perpendiculars (projected points on the plane), smaller and semi-transparent ---
ax.scatter(X_proj[:, 0], X_proj[:, 1], X_proj[:, 2],
           s=8, color=C_WARN, alpha=0.4, zorder=2)

# --- original data points ---
ax.scatter(X[:, 0], X[:, 1], X[:, 2], s=22, color=C_AUX, alpha=0.9,
           edgecolor=C_AXIS, linewidth=0.3, label='data points', zorder=3)

# --- arrows for the principal component directions ---
ax.quiver(*x_bar, *(v0 * sigma[0] / np.sqrt(n - 1) * 1.8), color=C_V1, linewidth=2.5, label='0-th principal component v0')
ax.quiver(*x_bar, *(v1 * sigma[1] / np.sqrt(n - 1) * 1.8), color=C_V2, linewidth=2.5, label='1st principal component v1')

ax.set_xlabel('x', color=C_AXIS)
ax.set_ylabel('y', color=C_AXIS)
ax.set_zlabel('z', color=C_AXIS)
ax.set_title('Ellipsoidal normal point cloud and PCA best-fit plane (with perpendiculars)', color=C_AXIS)
ax.xaxis.pane.set_facecolor(C_BG)
ax.yaxis.pane.set_facecolor(C_BG)
ax.zaxis.pane.set_facecolor(C_BG)
ax.grid(True, color=C_GRID, linewidth=0.6, alpha=0.6)
ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

### 11.3.1 From Low-Rank Approximation to Variance Decomposition

The Eckart–Young theorem of §11.2.2 ({prf:ref}`thm-eckart-young-frobenius`) is a matrix-optimization result. This section translates it into the language of statistics and clarifies the true logical order of PCA: **first comes the invariant "every projection conserves the total variance," and then Eckart–Young picks the best projection from among them.**

We continue with the centered data matrix $\tilde{\mathbf{X}} \in \mathbb{R}^{m\times n}$ of Experiment 5 (samples as rows), the sample covariance matrix $\mathbf{S} = \frac{1}{m-1}\tilde{\mathbf{X}}^{\top}\tilde{\mathbf{X}}$, and the total variance

$$V_{\text{total}} = \operatorname{tr}(\mathbf{S}) = \frac{1}{m-1}\|\tilde{\mathbf{X}}\|_F^2$$

(For readers who have not done Experiment 5, one line suffices: $\operatorname{tr}(\mathbf{S}) = \sum_j \frac{1}{m-1}\sum_i \tilde{x}_{ij}^2 = \frac{1}{m-1}\|\tilde{\mathbf{X}}\|_F^2$.) Take the SVD of $\tilde{\mathbf{X}}$ and write it as an outer-product expansion (§11.1):

$$\tilde{\mathbf{X}} = \sum_{i=0}^{r-1}\sigma_i\,\mathbf{u}_i\mathbf{v}_i^{\top} = \sum_{i=0}^{r-1}\sigma_i\,|\mathbf{u}_i\rangle\langle\mathbf{v}_i|$$

:::{prf:remark} The Dual Identity of the Outer-Product Expansion: Things Cluster by Kind, People Form Groups
:label: rem-svd-dual-embedding

Each term $\sigma_i\,\mathbf{u}_i\mathbf{v}_i^{\top}$ involves two spaces at once, corresponding exactly to the two viewpoints of a data matrix—features (columns) and samples (rows):

- $\mathbf{v}_i \in \mathbb{R}^n$ lives in the **feature space** and is a set of weights on the $n$ features—it recombines the original features into the $i$-th latent factor. Features governed by the same factor cluster together: "**things cluster by kind**" (in LSA for text analysis, $\mathbf{v}_i$ is exactly the set of weights that clusters terms into topics).
- $\mathbf{u}_i \in \mathbb{R}^m$ lives in the **sample space**, and $\sigma_i\mathbf{u}_i = \tilde{\mathbf{X}}\mathbf{v}_i$ is exactly the coordinates of the $m$ samples along the $i$-th principal axis (the principal component scores). Samples with similar scores on this axis are placed in the same group: "**people form groups**."

In other words, the same index $i$ ties "how the features cluster ($\mathbf{v}_i$)" and "how the samples group ($\mathbf{u}_i$)" to the same latent factor, with $\sigma_i$ measuring the strength of that factor. This row/column duality is the underlying structure shared by all subsequent applications of the SVD (the sample scatter of PCA, the term–document dual embedding of LSA, the user–item factorization of recommender systems).
:::

#### Step 1: Every Orthogonal Projection Conserves the Total Variance

Dimensionality reduction means projecting the data onto some low-dimensional subspace. Take **any** orthogonal projection matrix $\boldsymbol{\Pi}\in\mathbb{R}^{n\times n}$ (not yet specified); the projected data are $\tilde{\mathbf{X}}\boldsymbol{\Pi}$ and the residual is $\tilde{\mathbf{X}}(\mathbf{I}-\boldsymbol{\Pi})$. How do the variances of these two divide the total variance?

:::{prf:theorem} Conservation of Total Variance
:label: thm-variance-conservation

Let $\boldsymbol{\Pi}\in\mathbb{R}^{n\times n}$ be any orthogonal projection matrix, and denote the approximation variance and the error variance by

$$V_{\text{approx}}(\boldsymbol{\Pi}) = \frac{1}{m-1}\|\tilde{\mathbf{X}}\boldsymbol{\Pi}\|_F^2, \qquad V_{\text{err}}(\boldsymbol{\Pi}) = \frac{1}{m-1}\|\tilde{\mathbf{X}}(\mathbf{I}-\boldsymbol{\Pi})\|_F^2$$

Then

$$V_{\text{total}} = V_{\text{approx}}(\boldsymbol{\Pi}) + V_{\text{err}}(\boldsymbol{\Pi})$$

and the left-hand side does **not** depend on the choice of the projection $\boldsymbol{\Pi}$.
:::

:::{prf:proof} Conservation of Total Variance
:label: prf-variance-conservation

Applying the Frobenius Pythagorean decomposition ({prf:ref}`lem-frobenius-pythagoras`) to $\boldsymbol{\Pi}$ gives $\|\tilde{\mathbf{X}}\|_F^2 = \|\tilde{\mathbf{X}}\boldsymbol{\Pi}\|_F^2 + \|\tilde{\mathbf{X}}(\mathbf{I}-\boldsymbol{\Pi})\|_F^2$. Dividing both sides by $m-1$ gives the equality; and the left-hand side $V_{\text{total}} = \frac{1}{m-1}\|\tilde{\mathbf{X}}\|_F^2$ does not involve $\boldsymbol{\Pi}$, so it does not depend on the choice of projection. $\square$
:::

The whole point of this equality lies in the word "conservation": **no matter which subspace we project onto, the total variance splits exactly into two pieces, approximation and error, and the total does not change one bit.** So dimensionality reduction is never a question of "how much variance can be preserved"—that holds automatically; the real question is "within the fixed total, how to keep as much variance as possible in the approximation and lose as little as possible to the error." This leads to Step 2.

#### Step 2: Eckart–Young Selects the Best Projection

:::{prf:theorem} The Best Projection
:label: thm-pca-best-projection

Among all orthogonal projections of rank at most $k$, the projection onto the first $k$ right singular vectors, $\boldsymbol{\Pi}_k = \mathbf{V}_k\mathbf{V}_k^{\top}$ (where $\mathbf{V}_k = [\,\mathbf{v}_0\ \cdots\ \mathbf{v}_{k-1}\,]$), maximizes $V_{\text{approx}}$ and minimizes $V_{\text{err}}$, and

$$V_{\text{approx}}(\boldsymbol{\Pi}_k) = \frac{1}{m-1}\sum_{i=0}^{k-1}\sigma_i^2, \qquad V_{\text{err}}(\boldsymbol{\Pi}_k) = \frac{1}{m-1}\sum_{i=k}^{r-1}\sigma_i^2$$
:::

:::{prf:proof} The Best Projection
:label: prf-pca-best-projection

By {prf:ref}`thm-variance-conservation`, maximizing $V_{\text{approx}}(\boldsymbol{\Pi})$ is equivalent to minimizing $V_{\text{err}}(\boldsymbol{\Pi}) = \frac{1}{m-1}\|\tilde{\mathbf{X}} - \tilde{\mathbf{X}}\boldsymbol{\Pi}\|_F^2$, that is, to finding, among projections of rank $k$, the one with the smallest reconstruction error for $\tilde{\mathbf{X}}$. The Eckart–Young theorem ({prf:ref}`thm-eckart-young-frobenius`) states that this minimum is attained by the truncated SVD $\tilde{\mathbf{X}}_k = \tilde{\mathbf{X}}\boldsymbol{\Pi}_k$, with error $\|\tilde{\mathbf{X}}-\tilde{\mathbf{X}}_k\|_F^2 = \sum_{i=k}^{r-1}\sigma_i^2$ ({prf:ref}`thm-frobenius-norm-svd`). Substituting into $V_{\text{err}}$ and then using conservation gives $V_{\text{approx}}$. $\square$
:::

The choice made by PCA is thus justified: **we choose the right singular vectors of the SVD not because these directions are special in themselves, but because projecting onto the subspace they span preserves the most variance among all projections of the same dimension.** As for the geometric appearance of these best projection directions—they happen to be "the directions of greatest variation in the data"—we leave that to §11.3.2 as an observation.

:::{prf:remark} Why Take the SVD of $\tilde{\mathbf{X}}$ Directly Instead of Forming $\mathbf{S}$ First
:label: rem-pca-svd-stability

In theory, the principal components can be obtained either from the eigendecomposition of $\mathbf{S}$ or from the SVD of $\tilde{\mathbf{X}}$ ($\mathbf{v}_i$ are the eigenvectors of $\mathbf{S}$, and $\lambda_i = \sigma_i^2/(m-1)$). But forming $\mathbf{S} = \frac{1}{m-1}\tilde{\mathbf{X}}^{\top}\tilde{\mathbf{X}}$ **squares** the condition number: $\kappa(\mathbf{S}) = \kappa(\tilde{\mathbf{X}})^2$. If $\kappa(\tilde{\mathbf{X}})\sim 10^8$, then $\kappa(\mathbf{S})\sim 10^{16}$, already approaching the double-precision machine precision $\epsilon\approx 2.2\times10^{-16}$, and the smallest few principal components will be drowned in rounding error. Taking the SVD of $\tilde{\mathbf{X}}$ directly avoids this matrix multiplication, keeps the condition number at $\kappa(\tilde{\mathbf{X}})$, and is far more numerically stable. **Rule of practice: always take the SVD of the centered data matrix, and never form the covariance matrix explicitly.**
:::

### 11.3.2 Directions of Greatest Variation, and the Limitations of PCA

Look back at the best projection $\boldsymbol{\Pi}_k$ chosen in §11.3.1. Taking $k=1$, its direction is $\mathbf{v}_0$, which preserves the most variance among all unit directions; in turn, $\mathbf{v}_0, \mathbf{v}_1, \ldots$ are "the directions of variation in the data from largest to smallest"—the 0-th, the 1st, ... principal components. This is also the best-known description of PCA: **finding the directions of greatest variation in the data**. But we must stress its place in the logic of this chapter: it is an **observation** that falls out of the "best projection" incidentally, not the goal we are pursuing. What we want throughout is the best low-rank approximation; the direction of greatest variation is just how it looks when $k=1$.

#### The Variance-Maximization Viewpoint

Let us write the observation above as a precise step-by-step definition. The 0-th principal component $\mathbf{v}_0$ is the unit vector that maximizes the variance of the projected data:

$$\mathbf{v}_0 = \arg\max_{\|\mathbf{v}\|=1} \text{Var}(\tilde{\mathbf{X}}\mathbf{v}) = \arg\max_{\|\mathbf{v}\|=1} \frac{1}{m-1}\|\tilde{\mathbf{X}}\mathbf{v}\|^2$$

(Since $\tilde{\mathbf{X}}$ is already centered, the projection scores $\tilde{\mathbf{X}}\mathbf{v}$ have mean zero, so their variance is $\frac{1}{m-1}\|\tilde{\mathbf{X}}\mathbf{v}\|^2$.) This is exactly the variational characterization of the SVD in §11.1.3 ({prf:ref}`thm-singular-value-extremal`)! The 1st principal component $\mathbf{v}_1$ is the vector that maximizes the variance among directions orthogonal to $\mathbf{v}_0$:

$$\mathbf{v}_1 = \arg\max_{\mathbf{v}:\, \|\mathbf{v}\|=1,\; \mathbf{v} \perp \mathbf{v}_0} \frac{1}{m-1}\|\tilde{\mathbf{X}}\mathbf{v}\|^2$$

Continuing in this way, the $i$-th principal component $\mathbf{v}_i$ is the direction of maximal variance orthogonal to the first $i$ principal components $\mathbf{v}_0, \ldots, \mathbf{v}_{i-1}$. This characterization—"successively maximize the variance in the orthogonal complement"—arrives at the same place as the "best projection" of §11.3.1: the former picks greedily one direction at a time, the latter chooses the whole subspace at once, and both select the right singular vectors of the SVD.

The two are not equally "strong," however. Eckart–Young is a **global** result: it weighs all $k$-dimensional subspaces simultaneously and determines the entire optimal subspace in one stroke. The greedy algorithm is **local (greedy)**: it picks one direction after another, at each step performing only a one-dimensional maximization under the constraint "orthogonal to the directions already chosen," and never goes back. In general, local greed is a weaker, operational result—stepwise optimality does not guarantee global optimality. In this problem, however, the two are **exactly equivalent**: each step maximizes the variance in the orthogonal complement of the first $i-1$ principal components and selects precisely the next singular direction, and the accumulated steps span exactly the globally optimal subspace of Eckart–Young. This coincidence of "greedy equals global" stems from the nested structure of the eigenspaces of a symmetric positive semidefinite matrix and is not a property enjoyed by optimization in general—and precisely because of it, PCA can reach the global optimum by the plainest step-by-step procedure.

#### The Limitations of PCA

It is precisely this nature of "looking only at linear variance" that marks the limits of PCA. Once the structure of the data is not linear, the conservation of §11.3.1 still holds (the total variance is always conserved), but "keeping the directions of greatest variance" need not mean "keeping the most useful structure." Here are several limitations to remember:

1. **Linearity assumption**: PCA can capture only linear relationships. For nonlinear structures (spirals, rings), kernel PCA or manifold learning (t-SNE, UMAP) is needed.
2. **Global method**: PCA looks for the globally optimal linear subspace and is insensitive to local structure.
3. **Sensitivity to outliers**: variance maximization can be dominated by outliers. Robust alternatives include Robust PCA and L1-PCA.
4. **Interpretability**: principal components are linear combinations of the original features and often have a less clear meaning than the original features.
5. **Unsupervised**: PCA does not use labels, so in classification/regression tasks it may discard directions useful for prediction. In that case, supervised dimensionality reduction (such as Linear Discriminant Analysis, LDA) should be considered.

The first of these is the most fatal and deserves to be seen with your own eyes. Consider the classic **Swiss roll**: the data lie on a rolled-up two-dimensional sheet in three-dimensional space. Below, the height direction $y$ is deliberately stretched so that it becomes the direction of greatest variance (the colors mark four segments cut along $t$). The left panel shows the original 3D structure—the spiral is clearly visible, and the height axis is just a noise direction superimposed on it. The middle panel takes PCA's default first two principal components PC0–PC1, but PC0 is exactly the height axis that has nothing to do with the structure, so the four segments are stirred into overlapping bands and no spiral can be seen. The right panel takes PC1–PC2 (the spiral plane), which PCA regards as secondary, and there the four segments separate clearly along the spiral. The lesson is sharp: PCA always picks the directions of greatest variance, but **the greatest variance is not necessarily the most meaningful direction**—when an irrelevant high-variance axis gets mixed in, PCA's default projection actually hides the structure. **Remedy**: at the linear level, one can look at the secondary principal components instead or perform feature selection first; to truly "unroll" the spiral (the spiral in the right panel is still curled), one needs manifold learning (Isomap, LLE) or kernel PCA, which measure distance along the manifold rather than along Euclidean straight lines.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Color scheme: NYU Official (Accent Mix) ---
C_BG    = "#F8F8F8"   # canvas background
C_GRID  = "#D6D6D6"   # grid lines
C_AXIS  = "#000000"   # axes / title text
C_V1    = "#57068C"   # NYU Violet
C_V2    = "#006385"   # Bleecker Deep Blue
C_T1    = "#2AD2C9"   # Mercer Teal
C_WARN  = "#FF5D47"   # Hudson Orange

# Fonts: the English edition needs no CJK font, so nothing is downloaded when _lang == 'en';
# the Chinese editions use this same block to fetch Noto Sans TC/SC where no CJK font is installed
import os, urllib.request
import matplotlib.font_manager as fm
_lang = 'en'
_cjk = ['Microsoft JhengHei', 'PingFang TC', 'Noto Sans CJK TC', 'Noto Sans TC']
_have = {f.name for f in fm.fontManager.ttflist}
if _lang != 'en' and not _have & set(_cjk):
    _font = os.path.join(os.path.expanduser('~'), '.cache', 'fonts', 'NotoSansTC.ttf')
    try:
        if not os.path.exists(_font):
            os.makedirs(os.path.dirname(_font), exist_ok=True)
            urllib.request.urlretrieve('https://github.com/google/fonts/raw/main/ofl/notosanstc/NotoSansTC%5Bwght%5D.ttf', _font + '.part')
            os.replace(_font + '.part', _font)
        fm.fontManager.addfont(_font)
        _have.add('Noto Sans TC')
    except OSError as err:
        print('Could not download the CJK font; Chinese text in figures may not display:', err)
plt.rcParams['font.family'] = [f for f in _cjk if f in _have] + ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.set_printoptions(precision=4, suppress=True, linewidth=100)

def print_header(title):
    print("=" * 60)
    print(f"  {title}")
    print("=" * 60)

def print_step(step, desc):
    print(f"\n▶ Step {step}: {desc}")
    print("-" * 40)

# ============================================================
print_header("Swiss Roll | The Pitfall of PCA's Choice of Axes")
# ============================================================

print_step(1, "Generate Swiss roll data (the height y is deliberately stretched to be the direction of greatest variance)")
rng = np.random.default_rng(0)
n = 1600
t = 1.5 * np.pi * (1 + 2 * rng.random(n))          # arc-length parameter along the roll
h = 60.0 * rng.random(n)                             # height: unrelated to t, but with the greatest variance
X = np.column_stack([t * np.cos(t), h, t * np.sin(t)])
cls = np.digitize(t, np.quantile(t, [0.25, 0.5, 0.75]))  # cut into 4 segments along t
seg_colors = [C_V1, C_V2, C_T1, C_WARN]
print(f"  Number of samples n = {n}, original dimension = {X.shape[1]}")

print_step(2, "Center, take the SVD, and obtain the principal component scores")
X_tilde = X - X.mean(axis=0)
U, sigma, Vt = np.linalg.svd(X_tilde, full_matrices=False)
Z = X_tilde @ Vt.T
rho = sigma**2 / (sigma**2).sum()
print(f"  Variance share of each principal component ρ = {np.round(rho, 3)}")
print(f"  Alignment of PC0 with the original axes |cos⟨v_i, e_j⟩| = {np.round(np.abs(Vt[0]), 2)} (≈ the height axis)")

print_step(3, "Plot: original 3D structure vs. two PCA projections")
fig = plt.figure(figsize=(15, 4.8))

# --- left panel: original 3D Swiss roll structure ---
ax0 = fig.add_subplot(1, 3, 1, projection='3d')
ax0.set_facecolor(C_BG)
for c in range(4):
    mask = cls == c
    ax0.scatter(X[mask, 0], X[mask, 1], X[mask, 2], s=6, color=seg_colors[c], label=f'segment {c}')
ax0.set_title('Original 3D structure (height axis has greatest variance)', color=C_AXIS)
ax0.set_xlabel('x', color=C_AXIS)
ax0.set_ylabel('y (height, greatest variance)', color=C_AXIS)
ax0.set_zlabel('z', color=C_AXIS)
ax0.xaxis.pane.set_facecolor(C_BG)
ax0.yaxis.pane.set_facecolor(C_BG)
ax0.zaxis.pane.set_facecolor(C_BG)
ax0.grid(True, color=C_GRID, linewidth=0.6, alpha=0.6)
ax0.legend(markerscale=2, fontsize=7, loc='upper left')

# --- middle panel: first 2 PCA components PC0-PC1 ---
ax1 = fig.add_subplot(1, 3, 2)
ax1.set_facecolor(C_BG)
ax1.grid(True, color=C_GRID, linewidth=0.8, alpha=0.7)
for c in range(4):
    mask = cls == c
    ax1.scatter(Z[mask, 0], Z[mask, 1], s=8, color=seg_colors[c], label=f'segment {c}')
ax1.set_title('PCA first 2 components PC0–PC1 (greatest variance): overlap', color=C_AXIS)
ax1.set_xlabel('PC0', color=C_AXIS)
ax1.set_ylabel('PC1', color=C_AXIS)
ax1.legend(markerscale=2, fontsize=7)

# --- right panel: secondary PCA plane PC1-PC2 ---
ax2 = fig.add_subplot(1, 3, 3)
ax2.set_facecolor(C_BG)
ax2.grid(True, color=C_GRID, linewidth=0.8, alpha=0.7)
for c in range(4):
    mask = cls == c
    ax2.scatter(Z[mask, 1], Z[mask, 2], s=8, color=seg_colors[c])
ax2.set_title('PCA secondary plane PC1–PC2 (spiral plane): separated', color=C_AXIS)
ax2.set_xlabel('PC1', color=C_AXIS)
ax2.set_ylabel('PC2', color=C_AXIS)

plt.tight_layout()
plt.show()


:::{admonition} §11.3 Summary
:class: tip

This section answered a classic question of data science: how can we find, from high-dimensional samples, the low-dimensional directions that best explain the variation in the data? PCA's answer is to take the SVD of the centered data matrix $\tilde{\mathbf{X}}$—the right singular vectors $\mathbf{v}_0, \mathbf{v}_1, \ldots$ give the first, second, ... principal component directions in turn, and the squares of the corresponding singular values (divided by $m-1$) are the variances explained by the principal components. This translates the Eckart–Young theorem of §11.2 directly into the language of statistics: keeping the first $k$ principal components is equivalent to minimizing the reconstruction error among all rank-$k$ approximations. More importantly, the total variance is a **conserved quantity**—the truncated SVD does not erase variance; it only redistributes it between the approximation and the error, and PCA always cuts off the largest pieces first. Numerically, one must take the SVD of $\tilde{\mathbf{X}}$ directly rather than first computing the covariance matrix $\mathbf{S} = \frac{1}{m-1}\tilde{\mathbf{X}}^{\top}\tilde{\mathbf{X}}$ and then finding its eigenvalues: the latter squares the condition number and may lose as many as eight orders of magnitude of accuracy. The whole process of applying this method to real data and seeing PCA reveal cluster structure with your own eyes (using the Iris dataset) is demonstrated in full in the PCA experiment notebook of this chapter; there you will also see how nonlinear manifolds such as the Swiss roll make PCA fail.

§11.4 then turns the problem of "best approximation" from dimensionality reduction of data to the solution of equations: when a system of linear equations has no exact solution, how can the SVD be used to find the "best approximate solution"? This is the stage of the Moore–Penrose pseudoinverse.
:::

## 11.4 Application 2: The Moore–Penrose Pseudoinverse

In the discussion of systems of linear equations in Chapter 6, we saw that for a square matrix $\mathbf{A} \in \mathbb{R}^{n \times n}$, if $\mathbf{A}$ is invertible, then the equation $\mathbf{A}\mathbf{x} = \mathbf{b}$ has the unique solution $\mathbf{x} = \mathbf{A}^{-1}\mathbf{b}$. Real-world problems, however, often involve **non-square** or **rank-deficient matrices**:

- **Overdetermined systems**: $m > n$, more equations than unknowns, usually **no exact solution**
- **Underdetermined systems**: $m < n$, fewer equations than unknowns, usually **infinitely many solutions**
- **Rank-deficient square matrices**: $\text{rank}(\mathbf{A}) < n$, not invertible even though square

**Question.** Can we define a "generalized inverse" that exists for **every matrix** $\mathbf{A} \in \mathbb{R}^{m \times n}$ and reduces to the ordinary inverse when $\mathbf{A}$ is invertible?

This section takes a path from the concrete to the abstract: first we let the simplest case, **diagonal matrices**, reveal the rudiments of the answer (§11.4.1); then we use the SVD, a "universal rotation," to carry the same trick over unchanged to an arbitrary matrix, obtaining the general least-squares solution (§11.4.2); only at the end do we go back and verify that the matrix so constructed satisfies the four Penrose conditions, and cite its uniqueness theorem—this is the formal theory of the **Moore–Penrose pseudoinverse** (§11.4.3).

---

### 11.4.1 Starting from Diagonal Matrices: The Rudiments of the Pseudoinverse

In a system of equations with an arbitrary matrix, the variables are entangled with one another, and it is not easy to see at a glance how "inversion" should be generalized. But in a system with a **diagonal matrix**, the components do not interfere with one another, and the answer can be read off term by term.

**The trivial case.** If $\boldsymbol{\Sigma} = \text{diag}(\sigma_0,\ldots,\sigma_{n-1})$ is an $n\times n$ square matrix and all $\sigma_i \neq 0$, then $\boldsymbol{\Sigma}\mathbf{x}=\mathbf{b}$ can be solved term by term: $x_i = b_i/\sigma_i$, that is, $\boldsymbol{\Sigma}^{-1}=\text{diag}(1/\sigma_0,\ldots,1/\sigma_{n-1})$—the familiar inverse matrix, with nothing new.

The really interesting case is the **rank-deficient** one:

:::{prf:example} Least-Squares Solution for a Rank-Deficient Diagonal Matrix
:label: ex-diagonal-pseudoinverse

Consider

$$\boldsymbol{\Sigma} = \begin{bmatrix}2&0&0\\0&3&0\\0&0&0\end{bmatrix}, \qquad \mathbf{b} = \begin{bmatrix}4\\9\\5\end{bmatrix}$$

$\boldsymbol{\Sigma}$ is a $3\times 3$ square matrix, but its diagonal entry number 2, $\Sigma_{22}$, is zero, so its rank is only 2. Writing out the equation $\boldsymbol{\Sigma}\mathbf{x}=\mathbf{b}$ row by row:

$$2x_0 = 4, \qquad 3x_1 = 9, \qquad 0\cdot x_2 = 5$$

**Analysis.**
1. Rows 0 and 1 can be solved exactly: $x_0=2$, $x_1=3$.
2. Row 2 reads $0=5$, which cannot be satisfied whatever value $x_2$ takes—this equation contributes an **irreducible residual** of $5$.
3. The variable $x_2$ does not appear in $\boldsymbol{\Sigma}\mathbf{x}$ at all and has no effect on the residual; but it does increase $\|\mathbf{x}\|$. To make the norm of the solution as small as possible, the only reasonable choice is $x_2=0$.

Thus the least-squares, minimum-norm solution is $\hat{\mathbf{x}}=(2,3,0)^{\top}$, with residual norm $\|\mathbf{r}\|=5$.

**Check.** Define $\boldsymbol{\Sigma}^{+}=\text{diag}(1/2,\,1/3,\,0)$ (take reciprocals of the nonzero diagonal entries and keep the zero entries zero). Then

$$\boldsymbol{\Sigma}^{+}\mathbf{b} = \begin{bmatrix}1/2&0&0\\0&1/3&0\\0&0&0\end{bmatrix}\begin{bmatrix}4\\9\\5\end{bmatrix} = \begin{bmatrix}2\\3\\0\end{bmatrix} = \hat{\mathbf{x}} \quad\checkmark$$
:::

This example states the rule clearly: **the pseudoinverse $\boldsymbol{\Sigma}^{+}$ of a diagonal matrix (including a rectangular diagonal matrix) is the matrix obtained by taking reciprocals of the nonzero diagonal entries, transposing the shape, and filling all other positions with zeros.** A column corresponding to a zero diagonal entry (a free variable) naturally becomes a zero row in $\boldsymbol{\Sigma}^{+}$—because setting that variable to zero is exactly the choice that minimizes the norm; a row corresponding to a zero diagonal entry (an equation that cannot be satisfied) is ignored, contributes nothing to the solution, but leaves an irreducible residual.

This intuition holds for every diagonal matrix, square or rectangular, full rank or rank-deficient. The problem is that a general matrix $\mathbf{A}$ is not diagonal, and its components are entangled with one another. The next subsection uses the SVD to "untangle" them, reducing the general case exactly to the diagonal case already solved here.

---


### 11.4.2 Generalizing to Least Squares with the SVD

The key idea is to use the SVD to "rotate" the problem for an arbitrary matrix into the diagonal problem already solved in §11.4.1. Let $\mathbf{A}=\mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{\top}\in\mathbb{R}^{m\times n}$ and consider

$$\min_{\mathbf{x}\in\mathbb{R}^n}\|\mathbf{A}\mathbf{x}-\mathbf{b}\|^2$$

**Change of coordinates.** $\mathbf{U}$ and $\mathbf{V}$ are both orthogonal matrices, and multiplying by an orthogonal matrix does not change the length of a vector (for any $\mathbf{w}$, $\|\mathbf{U}\mathbf{w}\|=\|\mathbf{w}\|$). Therefore

$$\|\mathbf{A}\mathbf{x}-\mathbf{b}\|^2 = \|\mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{\top}\mathbf{x}-\mathbf{b}\|^2 = \|\boldsymbol{\Sigma}\mathbf{V}^{\top}\mathbf{x}-\mathbf{U}^{\top}\mathbf{b}\|^2$$

Let $\mathbf{y}:=\mathbf{V}^{\top}\mathbf{x}$ (again, since $\mathbf{V}$ is orthogonal, $\|\mathbf{y}\|=\|\mathbf{x}\|$) and $\mathbf{c}:=\mathbf{U}^{\top}\mathbf{b}$. The problem becomes

$$\min_{\mathbf{y}\in\mathbb{R}^n}\|\boldsymbol{\Sigma}\mathbf{y}-\mathbf{c}\|^2$$

—exactly the diagonal problem already solved in §11.4.1! Moreover, $\|\mathbf{x}\|=\|\mathbf{y}\|$ guarantees that the minimum-norm solution found in $\mathbf{y}$-coordinates is still the minimum-norm solution after changing back to $\mathbf{x}$-coordinates. By §11.4.1, this solution is $\hat{\mathbf{y}}=\boldsymbol{\Sigma}^{+}\mathbf{c}$ (reciprocals of the nonzero singular values, zeros elsewhere). Changing back to the original coordinates:

$$\hat{\mathbf{x}}=\mathbf{V}\hat{\mathbf{y}}=\mathbf{V}\boldsymbol{\Sigma}^{+}\mathbf{U}^{\top}\mathbf{b}=:\mathbf{A}^{+}\mathbf{b}$$

This is the construction formula for the pseudoinverse—it is not assumed out of thin air, but is the inevitable result of carrying the solution method for diagonal matrices over unchanged to general matrices along the "universal rotation" of the SVD. Let us state this argument as a formal theorem:

:::{prf:theorem} The Pseudoinverse and Least-Squares Solutions
:label: thm-pseudoinverse-least-squares

Let $\mathbf{A} \in \mathbb{R}^{m \times n}$ and $\mathbf{b} \in \mathbb{R}^m$, with SVD $\mathbf{A}=\mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{\top}$ and $r=\text{rank}(\mathbf{A})$. Then:

1. **A minimum-residual solution exists**: $\hat{\mathbf{x}} = \mathbf{A}^{+}\mathbf{b} = \mathbf{V}\boldsymbol{\Sigma}^{+}\mathbf{U}^{\top}\mathbf{b}$ minimizes $\|\mathbf{A}\mathbf{x} - \mathbf{b}\|^2$

2. **Minimum norm**: if there are several least-squares solutions, $\mathbf{A}^{+}\mathbf{b}$ is the one of **smallest norm**:

   $$\|\mathbf{A}^{+}\mathbf{b}\| = \min\{\|\mathbf{x}\| : \|\mathbf{A}\mathbf{x} - \mathbf{b}\| \text{ is minimal}\}$$

3. **Residual**: the minimum residual is

   $$\|\mathbf{A}\hat{\mathbf{x}} - \mathbf{b}\|^2 = \sum_{i=r}^{m-1}\langle\mathbf{u}_i|\mathbf{b}\rangle^2$$

   that is, the squared length of the projection of $\mathbf{b}$ onto the left null space $\ker(\mathbf{A}^{\top})$.
:::

```{prf:proof}
By the change of coordinates above, $\|\mathbf{A}\mathbf{x}-\mathbf{b}\|=\|\boldsymbol{\Sigma}\mathbf{y}-\mathbf{c}\|$, where $\mathbf{y}=\mathbf{V}^{\top}\mathbf{x}$, $\mathbf{c}=\mathbf{U}^{\top}\mathbf{b}$, and $\|\mathbf{x}\|=\|\mathbf{y}\|$.

Write out $\boldsymbol{\Sigma}\mathbf{y}-\mathbf{c}$ component by component: for $i<r$, the $i$-th component is $\sigma_i y_i - c_i$, which can be made exactly zero (take $y_i=c_i/\sigma_i$); for $i\geq r$, row $i$ of $\boldsymbol{\Sigma}$ is entirely zero, so the component is simply $-c_i$, independent of $\mathbf{y}$—an irreducible residual. Therefore

$$\|\boldsymbol{\Sigma}\mathbf{y}-\mathbf{c}\|^2 = \sum_{i=0}^{r-1}(\sigma_iy_i-c_i)^2 + \sum_{i=r}^{m-1}c_i^2 \geq \sum_{i=r}^{m-1}c_i^2$$

Equality holds if and only if $y_i=c_i/\sigma_i\,(i<r)$; for $i\geq r$, $y_i$ does not affect the residual, and to make $\|\mathbf{y}\|$ as small as possible we must take $y_i=0$. This gives $\hat{\mathbf{y}}=\boldsymbol{\Sigma}^{+}\mathbf{c}$, the unique least-squares, minimum-norm solution (the argument for the diagonal case in §11.4.1 holds word for word). Changing back gives $\hat{\mathbf{x}}=\mathbf{V}\hat{\mathbf{y}}=\mathbf{A}^{+}\mathbf{b}$, and since $\|\mathbf{x}\|=\|\mathbf{y}\|$, $\hat{\mathbf{x}}$ is also the least-squares solution of smallest norm for the original problem. The residual is $\|\mathbf{r}\|^2=\sum_{i=r}^{m-1}c_i^2=\sum_{i=r}^{m-1}\langle\mathbf{u}_i|\mathbf{b}\rangle^2$ (because $c_i=(\mathbf{U}^{\top}\mathbf{b})_i=\langle\mathbf{u}_i|\mathbf{b}\rangle$). $\square$
```

:::{prf:example} Least Squares for a Non-Diagonal Matrix via the SVD
:label: ex-pinv-nondiagonal-least-squares

Consider a matrix that is genuinely "not diagonal"—$\mathbf{A}$ itself is clearly not a diagonal matrix; it just happens that $\mathbf{A}^{\top}\mathbf{A}$ is diagonal:

$$\mathbf{A} = \begin{bmatrix}1&1\\1&-1\\1&0\end{bmatrix}, \quad \mathbf{b} = \begin{bmatrix}2\\0\\3\end{bmatrix}$$

This is a $3\times 2$ overdetermined system, and $\mathbf{b}$ is not in $\text{Col}(\mathbf{A})$ (substitution shows there is no exact solution).

**Step 1. Compute $\mathbf{A}^{\top}\mathbf{A}$ and find the SVD**

$$\mathbf{A}^{\top}\mathbf{A} = \begin{bmatrix}3&0\\0&2\end{bmatrix}$$

This is already diagonal, so the right singular vectors are the standard basis: $\mathbf{v}_0=\begin{bmatrix}1\\0\end{bmatrix}$, $\mathbf{v}_1=\begin{bmatrix}0\\1\end{bmatrix}$, and the singular values are $\sigma_0=\sqrt{3}$, $\sigma_1=\sqrt{2}$.

**Step 2. Construct the left singular vectors**

$$\mathbf{u}_0 = \frac{\mathbf{A}\mathbf{v}_0}{\sigma_0} = \frac{1}{\sqrt{3}}\begin{bmatrix}1\\1\\1\end{bmatrix}, \qquad \mathbf{u}_1 = \frac{\mathbf{A}\mathbf{v}_1}{\sigma_1} = \frac{1}{\sqrt{2}}\begin{bmatrix}1\\-1\\0\end{bmatrix}$$

Since $m=3>r=2$, we need to add a vector $\mathbf{u}_2$ orthogonal to $\mathbf{u}_0,\mathbf{u}_1$ (spanning the left null space $\ker(\mathbf{A}^{\top})$):

$$\mathbf{u}_2 = \frac{1}{\sqrt{6}}\begin{bmatrix}1\\1\\-2\end{bmatrix}$$

**Step 3. Compute the pseudoinverse**

$$\mathbf{A}^{+} = \frac{1}{\sigma_0}\mathbf{v}_0\mathbf{u}_0^{\top} + \frac{1}{\sigma_1}\mathbf{v}_1\mathbf{u}_1^{\top} = \begin{bmatrix}1/3 & 1/3 & 1/3 \\ 1/2 & -1/2 & 0\end{bmatrix}$$

**Step 4. Compute the least-squares solution with the diagonal recipe**

Project $\mathbf{b}$ onto the three left singular vectors:

$$\langle\mathbf{u}_0|\mathbf{b}\rangle = \frac{5}{\sqrt{3}}, \quad \langle\mathbf{u}_1|\mathbf{b}\rangle = \sqrt{2}, \quad \langle\mathbf{u}_2|\mathbf{b}\rangle = -\frac{4}{\sqrt{6}}$$

The first two components are each divided by the corresponding singular value; component 2 ($\langle\mathbf{u}_2|\mathbf{b}\rangle$, corresponding to the zero row of $\boldsymbol{\Sigma}$) is simply the irreducible residual:

$$\hat{\mathbf{x}} = \frac{5/\sqrt{3}}{\sqrt{3}}\mathbf{v}_0 + \frac{\sqrt{2}}{\sqrt{2}}\mathbf{v}_1 = \frac{5}{3}\mathbf{v}_0 + \mathbf{v}_1 = \begin{bmatrix}5/3\\1\end{bmatrix}$$

**Step 5. Check**

$$\mathbf{A}\hat{\mathbf{x}} = \begin{bmatrix}8/3\\2/3\\5/3\end{bmatrix}, \qquad \mathbf{r} = \mathbf{b} - \mathbf{A}\hat{\mathbf{x}} = \begin{bmatrix}-2/3\\-2/3\\4/3\end{bmatrix}, \qquad \|\mathbf{r}\|^2 = \frac{4}{9}+\frac{4}{9}+\frac{16}{9} = \frac{8}{3}$$

Compare with the residual formula of {prf:ref}`thm-pseudoinverse-least-squares`: $\|\mathbf{r}\|^2 = \langle\mathbf{u}_2|\mathbf{b}\rangle^2 = \left(-\dfrac{4}{\sqrt{6}}\right)^2 = \dfrac{8}{3}$ ✓ The two agree.

This example uses exactly the same recipe as the diagonal example of §11.4.1—the only difference is that in the diagonal case $\langle\mathbf{u}_i|\mathbf{b}\rangle$ can be read off directly (because $\mathbf{u}_i$ are the standard basis vectors), whereas here the SVD first "straightens" $\mathbf{A}$ into diagonal form before the same rule is applied.
:::

---


### 11.4.3 The Moore–Penrose Pseudoinverse in Theory

We have now constructed the same matrix $\mathbf{A}^{+}=\mathbf{V}\boldsymbol{\Sigma}^{+}\mathbf{U}^{\top}$ concretely twice—first seeing its logic in the rudimentary diagonal case of §11.4.1, then carrying it over to general matrices with the SVD in §11.4.2 and verifying that it does give the least-squares, minimum-norm solution. This subsection closes the theory: we verify that the constructed matrix satisfies the four Penrose conditions; for uniqueness we cite Penrose's theorem without proving it here—this is precisely the characterization Penrose gave in 1955, and it is the origin of the name "Moore–Penrose pseudoinverse."

#### Definition of the Moore–Penrose Pseudoinverse

:::{prf:definition} Moore–Penrose Pseudoinverse
:label: def-moore-penrose-pseudoinverse

Let $\mathbf{A} \in \mathbb{R}^{m \times n}$. A matrix $\mathbf{A}^{+} \in \mathbb{R}^{n \times m}$ is called the **Moore–Penrose pseudoinverse** of $\mathbf{A}$ (also called the **generalized inverse** or simply the **pseudoinverse**) if it satisfies the following **four Penrose conditions**:

1. $\mathbf{A}\mathbf{A}^{+}\mathbf{A} = \mathbf{A}$
2. $\mathbf{A}^{+}\mathbf{A}\mathbf{A}^{+} = \mathbf{A}^{+}$
3. $(\mathbf{A}\mathbf{A}^{+})^{\top} = \mathbf{A}\mathbf{A}^{+}$ (that is, $\mathbf{A}\mathbf{A}^{+}$ is a symmetric matrix)
4. $(\mathbf{A}^{+}\mathbf{A})^{\top} = \mathbf{A}^{+}\mathbf{A}$ (that is, $\mathbf{A}^{+}\mathbf{A}$ is a symmetric matrix)

These four conditions determine $\mathbf{A}^{+}$ uniquely.
:::

```{note}
:label: note-penrose-conditions-meaning

**The geometric meaning of the Penrose conditions**:
- Conditions 1 and 2: $\mathbf{A}^{+}$ is a "weak inverse"
- Condition 3: $\mathbf{A}\mathbf{A}^{+}$ is the **orthogonal projection** onto the image $\text{Col}(\mathbf{A})$
- Condition 4: $\mathbf{A}^{+}\mathbf{A}$ is the **orthogonal projection** onto the row space $\text{Row}(\mathbf{A})$

These four conditions guarantee the uniqueness of $\mathbf{A}^{+}$ and its good algebraic properties.
```

#### Constructing the Pseudoinverse from the SVD

Through the SVD, the pseudoinverse has an extremely concise expression (this is precisely the construction formula we derived in §11.4.2; here we take a different angle and formally prove that it satisfies the Penrose axioms):

:::{prf:theorem} SVD Construction of the Pseudoinverse
:label: thm-pseudoinverse-svd

Let $\mathbf{A} \in \mathbb{R}^{m \times n}$ have SVD

$$\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{\top} = \sum_{i=0}^{r-1}\sigma_i \mathbf{u}_i\mathbf{v}_i^{\top}$$

where $r = \text{rank}(\mathbf{A})$. Then its Moore–Penrose pseudoinverse is

$$\mathbf{A}^{+} = \mathbf{V}\boldsymbol{\Sigma}^{+}\mathbf{U}^{\top} = \sum_{i=0}^{r-1}\frac{1}{\sigma_i}\mathbf{v}_i\mathbf{u}_i^{\top}$$

where $\boldsymbol{\Sigma}^{+} \in \mathbb{R}^{n \times m}$ is obtained by **taking reciprocals of the nonzero diagonal entries of $\boldsymbol{\Sigma}$ and transposing**:

$$\boldsymbol{\Sigma}^{+}_{ji} = \begin{cases}
1/\sigma_i, & \text{if } i = j \text{ and } i < r \\
0, & \text{otherwise}
\end{cases}$$
:::

```{prf:proof}
We need to verify the four Penrose conditions.

**Condition 1**: $\mathbf{A}\mathbf{A}^{+}\mathbf{A} = \mathbf{A}$

$$\mathbf{A}\mathbf{A}^{+}\mathbf{A} = (\mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{\top})(\mathbf{V}\boldsymbol{\Sigma}^{+}\mathbf{U}^{\top})(\mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{\top})$$

$$= \mathbf{U}\boldsymbol{\Sigma}\boldsymbol{\Sigma}^{+}\boldsymbol{\Sigma}\mathbf{V}^{\top}$$

Note that the diagonal entries of $\boldsymbol{\Sigma}\boldsymbol{\Sigma}^{+}\boldsymbol{\Sigma}$ are:
- for $i < r$: $\sigma_i \cdot (1/\sigma_i) \cdot \sigma_i = \sigma_i$
- for $i \geq r$: $0 \cdot 0 \cdot 0 = 0$

Hence $\boldsymbol{\Sigma}\boldsymbol{\Sigma}^{+}\boldsymbol{\Sigma} = \boldsymbol{\Sigma}$, and $\mathbf{A}\mathbf{A}^{+}\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{\top} = \mathbf{A}$ ✓

**Condition 2**: verified similarly, $\mathbf{A}^{+}\mathbf{A}\mathbf{A}^{+} = \mathbf{A}^{+}$ ✓

**Condition 3**: $(\mathbf{A}\mathbf{A}^{+})^{\top} = \mathbf{A}\mathbf{A}^{+}$

$$\mathbf{A}\mathbf{A}^{+} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{\top}\mathbf{V}\boldsymbol{\Sigma}^{+}\mathbf{U}^{\top} = \mathbf{U}\boldsymbol{\Sigma}\boldsymbol{\Sigma}^{+}\mathbf{U}^{\top}$$

Since $\boldsymbol{\Sigma}\boldsymbol{\Sigma}^{+}$ is a diagonal matrix (with diagonal entries 1 or 0), it is symmetric. Therefore

$$(\mathbf{A}\mathbf{A}^{+})^{\top} = \mathbf{U}(\boldsymbol{\Sigma}\boldsymbol{\Sigma}^{+})^{\top}\mathbf{U}^{\top} = \mathbf{U}\boldsymbol{\Sigma}\boldsymbol{\Sigma}^{+}\mathbf{U}^{\top} = \mathbf{A}\mathbf{A}^{+}$$ ✓

**Condition 4**: verified similarly, $(\mathbf{A}^{+}\mathbf{A})^{\top} = \mathbf{A}^{+}\mathbf{A}$ ✓

Hence $\mathbf{A}^{+} = \mathbf{V}\boldsymbol{\Sigma}^{+}\mathbf{U}^{\top}$ satisfies all four conditions. $\square$
```

**Outer-product expansion**:

$$\mathbf{A}^{+} = \sum_{i=0}^{r-1}\frac{1}{\sigma_i}\mathbf{v}_i\mathbf{u}_i^{\top}$$

This formula shows clearly that the pseudoinverse **takes the reciprocal of the singular value** in each term of the outer-product expansion of $\mathbf{A}$ and **swaps the left and right singular vectors**.

:::{prf:example} Pseudoinverse of a Full-Rank Square Matrix
:label: ex-pseudoinverse-full-rank

For an invertible square matrix $\mathbf{A} \in \mathbb{R}^{n \times n}$, the rank is $r = n$ and all singular values are nonzero. Then

$$\mathbf{A}^{+} = \mathbf{V}\boldsymbol{\Sigma}^{-1}\mathbf{U}^{\top} = (\mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{\top})^{-1} = \mathbf{A}^{-1}$$

that is, $\underline{\mathbf{A}^{+} = \mathbf{A}^{-1}}$: the pseudoinverse reduces to the ordinary inverse.
:::

#### The Geometric Action of the Pseudoinverse

The geometric meaning of $\mathbf{A}^{+}$ can be understood through the four fundamental subspaces (recall §11.1.2):

**How $\mathbf{A}^{+}$ acts**:

Given a vector $\mathbf{b} \in \mathbb{R}^m$, compute $\mathbf{x} = \mathbf{A}^{+}\mathbf{b}$:

1. **Filter**: project $\mathbf{b}$ onto the image $\text{Col}(\mathbf{A})$, ignoring its component in the left null space $\ker(\mathbf{A}^{\top})$

   $$\mathbf{b}_{\text{proj}} = \mathbf{A}\mathbf{A}^{+}\mathbf{b} = \sum_{i=0}^{r-1}⟨ \mathbf{u}_i|\mathbf{b}⟩\mathbf{u}_i$$

2. **Inverse mapping**: map the components in the image back to the row space $\text{Row}(\mathbf{A})$ through the **reciprocals of the singular values**

   $$\mathbf{x} = \mathbf{A}^{+}\mathbf{b} = \sum_{i=0}^{r-1}\frac{⟨ \mathbf{u}_i|\mathbf{b}⟩}{\sigma_i}\mathbf{v}_i$$

3. **Output**: we obtain the unique vector in the row space

**Diagram**:

```{mermaid}
%%{init: { "theme": "base", "themeVariables": {
  "background": "#FFFFFF", "primaryColor": "#FFFFFF",
  "primaryTextColor": "#57068C", "primaryBorderColor": "#57068C",
  "lineColor": "#7F8C8D", "clusterBkg": "#F9F9FB",
  "clusterBorder": "#DCDDE1", "fontSize": "13px" } }}%%
graph LR
    subgraph GIN["Input space ℝᵐ (where 𝐛 lives)"]
        COL["Image Col(𝐀)<br/>𝐮₀, …, 𝐮ᵣ₋₁"]
        KERAS["Left null space ker(𝐀^T)<br/>𝐮ᵣ, …, 𝐮ₘ₋₁"]
    end
    subgraph GOUT["Output space ℝⁿ (where 𝐱 lives)"]
        ROW["Row space Row(𝐀)<br/>𝐯₀, …, 𝐯ᵣ₋₁"]
        KERA["Null space ker(𝐀)<br/>𝐯ᵣ, …, 𝐯ₙ₋₁"]
        ZERO["Zero vector 𝟎"]
    end
    COL -->|"𝐀⁺: 𝐮ᵢ ↦ 𝐯ᵢ/σᵢ<br/>inverse mapping"| ROW
    KERAS -->|"𝐀⁺: mapped to zero<br/>(noise filtered out)"| ZERO
    classDef incls  fill:#E6F0F3,stroke:#006385,stroke-width:2px,color:#006385
    classDef outcls fill:#F4EEF9,stroke:#57068C,stroke-width:2px,color:#57068C
    classDef nullc  fill:#FFF1EE,stroke:#FF5D47,stroke-width:1.5px,color:#B33A2B
    classDef zeroc  fill:#FFFFFF,stroke:#DCDDE1,color:#2F3640
    class COL incls
    class KERAS nullc
    class ROW outcls
    class KERA nullc
    class ZERO zeroc
```

In the diagram, the null space $\ker(\mathbf{A})$ is not connected to any arrow: the range of $\mathbf{A}^{+}$ is the row space $\text{Row}(\mathbf{A})$, and the solution $\mathbf{A}^{+}\mathbf{b}$ never has a null-space component (it is orthogonal to the null space).


**Key properties**:

1. $\mathbf{A}\mathbf{A}^{+}$ is the orthogonal projection onto $\text{Col}(\mathbf{A})$
2. $\mathbf{A}^{+}\mathbf{A}$ is the orthogonal projection onto $\text{Row}(\mathbf{A})$
3. $\mathbf{A}^{+}\mathbf{b}$ always lies in the row space (that is, $\mathbf{A}^{+}\mathbf{b} \perp \ker(\mathbf{A})$)

---

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

print_header("Pseudoinverse Check: The Four Penrose Conditions and the Least-Squares Solution")

A = np.array([[1., 1.],
              [1., -1.],
              [1., 0.]])
b = np.array([2., 0., 3.])

print_step(1, "Construct the pseudoinverse A⁺ = V Σ⁺ U^T from the SVD")
U, s, Vt = np.linalg.svd(A, full_matrices=True)
Sp = np.zeros((2, 3))
Sp[0, 0], Sp[1, 1] = 1 / s[0], 1 / s[1]
A_pinv = Vt.T @ Sp @ U.T
print("A⁺ =\n", A_pinv)
print("Agrees with np.linalg.pinv? →", np.allclose(A_pinv, np.linalg.pinv(A)))

print_step(2, "Verify the four Penrose conditions")
P = A_pinv
print(f"  Condition 1  A A⁺ A  = A    → {np.allclose(A @ P @ A, A)}")
print(f"  Condition 2  A⁺ A A⁺ = A⁺   → {np.allclose(P @ A @ P, P)}")
print(f"  Condition 3  (A A⁺)^T = A A⁺  → {np.allclose((A @ P).T, A @ P)}")
print(f"  Condition 4  (A⁺ A)^T = A⁺ A  → {np.allclose((P @ A).T, P @ A)}")

print_step(3, "Least-squares solution and residual (compare with the hand computation in the example)")
x_hat = A_pinv @ b
r = b - A @ x_hat
print(f"  x̂ = {x_hat} (expected [5/3, 1] ≈ [{5/3:.4f}, 1.0000])")
print(f"  ‖r‖² = {r @ r:.6f} (expected 8/3 ≈ {8/3:.6f})")
print(f"  Residual ⊥ Col(A)? → {np.allclose(A.T @ r, 0)}")


### 11.4.4 Numerical Instability and Ill-Conditioned Problems

Although the pseudoinverse solves the least-squares problem perfectly in theory, in **numerical computation** it can be extremely **unstable**!

#### The Problem of Small Singular Values in Ill-Conditioned Matrices

Recall the formula for the pseudoinverse:

$$\mathbf{A}^{+} = \sum_{i=0}^{r-1}\frac{1}{\sigma_i}\mathbf{v}_i\mathbf{u}_i^{\top}$$

**The problem.** When some singular values $\sigma_i$ are **very small**, $1/\sigma_i$ becomes **very large**, which leads to:

1. **Sensitivity to noise**: tiny perturbations in $\mathbf{b}$ are amplified by a factor of $1/\sigma_i$
2. **Numerical overflow**: an extremely small $\sigma_i$ may make $1/\sigma_i$ exceed the range of floating-point numbers
3. **Unstable solutions**: different numerical algorithms may give completely different solutions

:::{prf:example} Pseudoinverse of an Ill-Conditioned $2 \times 2$ Matrix
:label: ex-illconditioned-pseudoinverse

Consider the matrix

$$\mathbf{A} = \begin{bmatrix}1&1\\1&1+\epsilon\end{bmatrix}, \quad \epsilon = 10^{-8}$$

This matrix is **nearly singular** (its two rows are almost collinear).

**Step 1.** Compute the singular values

$$\mathbf{A}^{\top}\mathbf{A} = \begin{bmatrix}2&2+\epsilon\\2+\epsilon&2+2\epsilon+\epsilon^2\end{bmatrix}$$

Here lies a classic trap: if we "ignore the $\epsilon^2$ term," the determinant becomes $2(2+2\epsilon)-(2+\epsilon)^2 = -\epsilon^2 \approx 0$—dropping $\epsilon^2$ wipes out exactly all the information about the smallest eigenvalue. The correct approach uses the determinant and the trace:

$$\det(\mathbf{A}^{\top}\mathbf{A}) = (\det\mathbf{A})^2 = \epsilon^2, \qquad \operatorname{tr}(\mathbf{A}^{\top}\mathbf{A}) = 4+2\epsilon+\epsilon^2 \approx 4$$

$$\lambda_0 \approx 4, \quad \lambda_1 = \frac{\det(\mathbf{A}^{\top}\mathbf{A})}{\lambda_0} \approx \frac{\epsilon^2}{4}$$

The singular values (which can also be double-checked by $\sigma_0\sigma_1 = |\det\mathbf{A}| = \epsilon$):

$$\sigma_0 \approx 2, \quad \sigma_1 \approx \frac{\epsilon}{2} = 5 \times 10^{-9}$$

**Condition number**:

$$\kappa(\mathbf{A}) = \frac{\sigma_0}{\sigma_1} \approx \frac{2}{5 \times 10^{-9}} = 4 \times 10^8$$

**Step 2.** The numerical problem with the pseudoinverse

$$\mathbf{A}^{+} = \frac{1}{\sigma_0}\mathbf{v}_0\mathbf{u}_0^{\top} + \frac{1}{\sigma_1}\mathbf{v}_1\mathbf{u}_1^{\top}$$

The coefficient of the second term is

$$\frac{1}{\sigma_1} \approx 2 \times 10^8$$

**Demonstrating the problem.** Suppose $\mathbf{b} = [1, 1]^{\top}$ has a tiny measurement error $\delta\mathbf{b} = [10^{-6}, 0]^{\top}$:

$$\mathbf{b}_{\text{noisy}} = \begin{bmatrix}1.000001\\1\end{bmatrix}$$

Here $\mathbf{u}_1 \approx \frac{1}{\sqrt{2}}[1, -1]^{\top}$, and the contribution of the error is

$$\Delta\mathbf{x} \approx \frac{⟨ \mathbf{u}_1|\delta\mathbf{b}⟩}{\sigma_1}\mathbf{v}_1 \approx \frac{10^{-6}/\sqrt{2}}{5 \times 10^{-9}}\mathbf{v}_1 \approx 1.4 \times 10^{2}\,\mathbf{v}_1$$

**A tiny input error of $10^{-6}$ is amplified to the order of $10^{2}$; the amplification factor for this particular perturbation is about $1.4\times10^8$, and the amplification factor in the worst direction is $1/\sigma_1\approx2\times10^8$**!
:::

:::{prf:example} An Ill-Conditioned $2 \times 3$ Underdetermined System
:label: ex-illconditioned-underdetermined

Ill-conditioning is not peculiar to square matrices. Consider an underdetermined system whose two equations are almost parallel:

$$\mathbf{A} = \begin{bmatrix}1&0&0\\1&\epsilon&0\end{bmatrix}, \quad \epsilon = 10^{-8}$$

$$\mathbf{A}\mathbf{A}^{\top} = \begin{bmatrix}1&1\\1&1+\epsilon^2\end{bmatrix}, \qquad \operatorname{tr} \approx 2, \quad \det = \epsilon^2$$

The two singular values are $\sigma_0 \approx \sqrt{2}$ and $\sigma_1 \approx \dfrac{\epsilon}{\sqrt{2}} \approx 7 \times 10^{-9}$, so

$$\frac{1}{\sigma_1} \approx 1.4 \times 10^8$$

In double-precision floating point (machine precision $\approx 2.2 \times 10^{-16}$), such a drastic amplification is enough to let any noise in $\mathbf{b}$ that has a component along the left singular vector of the small singular value affect the solution significantly.
:::

#### Truncated SVD

**The remedy.** Ignore singular values that are too small and keep only the first $k$:

$$\mathbf{A}^{+}_k := \sum_{i=0}^{k-1}\frac{1}{\sigma_i}\mathbf{v}_i\mathbf{u}_i^{\top}$$

Given a tolerance threshold $\tau\geq0$, let $k=\#\{i:0\leq i<r,\ \sigma_i>\tau\sigma_0\}$. If $0<k<r$, this is equivalent to $\sigma_{k-1}>\tau\sigma_0\geq\sigma_k$; when $k=0$ we take the zero matrix, and when $k=r$ all positive singular values are kept. For the zero matrix we set $k=0$ separately.

**Effect**: 
- **Advantage**: lowers the noise sensitivity of the truncated directions, but introduces truncation bias
- **Disadvantage**: it is not a strict least-squares solution, and it introduces bias

This leads to a more systematic regularization method—**ridge regression**.

---

### 11.4.5 Ridge Regression: Tikhonov Regularization

#### From Ill-Conditioned Problems to Regularization

The least-squares problem is

$$\min_{\mathbf{x}} \|\mathbf{A}\mathbf{x} - \mathbf{b}\|^2$$

When $\mathbf{A}$ is ill-conditioned (has small singular values), the norm of the solution $\hat{\mathbf{x}} = \mathbf{A}^{+}\mathbf{b}$ may be **very large** and extremely sensitive to noise.

**Tikhonov regularization** (also called **ridge regression**) stabilizes the solution by adding a **norm penalty term**:

$$\min_{\mathbf{x}} \|\mathbf{A}\mathbf{x} - \mathbf{b}\|^2 + \lambda\|\mathbf{x}\|^2$$

where $\lambda > 0$ is the **regularization parameter**.

:::{prf:theorem} Solution of Ridge Regression
:label: thm-ridge-regression-solution

The solution of the ridge regression problem is

$$\hat{\mathbf{x}}_{\lambda} = (\mathbf{A}^{\top}\mathbf{A} + \lambda\mathbf{I})^{-1}\mathbf{A}^{\top}\mathbf{b}$$

Through the SVD $\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{\top}$, it can be written as

$$\hat{\mathbf{x}}_{\lambda} = \sum_{i=0}^{r-1}\frac{\sigma_i}{\sigma_i^2 + \lambda}⟨ \mathbf{u}_i|\mathbf{b}⟩\mathbf{v}_i$$
:::

```{prf:proof}
Differentiate the objective function and set the derivative to zero:

$$\frac{\partial}{\partial\mathbf{x}}(\|\mathbf{A}\mathbf{x} - \mathbf{b}\|^2 + \lambda\|\mathbf{x}\|^2) = 2\mathbf{A}^{\top}(\mathbf{A}\mathbf{x} - \mathbf{b}) + 2\lambda\mathbf{x} = \mathbf{0}$$

Rearranging gives

$$(\mathbf{A}^{\top}\mathbf{A} + \lambda\mathbf{I})\mathbf{x} = \mathbf{A}^{\top}\mathbf{b}$$

This is the regularized version of the **normal equations**. Since $\mathbf{A}^{\top}\mathbf{A} + \lambda\mathbf{I}$ is positive definite (its eigenvalues are $\geq \lambda > 0$), it is invertible:

$$\hat{\mathbf{x}}_{\lambda} = (\mathbf{A}^{\top}\mathbf{A} + \lambda\mathbf{I})^{-1}\mathbf{A}^{\top}\mathbf{b}$$

Expanding with the SVD:

$$\mathbf{A}^{\top}\mathbf{A} = \mathbf{V}\boldsymbol{\Sigma}^{\top}\boldsymbol{\Sigma}\mathbf{V}^{\top} = \mathbf{V}\text{diag}(\sigma_0^2, \ldots, \sigma_{r-1}^2, 0, \ldots, 0)\mathbf{V}^{\top}$$

$$\mathbf{A}^{\top}\mathbf{A} + \lambda\mathbf{I} = \mathbf{V}\text{diag}(\sigma_0^2 + \lambda, \ldots, \sigma_{r-1}^2 + \lambda, \lambda, \ldots, \lambda)\mathbf{V}^{\top}$$

Therefore

$$(\mathbf{A}^{\top}\mathbf{A} + \lambda\mathbf{I})^{-1} = \mathbf{V}\text{diag}(\frac{1}{\sigma_0^2+\lambda}, \ldots, \frac{1}{\sigma_{r-1}^2+\lambda}, \frac{1}{\lambda}, \ldots, \frac{1}{\lambda})\mathbf{V}^{\top}$$

Substituting gives

$$\hat{\mathbf{x}}_{\lambda} = \mathbf{V}\text{diag}(\frac{1}{\sigma_0^2+\lambda}, \ldots)\mathbf{V}^{\top}\mathbf{V}\boldsymbol{\Sigma}^{\top}\mathbf{U}^{\top}\mathbf{b}$$

$$= \sum_{i=0}^{r-1}\frac{\sigma_i}{\sigma_i^2+\lambda}⟨ \mathbf{u}_i|\mathbf{b}⟩\mathbf{v}_i$$

$\square$
```

#### Key Properties of Ridge Regression

**1. Numerical stability**

Compare the coefficients:
- **Pseudoinverse**: $\frac{1}{\sigma_i}$ ($\to \infty$ as $\sigma_i \to 0$)
- **Ridge regression**: $\frac{\sigma_i}{\sigma_i^2 + \lambda}$ ($\to 0$ as $\sigma_i \to 0$)

The ridge regression coefficients are **bounded**:

$$\frac{\sigma_i}{\sigma_i^2 + \lambda} \leq \frac{1}{2\sqrt{\lambda}}$$

**The amplification along small-singular-value directions is suppressed**, trading bias for lower noise sensitivity.

**2. The bias-variance trade-off**

- **$\lambda\to0^+$**: the solution approaches the minimum-norm least-squares solution $\mathbf{A}^+\mathbf{b}$; in the rank-deficient case one cannot simply substitute $\lambda=0$ into the inverse-matrix formula
- **$\lambda \to \infty$**: $\hat{\mathbf{x}}_{\lambda} \to \mathbf{0}$, with small variance and large bias
- **Moderate $\lambda$**: strikes a balance between bias and variance

**3. The filter viewpoint**

Define the **filter factor**

$$f_i(\lambda) := \frac{\sigma_i^2}{\sigma_i^2 + \lambda}$$

Then

$$\hat{\mathbf{x}}_{\lambda} = \sum_{i=0}^{r-1}\frac{f_i(\lambda)}{\sigma_i}⟨ \mathbf{u}_i|\mathbf{b}⟩\mathbf{v}_i$$

Properties of the filter factor:
- when $\sigma_i \gg \sqrt{\lambda}$, $f_i \approx 1$ (almost no attenuation)
- when $\sigma_i \ll \sqrt{\lambda}$, $f_i \approx \sigma_i^2/\lambda$ (strong attenuation)

Ridge regression is **singular-mode filtering** that suppresses the amplification along small-singular-value directions; the size of a singular value is in general not a temporal or spatial frequency, so it cannot be called low-pass filtering across the board.

:::{prf:example} Ridge Regression Fixes an Ill-Conditioned Problem
:label: ex-ridge-illconditioned

Return to the ill-conditioned matrix of {prf:ref}`ex-illconditioned-pseudoinverse`:

$$\mathbf{A} = \begin{bmatrix}1&1\\1&1+\epsilon\end{bmatrix}, \quad \epsilon = 10^{-8}$$

Singular values: $\sigma_0 \approx 2$, $\sigma_1 \approx 5 \times 10^{-9}$

Suppose $\mathbf{b} = [1, 1]^{\top}$, and choose $\lambda = 10^{-6}$.

**Ridge regression coefficients**:

$$\frac{\sigma_0}{\sigma_0^2 + \lambda} \approx \frac{2}{4 + 10^{-6}} \approx 0.5$$

$$\frac{\sigma_1}{\sigma_1^2 + \lambda} \approx \frac{5 \times 10^{-9}}{2.5 \times 10^{-17} + 10^{-6}} \approx \frac{5 \times 10^{-9}}{10^{-6}} = 5 \times 10^{-3}$$

**Compare with the pseudoinverse coefficients**:

$$\frac{1}{\sigma_0} \approx 0.5 \quad (\text{the same})$$

$$\frac{1}{\sigma_1} \approx 2 \times 10^8 \quad (\text{ridge regression gives } 5 \times 10^{-3}\text{, smaller by a factor of about } 4 \times 10^{10} \text{!})$$

**Improved numerical stability.** The contribution along the small-singular-value direction is pushed down from an explosive $2 \times 10^8$ to $5 \times 10^{-3}$—strongly suppressed for this parameter (and far below the theoretical upper bound $\frac{1}{2\sqrt{\lambda}} = 500$)—and the solution is no longer held hostage by the noise direction.
:::

#### Choosing the Regularization Parameter $\lambda$

In practice, the choice of $\lambda$ is crucial.

**Common methods**:

1. **Cross-validation**: choose the $\lambda$ that minimizes the prediction error on a validation set

2. **The L-curve method**: plot $\log\|\mathbf{A}\hat{\mathbf{x}}_{\lambda} - \mathbf{b}\|$ against $\log\|\hat{\mathbf{x}}_{\lambda}\|$ and choose the $\lambda$ at the "corner"

3. **Generalized cross-validation** (GCV): a statistical method that balances bias and variance automatically

4. **Mode-transition scale**: when $\lambda=\sigma_i^2$, $f_i=1/2$. To approximately keep the first $k$ modes ($0<k<r$), one may consider $\sigma_k^2\lesssim\lambda\lesssim\sigma_{k-1}^2$, but the parameter must be chosen according to the noise and the data; this is not an optimality theorem

The code below actually scans $\lambda$ from $10^{-18}$ to $10^{-2}$ for the ill-conditioned system of {prf:ref}`ex-illconditioned-pseudoinverse` and draws the **L-curve**: the horizontal axis is the norm of the solution $\|\hat{\mathbf{x}}_{\lambda}\|$ and the vertical axis is the residual $\|\mathbf{A}\hat{\mathbf{x}}_{\lambda}-\mathbf{b}\|$ (log-log axes). The corner of the L-curve is a heuristic candidate for the parameter; the marker in the figure only demonstrates a preset parameter and does not compute the corner or the optimum automatically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C_BG, C_GRID, C_AXIS = "#F8F8F8", "#D6D6D6", "#000000"
C_V1, C_WARN = "#57068C", "#FF5D47"

print_header("An Ill-Conditioned System in Practice: Singular Values, Condition Number, and the L-Curve")

eps = 1e-8
A = np.array([[1., 1.],
              [1., 1. + eps]])
b = np.array([1., 1.]) + np.array([1e-6, 0.])   # noisy observation

print_step(1, "Singular values and condition number (compare with the corrected figures in the text)")
U, s, Vt = np.linalg.svd(A)
print(f"  σ0 = {s[0]:.4f},  σ1 = {s[1]:.3e} (theory ε/2 = {eps/2:.1e})")
print(f"  κ(A) = σ0/σ1 = {s[0]/s[1]:.3e} (theory ≈ 4×10⁸)")
print(f"  1/σ1 = {1/s[1]:.3e}")

print_step(2, "Pseudoinverse solution vs. ridge regression solution")
x_pinv = np.linalg.pinv(A) @ b
print(f"  pseudoinverse solution x̂   = {np.round(x_pinv, 3)}")
print("    → the noise 10⁻⁶ along the ill-conditioned direction v1 is amplified into an O(10²) shift; the solution is held hostage by the noise")
lam_demo = 1e-6
x_ridge = Vt.T @ ((s / (s**2 + lam_demo)) * (U.T @ b))
print(f"  ridge solution (λ=10⁻⁶) x̂_λ = {np.round(x_ridge, 6)}")
print("    → keeps the information in the well-conditioned direction (x0 + x1 ≈ 1) and strongly suppresses the ill-conditioned direction (x0 − x1), greatly reducing the sensitivity in that direction")

print_step(3, "Scan λ and plot the L-curve")
lams = np.logspace(-18, -2, 80)
sol_norm, res_norm = [], []
for lam in lams:
    x_l = Vt.T @ ((s / (s**2 + lam)) * (U.T @ b))
    sol_norm.append(np.linalg.norm(x_l))
    res_norm.append(np.linalg.norm(A @ x_l - b))

fig, ax = plt.subplots(figsize=(6.8, 5.2))
ax.set_facecolor(C_BG)
ax.grid(True, color=C_GRID, linewidth=0.8, alpha=0.7, which='both')
ax.loglog(sol_norm, res_norm, '-o', color=C_V1, markersize=3.5, linewidth=1.5)
i_demo = int(np.argmin(np.abs(np.log(lams) - np.log(1e-15))))  # preset demonstration parameter, not an automatically found corner
ax.loglog(sol_norm[i_demo], res_norm[i_demo], 'o', color=C_WARN,
          markersize=11, label='demo parameter λ ≈ 10⁻¹⁵ (not chosen automatically)')
ax.set_xlabel('‖x̂_λ‖ (norm of the solution)', color=C_AXIS)
ax.set_ylabel('‖A x̂_λ − b‖ (residual)', color=C_AXIS)
ax.set_title('L-curve: lower right = under-regularized (solution blows up),\nupper left = over-regularized (residual rises)', color=C_AXIS, fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"  Marked demo λ ≈ {lams[i_demo]:.0e}; σ1² = {s[1]**2:.1e}. This figure does not compute the optimal λ")


:::{prf:remark} §11.3 ↔ §11.4: A Dialogue Between Two Uses of the SVD (Unsupervised vs. Supervised)
:label: rem-pca-vs-regression

PCA and linear regression represent the two great paradigms of machine learning, yet they share the same SVD tool; the difference lies in "whether there are labels":

| | PCA (§11.3) | Linear regression / least squares (§11.4) |
|---|---|---|
| Learning paradigm | **Unsupervised**: only the data $\tilde{\mathbf{X}}$, no answers | **Supervised**: features $\mathbf{A}$ paired with labels $\mathbf{b}$ |
| Question asked | In which directions does the data itself vary most? | How should the features be combined to approximate the labels? |
| SVD ingredient used | the right singular vectors $\mathbf{v}_i$ as new coordinate axes | the pseudoinverse $\mathbf{A}^{+}=\mathbf{V}\boldsymbol{\Sigma}^{+}\mathbf{U}^{\top}$ |
| Treatment of singular values | **Truncation**: keep the large $\sigma_i$, discard the small $\sigma_i$ | **Reciprocals**: amplify each direction by $1/\sigma_i$ |
| Optimization objective | $\min\|\tilde{\mathbf{X}}-\tilde{\mathbf{X}}\boldsymbol{\Pi}\|_F$ | $\min\|\mathbf{A}\mathbf{x}-\mathbf{b}\|^2$ |

Most intriguing are the last two rows: **the two take exactly opposite attitudes toward "small-singular-value directions."** PCA discards the low-variance directions of small $\sigma_i$, which has a denoising effect only when the low-rank signal is separable from the noise; least squares instead amplifies them by $1/\sigma_i$, so small $\sigma_i$ become the culprits of numerical instability (§11.4.4). The $\sigma_i/(\sigma_i^2+\lambda)$ filtering of ridge regression (§11.4.5) essentially lets regression "learn" PCA's caution toward small singular values—borrowing unsupervised wisdom in a supervised task.
:::

:::{admonition} §11.4 Summary
:class: tip

The question this section answered is: when a matrix is not invertible, or not even square, can the notion of "inversion" be extended meaningfully? The rudiments of the answer are actually hidden in the simplest diagonal matrices (§11.4.1): take reciprocals of the nonzero singular values, keep the zero singular values zero, and set the extra free variables to zero to obtain the minimum norm. §11.4.2 proved that these rudiments can be carried over unchanged to an arbitrary matrix through the SVD—orthogonal transformations do not change the length of vectors, so after changing $\mathbf{A}\mathbf{x}=\mathbf{b}$ to the coordinates $\mathbf{y}=\mathbf{V}^{\top}\mathbf{x}$ it becomes the diagonal problem already solved, and changing back gives $\mathbf{A}^{+}=\mathbf{V}\boldsymbol{\Sigma}^{+}\mathbf{U}^{\top}$. §11.4.3 closed the theory: we verified that the constructed matrix satisfies the four Penrose axioms and cited its uniqueness theorem; geometrically, $\mathbf{A}^{+}\mathbf{b}$ is precisely the minimum-norm least-squares solution in the row space—the residual is minimal and the solution is orthogonal to the null space, a double optimality completely described by orthogonal projections.

However, the reciprocal $1/\sigma_i$ of a singular value amplifies noise drastically when $\sigma_i$ is close to zero, making the result extremely sensitive to tiny perturbations of the data. Ridge regression corrects this problem with the regularization term $\lambda\|\mathbf{x}\|^2$, replacing the amplification coefficient $1/\sigma_i$ with the bounded $\sigma_i/(\sigma_i^2 + \lambda)$; in essence, it applies singular-mode filtering to the small-singular-value directions, trading a controllable bias for the stability of the solution. The L-curve method provides a heuristic visual tool for choosing $\lambda$ between bias and variance, and it does not guarantee statistical optimality.

§11.5 will bring the SVD into the setting of quantum mechanics: how can the singular value decomposition describe the entanglement structure between the subsystems of a quantum state of a composite system? The Schmidt decomposition will give the answer.
:::

## 11.5 ◆Application 3: The Schmidt Decomposition and Quantum Entanglement

```{tip}
This section uses the quantum mechanics notation of Chapter 10. If you have not yet read Chapter 10, here is the minimum background needed for this section:

**State vector**: a **pure state** of the composite system AB is a unit vector
$|\psi\rangle_{AB} \in \mathcal{H}_A \otimes \mathcal{H}_B$,
where the dimension of the tensor product space is $\dim(\mathcal{H}_A) \times \dim(\mathcal{H}_B)$,
and the standard basis is $\{|i\rangle_A \otimes |j\rangle_B\}$, usually abbreviated as $|ij\rangle$.

**Dirac notation**: $|\psi\rangle$ (a ket) is a column vector, $\langle\psi|$ (a bra) is its conjugate transpose,
$\langle\phi|\psi\rangle$ is the inner product, and $|\psi\rangle\langle\phi|$ is the outer product (a rank-one matrix).

**Density matrix**: the pure state $|\psi\rangle$ corresponds to the density matrix $\rho = |\psi\rangle\langle\psi|$,
which is a positive semidefinite Hermitian matrix with $\text{Tr}(\rho) = 1$.

**Separable states vs. entangled states**: if $|\psi\rangle_{AB} = |\alpha\rangle_A \otimes |\beta\rangle_B$,
the state is called **separable** (no entanglement); otherwise it is called **entangled**.
The central question of this section is: how can the SVD detect and quantify this distinction?
```

Quantum entanglement is one of the most mysterious and most important phenomena of quantum mechanics—Einstein called it "spooky action at a distance"—and it is also the theoretical foundation of technologies such as quantum computing, quantum communication, and quantum sensing. The SVD provides a precise mathematical tool for quantifying entanglement—the Schmidt decomposition: given the state vector of a composite quantum system, the Schmidt decomposition reveals the structure and strength of the entanglement between the subsystems.

**The role of mathematics.** The SVD/Schmidt decomposition provides:
1. **A quantitative tool**: how can the "strength" of entanglement be measured?
2. **Structural understanding**: what is the internal structure of an entangled state?
3. **A computational method**: how can high-dimensional entangled states be handled efficiently?

---

### 11.5.1 Matricizing the State Vector and the Schmidt Decomposition

#### The Tensor Product Structure of Composite Quantum Systems

 **Two views of the tensor product, revisited**
In Chapter 4 we introduced the abstract definition of the tensor product space. Before entering this section, let us become familiar with it again through a concrete example and emphasize a key idea that recurs throughout quantum mechanics: **the state vector of a composite system can be viewed either as a "long vector" or as a "matrix."** These two views do not contradict each other; rather, they are where the SVD comes into play.

::::{exercise} The Tensor Product: Vector View and Matrix View
:label: exer-ch11-tensor-vec-mat

Let $d_A = 2$ and $d_B = 2$, and consider the following pure state of the composite system AB:

$$|\psi⟩_{AB} = \frac{1}{2}|00⟩ + \frac{i}{2}|01⟩ + \frac{1}{2}|10⟩ + \frac{i}{2}|11⟩$$

(1) **Vector view**: write $|\psi⟩_{AB}$ as a column vector in $\mathbb{C}^4$, ordered as $|00⟩, |01⟩, |10⟩, |11⟩$.

(2) **Matrix view**: fill the amplitudes $c_{ij}$ into a $2\times 2$ coefficient matrix $\mathbf{C}$ with $(\mathbf{C})_{ij} = c_{ij}$ (row $i$, column $j$).

(3) Verify that $\|\mathbf{C}\|_F^2 = 1$ (that is, the state vector is normalized).

(4) What is the rank of the coefficient matrix $\mathbf{C}$ of this state? Can you predict the Schmidt rank?

:::{solution} exer-ch11-tensor-vec-mat
:class: dropdown

**(1) Vector view**

Order the entries by the lexicographic order of $|ij⟩$ ($i$ is the index of system A, $j$ the index of system B):

$$|\psi⟩_{AB} \;⟺\; \begin{pmatrix} c_{00} \\ c_{01} \\ c_{10} \\ c_{11} \end{pmatrix} = \begin{pmatrix} 1/2 \\ i/2 \\ 1/2 \\ i/2 \end{pmatrix} \in \mathbb{C}^4$$

This is an element of the $d_A d_B = 4$-dimensional Hilbert space $\mathcal{H}_{AB}$.

**(2) Matrix view**

Rearrange the same coefficients: row $i$ corresponds to the basis state $|i⟩_A$ of system A, and column $j$ corresponds to the basis state $|j⟩_B$ of system B:

$$\mathbf{C} = \begin{pmatrix} c_{00} & c_{01} \\ c_{10} & c_{11} \end{pmatrix} = \begin{pmatrix} 1/2 & i/2 \\ 1/2 & i/2 \end{pmatrix}$$

**Note.** The data here are exactly the same; only "the frame in which we view them" has changed—the former is a 4-dimensional vector, the latter a $2\times 2$ matrix.

**(3) Checking the Frobenius norm**

$$\|\mathbf{C}\|_F^2 = |1/2|^2 + |i/2|^2 + |1/2|^2 + |i/2|^2 = \frac{1}{4}+\frac{1}{4}+\frac{1}{4}+\frac{1}{4} = 1 \quad \checkmark$$

This is exactly the normalization in the vector view: $\sum_{i,j}|c_{ij}|^2 = 1$.

**(4) Determining the rank**

Look at the two rows of $\mathbf{C}$:
- row 0: $(1/2,\ i/2)$
- row 1: $(1/2,\ i/2)$

The two rows are identical, so $\text{rank}(\mathbf{C}) = 1$.

By the separability criterion (the theorem below), a rank-one matrix corresponds to **Schmidt rank = 1**, and this state is a **separable state**.

In fact, one can verify directly that

$$|\psi⟩_{AB} = (\frac{1}{\sqrt{2}}|0⟩_A + \frac{1}{\sqrt{2}}|1⟩_A)\otimes(\frac{1}{\sqrt{2}}|0⟩_B + \frac{i}{\sqrt{2}}|1⟩_B)$$

(The reader may expand to check.)
:::

::::


Consider two quantum systems:
- **System A**: Hilbert space $\mathcal{H}_A$, dimension $d_A$, orthonormal basis $\{|i⟩_A\}_{i=0}^{d_A-1}$
- **System B**: Hilbert space $\mathcal{H}_B$, dimension $d_B$, orthonormal basis $\{|j⟩_B\}_{j=0}^{d_B-1}$

The state space of the **composite system** A⊗B is the **tensor product space**

$$\mathcal{H}_{AB} = \mathcal{H}_A \otimes \mathcal{H}_B$$

of dimension $d_{AB} = d_A \times d_B$, with orthonormal basis

$$\{|i⟩_A \otimes |j⟩_B : i=0,\ldots,d_A-1,\ j=0,\ldots,d_B-1\}$$

abbreviated as $\{|ij⟩\}$.

A **general pure state** can be expanded as

$$|\psi⟩_{AB} = \sum_{i=0}^{d_A-1}\sum_{j=0}^{d_B-1} c_{ij} |i⟩_A \otimes |j⟩_B = \sum_{i,j} c_{ij} |ij⟩$$

where the $c_{ij} \in \mathbb{C}$ are complex amplitudes satisfying the normalization condition

$$\sum_{i,j}|c_{ij}|^2 = 1$$

**Key observation.** The coefficients $c_{ij}$ can be arranged into a $d_A \times d_B$ matrix!

#### The Coefficient Matrix and the SVD

Define the **coefficient matrix** $\mathbf{C} \in \mathbb{C}^{d_A \times d_B}$ by

$$(\mathbf{C})_{ij} := c_{ij}$$

Then the pure state can be written as

$$|\psi⟩_{AB} = \sum_{i,j} (\mathbf{C})_{ij} |i⟩_A |j⟩_B$$

and the **normalization condition** becomes

$$\|\mathbf{C}\|_F^2 = \sum_{i,j}|c_{ij}|^2 = 1$$

that is, $\mathbf{C}$ has Frobenius norm 1.

**The core idea.** Take the **SVD** of $\mathbf{C}$:

$$\mathbf{C} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^*$$

where:
- $\mathbf{U} \in \mathbb{C}^{d_A \times d_A}$ is a unitary matrix
- $\mathbf{V} \in \mathbb{C}^{d_B \times d_B}$ is a unitary matrix  
- $\boldsymbol{\Sigma} \in \mathbb{R}^{d_A \times d_B}$ is a diagonal matrix whose diagonal entries are the singular values $\sigma_0 \geq \sigma_1 \geq \cdots \geq 0$

Let the rank be $r = \text{rank}(\mathbf{C})$, so that the first $r$ singular values are nonzero.


::::{exercise} The General Rule for Vector–Matrix Conversion
:label: exer-ch11-reshaping

Let $d_A = 2$ and $d_B = 3$, so that the composite system has 6 basis states $|00⟩, |01⟩, |02⟩, |10⟩, |11⟩, |12⟩$.

The coefficient matrix of a pure state is

$$\mathbf{C} = \begin{pmatrix} 1 & 0 & 1 \\ 0 & 1 & 0 \end{pmatrix} \cdot \frac{1}{\sqrt{3}}$$

(1) Expand $\mathbf{C}$ to write out the corresponding state $|\psi⟩_{AB}$ (as a linear combination of the $|ij⟩$).

(2) Write this state as a column vector in $\mathbb{C}^6$ and explain the ordering.

(3) Compute $\text{rank}(\mathbf{C})$ and predict an upper bound for the Schmidt rank.


:::{solution} exer-ch11-reshaping
:class: dropdown

**(1) Matrix → state vector**

Read off the coefficients from $(\mathbf{C})_{ij} = c_{ij}$:

| $(i,j)$ | $c_{ij}$ |
|---------|---------|
| $(0,0)$ | $1/\sqrt{3}$ |
| $(0,1)$ | $0$ |
| $(0,2)$ | $1/\sqrt{3}$ |
| $(1,0)$ | $0$ |
| $(1,1)$ | $1/\sqrt{3}$ |
| $(1,2)$ | $0$ |

Therefore

$$|\psi⟩_{AB} = \frac{1}{\sqrt{3}}\bigl(|00⟩ + |02⟩ + |11⟩\bigr)$$

**(2) Vector view**

In the lexicographic order $|00⟩, |01⟩, |02⟩, |10⟩, |11⟩, |12⟩$:

$$|\psi⟩_{AB} \;⟺\; \frac{1}{\sqrt{3}}\begin{pmatrix}1\\0\\1\\0\\1\\0\end{pmatrix} \in \mathbb{C}^6$$

The relation between the "vector" and the "matrix" here can be understood as follows: concatenating the rows of the matrix one after another (**row-major order**) gives the long vector.

**(3) Computing the rank**

$$\mathbf{C} = \frac{1}{\sqrt{3}}\begin{pmatrix}1&0&1\\0&1&0\end{pmatrix}$$

Row 0 and row 1 are clearly linearly independent (one is nonzero at positions $j=0,2$, the other only at $j=1$), so $\text{rank}(\mathbf{C}) = 2$.

The Schmidt rank satisfies $r \leq \min(d_A, d_B) = \min(2,3) = 2$, and $\text{rank}(\mathbf{C}) = 2$, so we predict Schmidt rank $= 2$: this state is an **entangled state**.

(The specific Schmidt coefficients require actually computing the SVD; this is left for the next exercise.)
:::
::::

:::{prf:theorem} Schmidt Decomposition Theorem
:label: thm-schmidt-decomposition

Let $|\psi⟩_{AB} \in \mathcal{H}_A \otimes \mathcal{H}_B$ be a pure state. Then there exist **Schmidt bases**
- $\{|\alpha_k⟩_A\}_{k=0}^{r-1} \subset \mathcal{H}_A$ (a subset of an orthonormal basis of system A)
- $\{|\beta_k⟩_B\}_{k=0}^{r-1} \subset \mathcal{H}_B$ (a subset of an orthonormal basis of system B)

and **Schmidt coefficients** $\{\lambda_k > 0\}_{k=0}^{r-1}$ (satisfying $\sum_{k=0}^{r-1}\lambda_k^2 = 1$) such that

$$|\psi⟩_{AB} = \sum_{k=0}^{r-1}\lambda_k |\alpha_k⟩_A \otimes |\beta_k⟩_B$$

where $r$ is called the Schmidt rank, and the Schmidt coefficients are uniquely determined (counting multiplicity). When the Schmidt coefficients have repeated values (what physicists call degeneracy), the Schmidt basis within the corresponding subspace is not unique.

**Relation to the SVD**:
- the Schmidt coefficients are $\lambda_k = \sigma_k$ (the singular values of the coefficient matrix $\mathbf{C}$)
- the Schmidt basis vector $|\alpha_k⟩_A$ is the state corresponding to the $k$-th column of $\mathbf{U}$
- the Schmidt basis vector $|\beta_k⟩_B$ is the state corresponding to the $k$-th column of $\overline{\mathbf{V}}$ (entrywise conjugate)
:::

```{prf:proof}
From the SVD $\mathbf{C} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^*$, expand:

$$c_{ij} = \sum_{k=0}^{r-1}\sigma_k (\mathbf{U})_{ik} \overline{(\mathbf{V})_{jk}}$$

Substitute into the expansion of the state vector:

$$|\psi⟩_{AB} = \sum_{i,j}c_{ij}|i⟩_A|j⟩_B = \sum_{i,j}\sum_{k=0}^{r-1}\sigma_k (\mathbf{U})_{ik} \overline{(\mathbf{V})_{jk}} |i⟩_A|j⟩_B$$

Interchange the order of summation:

$$= \sum_{k=0}^{r-1}\sigma_k (\sum_i (\mathbf{U})_{ik} |i⟩_A) \otimes (\sum_j \overline{(\mathbf{V})_{jk}} |j⟩_B)$$

Define the Schmidt bases:

$$|\alpha_k⟩_A := \sum_{i=0}^{d_A-1} (\mathbf{U})_{ik}|i⟩_A, \quad |\beta_k⟩_B := \sum_{j=0}^{d_B-1} \overline{(\mathbf{V})_{jk}}|j⟩_B$$

That is, $|\alpha_k⟩_A$ is defined by the $k$-th column of $\mathbf{U}$, and $|\beta_k⟩_B$ by the $k$-th column of $\overline{\mathbf{V}}$. The columns of a unitary matrix form an orthonormal basis, and entrywise conjugation also preserves orthogonality, so both sets of Schmidt vectors are orthonormal systems.

Setting $\lambda_k = \sigma_k$, we obtain

$$\underline{|\psi⟩_{AB} = \sum_{k=0}^{r-1}\lambda_k |\alpha_k⟩_A \otimes |\beta_k⟩_B}$$

Checking the normalization:

$$\sum_{k=0}^{r-1}\lambda_k^2 = \sum_{k=0}^{r-1}\sigma_k^2 = \|\mathbf{C}\|_F^2 = 1 \quad \checkmark$$


```

**The meaning of the Schmidt decomposition**:

1. **Conciseness**: whereas $d_A \times d_B$ coefficients were originally needed, the Schmidt decomposition needs only $r$ coefficients ($r \leq \min(d_A, d_B)$)

2. **Symmetry**: the descriptions of the two subsystems A and B are completely symmetric, each with its own orthonormal basis

3. **Uniqueness**: the Schmidt coefficients are unique and directly describe the structure of the entanglement

:::{prf:example} Schmidt Decomposition of a Bell State
:label: ex-bell-schmidt-decomposition

Consider one of the most famous entangled states—a **Bell state** (also called an EPR pair):

$$|\Phi^+⟩ = \frac{1}{\sqrt{2}}(|00⟩ + |11⟩) = \frac{1}{\sqrt{2}}(|0⟩_A|0⟩_B + |1⟩_A|1⟩_B)$$

**Step 1.** Write down the coefficient matrix

In the computational basis $\{|0⟩, |1⟩\}$:

$$\mathbf{C} = \frac{1}{\sqrt{2}}\begin{bmatrix}1&0\\0&1\end{bmatrix} = \frac{1}{\sqrt{2}}\mathbf{I}_2$$

**Step 2.** SVD

The SVD of the identity matrix is trivial:

$$\mathbf{I}_2 = \mathbf{I}_2 \cdot \mathbf{I}_2 \cdot \mathbf{I}_2$$

Singular values: $\sigma_0 = \sigma_1 = 1$

**Step 3.** Schmidt decomposition

$$|\Phi^+⟩ = \frac{1}{\sqrt{2}}|0⟩_A|0⟩_B + \frac{1}{\sqrt{2}}|1⟩_A|1⟩_B$$

Schmidt coefficients: $\lambda_0 = \lambda_1 = 1/\sqrt{2}$

Schmidt rank: $r = 2$

Schmidt bases: $\{|0⟩_A, |1⟩_A\}$ and $\{|0⟩_B, |1⟩_B\}$ (just the computational basis itself)

**Observation.** The Bell state is a **maximally entangled state**: its two Schmidt coefficients are equal.
:::



::::{exercise} Two Views of a Bell State and Computing the Schmidt Coefficients
:label: exer-ch11-bell-svd

For the Bell state $|\Phi^+⟩ = \frac{1}{\sqrt{2}}(|00⟩ + |11⟩)$:

(1) Write down its vector representation (a column vector in $\mathbb{C}^4$).

(2) Write down the coefficient matrix $\mathbf{C}$ and compute its SVD.

(3) From the SVD, write explicit expressions for the Schmidt bases $|\alpha_k⟩_A$ and $|\beta_k⟩_B$ (using the formulas in the theorem).

(4) If we consider instead another Bell state $|\Psi^-⟩ = \frac{1}{\sqrt{2}}(|01⟩ - |10⟩)$, what is its coefficient matrix $\mathbf{C}'$? What are its singular values? What is its Schmidt rank?

:::{solution} exer-ch11-bell-svd
:class: dropdown

**(1) Vector view**

$$|\Phi^+⟩ \;⟺\; \frac{1}{\sqrt{2}}\begin{pmatrix}1\\0\\0\\1\end{pmatrix} \in \mathbb{C}^4$$

(corresponding in order to $|00⟩, |01⟩, |10⟩, |11⟩$.)

**(2) Coefficient matrix and SVD**

$$\mathbf{C} = \frac{1}{\sqrt{2}}\begin{pmatrix}1&0\\0&1\end{pmatrix} = \frac{1}{\sqrt{2}}\mathbf{I}_2$$

SVD: $\mathbf{I}_2 = \mathbf{I}_2 \cdot \mathbf{I}_2 \cdot \mathbf{I}_2$, so

$$\mathbf{C} = \underbrace{\mathbf{I}_2}_{\mathbf{U}} \cdot \underbrace{\frac{1}{\sqrt{2}}\mathbf{I}_2}_{\boldsymbol{\Sigma}} \cdot \underbrace{\mathbf{I}_2}_{\mathbf{V}^*}$$

Singular values: $\sigma_0 = \sigma_1 = \dfrac{1}{\sqrt{2}}$.

**(3) Explicit expressions for the Schmidt bases**

By the theorem, the Schmidt bases are given by the columns of $\mathbf{U}$ and the entrywise conjugates of the columns of $\mathbf{V}$:

$$|\alpha_k⟩_A = \sum_{i=0}^{1}(\mathbf{U})_{ik}|i⟩_A$$

Since $\mathbf{U} = \mathbf{I}_2$, we have $\mathbf{U}^* = \mathbf{I}_2$, and therefore:

- $|\alpha_0⟩_A = 1 \cdot |0⟩_A + 0 \cdot |1⟩_A = |0⟩_A$
- $|\alpha_1⟩_A = 0 \cdot |0⟩_A + 1 \cdot |1⟩_A = |1⟩_A$

Similarly, $\mathbf{V} = \mathbf{I}_2$, so:

- $|\beta_0⟩_B = |0⟩_B$, $|\beta_1⟩_B = |1⟩_B$

The Schmidt decomposition is just the computational basis itself, in agreement with the original expression:

$$|\Phi^+⟩ = \frac{1}{\sqrt{2}}|0⟩_A|0⟩_B + \frac{1}{\sqrt{2}}|1⟩_A|1⟩_B$$

**This shows** that the Bell state is "naturally" already in Schmidt form; no change of basis is needed.

**(4) Analysis of $|\Psi^-⟩$**

$$|\Psi^-⟩ = \frac{1}{\sqrt{2}}(|01⟩ - |10⟩) \;⟺\; \frac{1}{\sqrt{2}}\begin{pmatrix}0\\1\\-1\\0\end{pmatrix}$$

Coefficient matrix (reading off $c_{01} = 1/\sqrt{2}$ and $c_{10} = -1/\sqrt{2}$, all others 0):

$$\mathbf{C}' = \frac{1}{\sqrt{2}}\begin{pmatrix}0&1\\-1&0\end{pmatrix}$$

This is a skew-symmetric matrix. Compute its singular values:

$$\mathbf{C}'^* \mathbf{C}' = \frac{1}{2}\begin{pmatrix}0&-1\\1&0\end{pmatrix}\begin{pmatrix}0&1\\-1&0\end{pmatrix} = \frac{1}{2}\begin{pmatrix}1&0\\0&1\end{pmatrix}$$

So both eigenvalues are $1/2$, and the singular values are $\sigma_0 = \sigma_1 = 1/\sqrt{2}$.

The Schmidt rank is $r = 2$, so $|\Psi^-⟩$ is also a **maximally entangled state**, with the same entanglement structure as $|\Phi^+⟩$ (the same singular values), even though their state vectors differ.

**Key lesson.** The Schmidt coefficients, not the specific form of the coefficient matrix, are the essential quantities that describe entanglement.
:::
::::

In [ ]:
# ============================================================
print_header("Schmidt Decomposition: Numerical Check for a Bell State")
# ============================================================

# Bell state |Φ+⟩ = (1/√2)(|00⟩ + |11⟩)
# coefficient matrix C (row 0 corresponds to A=|0⟩, row 1 to A=|1⟩)
C = np.array([[1/np.sqrt(2), 0],
              [0,            1/np.sqrt(2)]])

print_step(1, "Take the SVD of the coefficient matrix C (i.e., the Schmidt decomposition)")
U, s, Vh = np.linalg.svd(C)
print("Schmidt coefficients λ_k =", s)
print("(theoretical values: λ_0 = λ_1 = 1/√2 ≈", round(1/np.sqrt(2), 4), ")")

print_step(2, "Schmidt rank = number of nonzero singular values")
r = np.sum(s > 1e-10)
print(f"Schmidt rank = {r}  →  {'entangled state (r > 1)' if r > 1 else 'separable state (r = 1)'}")

print_step(3, "Check the normalization condition Σ λ_k² = 1")
print(f"Σ λ_k² = {np.sum(s**2):.6f}  (should be 1.0)")

print_step(4, "von Neumann entanglement entropy S = -Σ λ_k² log₂(λ_k²)")
S = -np.sum(s**2 * np.log2(s**2 + 1e-15))
print(f"S = {S:.4f} bit  (maximal entanglement entropy of a Bell state = 1 bit ✓)")

:::{prf:example} Schmidt Decomposition of a Separable State
:label: ex-schmidt-separable

Consider a **separable state** (no entanglement):

$$|\psi⟩_{AB} = |\phi⟩_A \otimes |\chi⟩_B$$

Take the following concrete state as an example:

$$|\psi⟩ = (\frac{1}{\sqrt{2}}|0⟩_A + \frac{1}{\sqrt{2}}|1⟩_A) \otimes |0⟩_B$$

Expanding gives

$$|\psi⟩ = \frac{1}{\sqrt{2}}|00⟩ + \frac{1}{\sqrt{2}}|10⟩$$

The corresponding coefficient matrix is

$$\mathbf{C} = \frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix}\begin{bmatrix}1&0\end{bmatrix} = \frac{1}{\sqrt{2}}\begin{bmatrix}1&0\\1&0\end{bmatrix}$$

This is a matrix of rank 1, and its SVD has outer-product form:

$$\mathbf{C} = 1 \cdot \frac{1}{\sqrt{2}}\begin{bmatrix}1\\1\end{bmatrix}\begin{bmatrix}1&0\end{bmatrix}$$

The singular values are $\sigma_0 = 1$ and $\sigma_1 = 0$, so the Schmidt decomposition has only one term:

$$|\psi⟩_{AB} = 1 \cdot |\alpha_0⟩_A \otimes |\beta_0⟩_B$$

The Schmidt rank is $r = 1$, confirming that this state is separable.
:::

:::{prf:theorem} Separability Criterion
:label: thm-separability-criterion

A pure state $|\psi⟩_{AB}$ is **separable** (that is, can be written in the form $|\phi⟩_A \otimes |\chi⟩_B$) if and only if its **Schmidt rank is 1**.

Equivalently, a pure state is **entangled** if and only if its **Schmidt rank is $r \geq 2$**.
:::

**Proof.** This is immediate. Schmidt rank 1 means there is only one nonzero Schmidt coefficient, in which case $|\psi⟩ = \lambda_0|\alpha_0⟩_A|\beta_0⟩_B$ is separable. Conversely, if the Schmidt rank is $\geq 2$, there are at least two terms, and the state cannot be written as a single tensor product. $\square$

#### Schmidt Rank and the von Neumann Entanglement Entropy

The **Schmidt rank** $r$ is the coarsest measure of entanglement:

| Schmidt rank $r$ | Degree of entanglement | Physical meaning |
|---------------|---------|---------|
| $r = 1$ | no entanglement | separable state; the subsystems are completely independent |
| $r = 2$ | partial entanglement | the simplest entanglement, with two Schmidt terms |
| $r = \min(d_A, d_B)$ | full rank | possibly maximal entanglement (if all $\lambda_k$ are equal) |

But the Schmidt rank is not fine enough. For example:

$$|\psi_1⟩ = \frac{1}{\sqrt{2}}|00⟩ + \frac{1}{\sqrt{2}}|11⟩ \quad (\lambda_0 = \lambda_1 = 1/\sqrt{2})$$

$$|\psi_2⟩ = \sqrt{0.98}|00⟩ + \sqrt{0.02}|11⟩ \quad (\lambda_0 \approx 0.99, \lambda_1 \approx 0.14)$$

Both have Schmidt rank 2, but their degrees of entanglement are clearly different!

A finer measure is the **von Neumann entanglement entropy**:

$$S_{\text{ent}}(|\psi⟩_{AB}) := -\sum_{k=0}^{r-1}\lambda_k^2 \log_2(\lambda_k^2)$$

**Properties**:

1. **Nonnegativity**: $S_{\text{ent}} \geq 0$

2. **Zero for separable states**: $r = 1 \implies S_{\text{ent}} = 0$

3. **Maximally entangled states have the largest entropy**: for $d_A = d_B = d$, if $\lambda_k = 1/\sqrt{d}$ for all $k$, then

   $$S_{\text{ent}} = -d \cdot \frac{1}{d}\log_2(1/d) = \log_2 d$$

4. **Symmetry between subsystems**: $S_{\text{ent}}$ describes the entanglement between A and B and does not depend on which one is the "main" system

:::{prf:example} Computing the Entanglement Entropy
:label: ex-entanglement-entropy-computation

For the Bell state $|\Phi^+⟩$:

$$\lambda_0 = \lambda_1 = 1/\sqrt{2}, \quad \lambda_0^2 = \lambda_1^2 = 1/2$$

$$S_{\text{ent}} = -2 \cdot \frac{1}{2}\log_2(1/2) = -\log_2(1/2) = 1 \text{ bit}$$

This is the **maximal entanglement** of two qubits; the entanglement entropy is 1 bit.

For the weakly entangled state $|\psi_2⟩ = \sqrt{0.98}|00⟩ + \sqrt{0.02}|11⟩$:

$$\lambda_0^2 \approx 0.98, \quad \lambda_1^2 \approx 0.02$$

$$S_{\text{ent}} \approx -0.98\log_2(0.98) - 0.02\log_2(0.02) \approx 0.14 \text{ bit}$$

The degree of entanglement is far below that of the Bell state.
:::

::::{exercise} A Comprehensive Exercise: From the Coefficient Matrix to the Schmidt Decomposition
:label: exer-ch11-full-schmidt

Let $d_A = d_B = 2$, and let the coefficient matrix of a pure state be

$$\mathbf{C} = \begin{pmatrix}\cos\theta & 0 \\ 0 & \sin\theta\end{pmatrix}, \quad \theta \in [0, \frac{\pi}{4}]$$

(It is already normalized, because $\cos^2\theta + \sin^2\theta = 1$.)

(1) Write down the corresponding state vector (as an expansion in the $|ij⟩$).

(2) Write down the SVD directly (this matrix is itself already diagonal).

(3) Discuss the physical meaning of the state when $\theta = 0$ and when $\theta = \pi/4$.

(4) The Schmidt entropy (a measure of von Neumann entropy) is defined as $E = -\sum_k \lambda_k^2 \log_2 \lambda_k^2$. Compute $E$ in these two extreme cases and explain its meaning.


:::{solution} exer-ch11-full-schmidt
:class: dropdown

**(1) Expansion of the state vector**

$$|\psi(\theta)⟩_{AB} = \cos\theta\,|00⟩ + \sin\theta\,|11⟩$$

(because $c_{00} = \cos\theta$, $c_{11} = \sin\theta$, and all others are zero.)

**(2) SVD**

$\mathbf{C}$ is already a diagonal matrix, and its diagonal entries are nonnegative (on $\theta \in [0, \pi/4]$ we have $\cos\theta \geq \sin\theta \geq 0$), so

$$\mathbf{C} = \mathbf{I}_2 \cdot \begin{pmatrix}\cos\theta & 0\\0&\sin\theta\end{pmatrix} \cdot \mathbf{I}_2$$

that is, $\mathbf{U} = \mathbf{V} = \mathbf{I}_2$, and the Schmidt coefficients are $\lambda_0 = \cos\theta$ and $\lambda_1 = \sin\theta$.

The Schmidt decomposition is

$$|\psi(\theta)⟩_{AB} = \cos\theta\,|0⟩_A|0⟩_B + \sin\theta\,|1⟩_A|1⟩_B$$

**(3) The two extreme cases**

- **$\theta = 0$**: $\lambda_0 = 1$, $\lambda_1 = 0$, Schmidt rank $= 1$; the state is $|00⟩$, **completely separable**, and A and B are completely independent.

- **$\theta = \pi/4$**: $\lambda_0 = \lambda_1 = 1/\sqrt{2}$, Schmidt rank $= 2$; the state is $\frac{1}{\sqrt{2}}(|00⟩ + |11⟩) = |\Phi^+⟩$, which is exactly the **Bell state**, **maximally entangled**.

As $\theta$ increases from 0 to $\pi/4$, the degree of entanglement increases monotonically, and the distribution of the singular values of $\mathbf{C}$ changes from "all concentrated in one" to "spread evenly," echoing the discussion of matrix rank and low-rank approximation (§11.2).

**(4) Schmidt entropy**

- **$\theta = 0$**: there is only one nonzero term, $\lambda_0^2 = 1$:

  $$E = -1 \cdot \log_2 1 = 0$$

  Zero entropy corresponds to no entanglement (a completely definite state).

- **$\theta = \pi/4$**: $\lambda_0^2 = \lambda_1^2 = 1/2$:

  $$E = -\frac{1}{2}\log_2\frac{1}{2} - \frac{1}{2}\log_2\frac{1}{2} = 1 \text{ ebit}$$

  1 ebit is the **maximal amount of entanglement**, corresponding to a completely even distribution of singular values—just as for the matrices in §11.2 that are hardest to approximate by low rank, when the singular values are evenly distributed "every direction is equally important," and the same holds for quantum entanglement.

This computation reveals a deep connection: **the SVD is not only a tool for matrix decomposition; for a bipartite pure state, the distribution of the singular values describes its entanglement structure up to equivalence under local unitary transformations.**
:::
::::

---

### 11.5.2 Reduced Density Matrices and Purification
#### The Partial Trace and Quantum Measurement

The partial trace and the projective measurement of §10.4 are related but different operations, and they deserve to be distinguished carefully.

**Measuring B and the conditional pure state of A**

For a composite pure state $|\psi⟩_{AB}=\sum_{i,j}c_{ij}|i⟩_A|j⟩_B$, perform a rank-1 projective measurement of B in any orthonormal basis; if outcome $j_0$ has positive probability, the conditional state of A is the **definite pure state** obtained by normalizing $\sum_i c_{i,j_0}|i⟩_A$. The A states corresponding to different outcomes need not be orthogonal.

When measuring in the Schmidt basis $\{|\beta_k⟩_B\}$, $|\psi⟩_{AB}=\sum_k\lambda_k|\alpha_k⟩_A|\beta_k⟩_B$, and each outcome $k$ of positive probability corresponds to $|\alpha_k⟩_A$, giving orthogonal conditional states in one-to-one correspondence. A definite conditional pure state does not mean that measuring any observable on it shows no statistical fluctuation.



**The partial trace corresponds to the average with "outcomes discarded"**

Still measuring B in the basis $\{|j⟩_B\}$, the probability of outcome $j$ is

$$P(j) = \sum_i |c_{ij}|^2$$

Only when $P(j)>0$ is the corresponding conditional pure state of A defined, namely $|\phi_j⟩_A = \dfrac{1}{\sqrt{P(j)}}\sum_i c_{ij}|i⟩_A$, with density matrix $\rho_A^{(j)} = |\phi_j⟩_A⟨ \phi_j|$.

If the outcome is **not read** after the measurement (or B simply cannot be accessed), the effective state of A is the probability-weighted average of all the conditional states:

$$\rho_A^{\text{avg}} = \sum_{j:P(j)>0} P(j)\,\rho_A^{(j)} = \sum_{j:P(j)>0} P(j)\,|\phi_j⟩_A⟨ \phi_j|$$

This weighted average does not depend on the choice of measurement basis for B and equals the partial trace $\operatorname{Tr}_B(\rho_{AB})$. We verify this explicitly below in the language of matrix indices.

:::{prf:proposition} The Partial Trace Equals the Weighted Average with Outcomes Discarded
:label: prop-partial-trace-average

Let $|\psi⟩_{AB} = \sum_{i,k}c_{ik}|i⟩_A|k⟩_B$ be a composite pure state. When B is measured in the computational basis $\{|k⟩_B\}$ and the **outcome is discarded**, the effective density matrix of A (the probability-weighted average of all conditional states $\rho_A^{(j)}=|\phi_j⟩_A⟨ \phi_j|$) equals exactly the partial trace over B:

$$\rho_A^{\text{avg}} = \sum_{j:P(j)>0} P(j)\,|\phi_j⟩_A⟨ \phi_j| = \operatorname{Tr}_B(\rho_{AB})$$
:::

:::{prf:proof} The Partial Trace Equals the Weighted Average with Outcomes Discarded
:label: prf-partial-trace-average

1. **Setting up the index correspondence**

   The density matrix $\rho_{AB}$ of the composite system is a $d_A d_B \times d_A d_B$ matrix. By the 4D tensor convention, the entry $\mathcal{M}[i,j,k,l]$ corresponds to the matrix position $(d_B \cdot i+k,\; d_B \cdot j+l)$, where the indices $(i,j)$ are the block position for system A ($i,j \in \{0,\ldots,d_A-1\}$) and the indices $(k,l)$ are the position within the block for system B ($k,l \in \{0,\ldots,d_B-1\}$).

   The entries of the density matrix of the pure state $|\psi⟩_{AB} = \sum_{i,k}c_{ik}|i⟩_A|k⟩_B$ are

   $$\mathcal{M}[i,j,k,l] = c_{ik}\,c_{jl}^*$$

2. **Applying the partial trace formula**

   Taking the partial trace over system B, the $(i,j)$ entry of the result is

   $$[\operatorname{Tr}_B(\rho_{AB})]_{ij} = \sum_{k=0}^{d_B-1} \mathcal{M}[i,j,k,k] = \sum_k c_{ik}\,c_{jk}^*$$

   The meaning of the operation: fix A's row and column indices $(i,j)$, force B's row and column indices to be equal ($k=l$), and sum—this is exactly "summing along B's diagonal direction," that is, the action of the trace on a subsystem.

3. **Agreement with the weighted average**

   Normalized conditional states are defined only for outcomes with $P(k)>0$. If $P(k)=0$, then all $c_{ik}=0$, so the unnormalized outer-product term is zero and can still be included in the final sum of coefficients.

   Substituting $|\phi_k⟩_A = \dfrac{1}{\sqrt{P(k)}}\sum_i c_{ik}|i⟩_A$:

   $$[\sum_{k:P(k)>0} P(k)\,|\phi_k⟩_A⟨ \phi_k|]_{ij}
   = \sum_{k:P(k)>0} P(k)\cdot\frac{c_{ik}}{\sqrt{P(k)}}\cdot\frac{c_{jk}^*}{\sqrt{P(k)}}
   = \sum_k c_{ik}\,c_{jk}^*$$

   $$\underline{[\operatorname{Tr}_B(\rho_{AB})]_{ij} = \sum_k c_{ik}\,c_{jk}^* = [\sum_{k:P(k)>0} P(k)\,|\phi_k⟩_A⟨ \phi_k|]_{ij}}$$

$\square$

:::

---

**Why the partial trace does not depend on the measurement basis of B**

Suppose we choose another orthonormal basis $\{|\tilde{j}⟩_B\}$ of B, related to the original computational basis $\{|k⟩_B\}$ by a unitary matrix $\mathbf{W}$:

$$|\tilde{j}⟩_B = \sum_k W_{kj}|k⟩_B$$

When B is measured in the new basis, the projection operator acting on A is ${}_B⟨ \tilde{j}|\cdot⟩$. Expand the dual vector:

$${}_B⟨ \tilde{j}| = (\sum_k W_{kj}|k⟩_B)^* = \sum_k W_{kj}^*\,{}_B⟨  k|$$

Apply it to $|\psi⟩_{AB} = \sum_{i,k'}c_{ik'}|i⟩_A|k'⟩_B$, using ${}_B⟨  k|k'⟩_B = \delta_{kk'}$:

$${}_B⟨ \tilde{j}|\psi⟩_{AB}
= \sum_k W_{kj}^*\sum_{i,k'} c_{ik'}\,{}_B⟨  k|k'⟩_B\,|i⟩_A
= \sum_{i,k} c_{ik}\,W_{kj}^*\,|i⟩_A$$

Grouping the terms by $i$, we recognize the matrix product $[\mathbf{C}\overline{\mathbf{W}}]_{ij} = \sum_k c_{ik}\,W_{kj}^*$:

$${}_B⟨ \tilde{j}|\psi⟩_{AB} = \sum_i(\sum_k c_{ik}\,W_{kj}^*)|i⟩_A = \sum_i [\mathbf{C}\overline{\mathbf{W}}]_{ij}\,|i⟩_A$$

In other words, changing the basis amounts to replacing the coefficient matrix $\mathbf{C}$ with $\mathbf{C}\overline{\mathbf{W}}$.

:::{prf:proposition} The Partial Trace Does Not Depend on the Measurement Basis of B
:label: prop-partial-trace-basis-independent

Computing the weighted average above in **any** orthonormal basis of B gives the same result, which always equals

$$\operatorname{Tr}_B(\rho_{AB}) = \mathbf{C}\mathbf{C}^*, \qquad [\rho_A]_{ii'} = \sum_k c_{ik}\,c_{i'k}^*$$

independently of the unitary freedom $\mathbf{W}$ in the chosen measurement basis.
:::


:::{prf:proof} The Partial Trace Does Not Depend on the Measurement Basis of B
:label: prf-partial-trace-basis-independent

The $(i,i')$ matrix entry of the weighted average in the new basis is

$$[\sum_{j:\tilde{P}(j)>0} \tilde{P}(j)\,\tilde{\rho}_A^{(j)}]_{ii'} = \sum_j [\mathbf{C}\overline{\mathbf{W}}]_{ij}\,[\mathbf{C}\overline{\mathbf{W}}]_{i'j}^*$$

Note that $[\mathbf{C}\overline{\mathbf{W}}]_{i'j}^* = [\mathbf{W}^{\top}\mathbf{C}^*]_{ji'}$, so the right-hand side can be rearranged into a matrix product ($j$ is the summation index):

$$= \sum_j [\mathbf{C}\overline{\mathbf{W}}]_{ij}\,[\mathbf{W}^{\top}\mathbf{C}^*]_{ji'} = [\mathbf{C}\overline{\mathbf{W}}\mathbf{W}^{\top}\mathbf{C}^*]_{ii'}$$

Since $\mathbf{W}$ is unitary, $\overline{\mathbf{W}}\mathbf{W}^{\top} = \mathbf{I}$, and therefore

$$\underline{[\mathbf{C}\overline{\mathbf{W}}\mathbf{W}^{\top}\mathbf{C}^*]_{ii'} = [\mathbf{C}\mathbf{C}^*]_{ii'} = \sum_k c_{ik}\,c_{i'k}^* = [\rho_A]_{ii'}}$$

The result is exactly the same as in the original basis; $\mathbf{W}$ cancels out completely. $\square$

:::

**Intuition.** A change of basis for B turns the coefficient matrix into $\mathbf{C}\overline{\mathbf{W}}$, where the bar denotes entrywise conjugation. The product in the partial trace is $\mathbf{C}\overline{\mathbf{W}}\mathbf{W}^{\top}\mathbf{C}^*=\mathbf{C}\mathbf{C}^*$, so the result does not depend on the measurement basis of B.

#### Reduced Density Matrices and the Schmidt Decomposition

The **reduced density matrix** describes the state of a subsystem. For a pure state $|\psi⟩_{AB}$, the reduced density matrix of system A is defined as

$$\rho_A := \text{Tr}_B(|\psi⟩_{AB}⟨ \psi|)$$

where $\text{Tr}_B$ denotes the **partial trace** over system B.

:::{prf:theorem} The Schmidt Decomposition and Reduced Density Matrices
:label: thm-schmidt-reduced-density

Let $|\psi⟩_{AB}$ have Schmidt decomposition

$$|\psi⟩_{AB} = \sum_{k=0}^{r-1}\lambda_k |\alpha_k⟩_A |\beta_k⟩_B$$

Then the reduced density matrices are

$$\rho_A = \sum_{k=0}^{r-1}\lambda_k^2 |\alpha_k⟩_A⟨ \alpha_k|$$

$$\rho_B = \sum_{k=0}^{r-1}\lambda_k^2 |\beta_k⟩_B⟨ \beta_k|$$

**Key properties**:

1. $\rho_A$ and $\rho_B$ have **the same nonzero eigenvalues** $\{\lambda_k^2\}$
2. These eigenvalues are exactly the squares of the Schmidt coefficients
3. The **entanglement entropy** can be computed from the reduced density matrices:

   $$S_{\text{ent}} = -\text{Tr}(\rho_A \log_2 \rho_A) = -\text{Tr}(\rho_B \log_2 \rho_B)$$

   This is called the **von Neumann entropy**.
:::

```{prf:proof}
Compute $\rho_A$:

$$\rho_A = \text{Tr}_B(|\psi⟩⟨ \psi|)$$

$$= \text{Tr}_B(\sum_{k,l}\lambda_k\lambda_l |\alpha_k⟩_A|\beta_k⟩_B ⟨ \beta_l|_B⟨ \alpha_l|_A)$$

The partial trace sums over system B:

$$= \sum_{k,l}\lambda_k\lambda_l |\alpha_k⟩_A⟨ \alpha_l|_A \cdot ⟨ \beta_l|\beta_k⟩_B$$

Since the Schmidt bases are orthogonal, $⟨ \beta_l|\beta_k⟩ = \delta_{lk}$:

$$= \sum_k \lambda_k^2 |\alpha_k⟩_A⟨ \alpha_k|_A$$

This is exactly a diagonal matrix, with eigenvalues $\lambda_k^2$ and eigenvectors $|\alpha_k⟩_A$.

Similarly one proves $\rho_B = \sum_k \lambda_k^2 |\beta_k⟩_B⟨ \beta_k|_B$. $\square$
```


---

::::{exercise} Computing a Partial Trace with Matrices
:label: exer-ch11-partial-trace

Let $d_A = d_B = 2$, and let the pure state of the composite system be

$$|\psi⟩_{AB} = \frac{1}{\sqrt{2}}(|00⟩ + |11⟩) = |\Phi^+⟩$$

(1) Write down the density matrix of the whole system $\rho_{AB} = |\psi⟩⟨ \psi|$ (a $4\times 4$ matrix in the basis $|00⟩,|01⟩,|10⟩,|11⟩$).

(2) Using the definition of the partial trace $\rho_A = \sum_{j=0}^{1} {}_B⟨  j|\rho_{AB}|j⟩_B$, compute $\rho_A$ directly.

(3) Verify that the eigenvalues of $\rho_A$ equal the squares of the Schmidt coefficients.

(4) Compute $\text{Tr}(\rho_A^2)$ and explain whether $\rho_A$ is a pure state or a mixed state.


:::{solution} exer-ch11-partial-trace
:class: dropdown
**(1) The density matrix of the whole system**

$$\rho_{AB} = |\Phi^+⟩⟨ \Phi^+| = \frac{1}{2}(|00⟩ + |11⟩)(⟨  00| + ⟨  11|)$$

$$= \frac{1}{2}(|00⟩⟨  00| + |00⟩⟨  11| + |11⟩⟨  00| + |11⟩⟨  11|)$$

In the basis $\{|00⟩, |01⟩, |10⟩, |11⟩\}$ (in this order):

$$\rho_{AB} = \frac{1}{2}\begin{pmatrix}1&0&0&1\\0&0&0&0\\0&0&0&0\\1&0&0&1\end{pmatrix}$$

(The entries at positions $(0,0)$, $(0,3)$, $(3,0)$, $(3,3)$ are each $1/2$, corresponding to the terms $|00⟩⟨  00|$, $|00⟩⟨  11|$, and so on.)

**(2) Computing $\rho_A$**

The definition of the partial trace: "sandwich $\rho_{AB}$ between the basis vectors $\{|0⟩_B, |1⟩_B\}$ of system B, and then sum over all the sandwiches."

$$\rho_A = {}_B⟨  0|\rho_{AB}|0⟩_B + {}_B⟨  1|\rho_{AB}|1⟩_B$$

Compute each term. Note that $|ij⟩ = |i⟩_A \otimes |j⟩_B$, so ${}_B⟨  0|ij⟩ = |i⟩_A \delta_{j0}$:

$${}_B⟨  0|\rho_{AB}|0⟩_B = \frac{1}{2}|0⟩_A⟨  0|_A$$

$${}_B⟨  1|\rho_{AB}|1⟩_B = \frac{1}{2}|1⟩_A⟨  1|_A$$

Therefore

$$\rho_A = \frac{1}{2}|0⟩_A⟨  0|_A + \frac{1}{2}|1⟩_A⟨  1|_A = \frac{1}{2}\mathbf{I}_2 = \frac{1}{2}\begin{pmatrix}1&0\\0&1\end{pmatrix}$$

This is the **maximally mixed state**.

**(3) Checking the eigenvalues**

The eigenvalues of $\rho_A = \frac{1}{2}\mathbf{I}_2$ are both $\frac{1}{2}$.

The Schmidt coefficients of the Bell state are $\lambda_0 = \lambda_1 = \frac{1}{\sqrt{2}}$, whose squares are $\lambda_0^2 = \lambda_1^2 = \frac{1}{2}$. ✓

**(4) Computing the purity**

$$\rho_A^2 = \frac{1}{4}\mathbf{I}_2, \quad \text{Tr}(\rho_A^2) = \frac{1}{4} + \frac{1}{4} = \frac{1}{2} < 1$$

$\rho_A$ is a **mixed state**. This is not because our knowledge of A is incomplete, but because A and B are deeply entangled—even though we have a complete pure-state description $|\Phi^+⟩$ of the composite system, the subsystem A still behaves as a mixed state.

**Physical interpretation.** The Bell state is a pure state (completely definite as a whole), yet its subsystems are maximally mixed states (completely indefinite locally). This is precisely the fundamental difference between quantum entanglement and classical correlation.
:::
::::

---

**Deeper implications**:

The Schmidt decomposition of a pure state $|\psi⟩_{AB}$ directly gives:

1. **The mixedness of the subsystems**: $\rho_A$ and $\rho_B$ are mixed states (unless $r=1$)
2. **The equivalence of entanglement and mixedness**: the degree of entanglement of a pure state of the composite system = the degree of mixedness of the subsystems
3. **Conservation of information**: the von Neumann entropies of the two subsystems are equal

This shows that **entanglement is the unity of global purity and local mixedness**.

---

::::{exercise} The Symmetry Between $\rho_A$ and $\rho_B$
:label: exer-ch11-reduced-symmetry

For a general two-qubit pure state ($d_A = d_B = 2$), let the Schmidt decomposition be

$$|\psi(\theta)⟩ = \cos\theta\,|00⟩ + \sin\theta\,|11⟩, \quad \theta \in [0, \frac{\pi}{4}]$$

(This is the family of states in Exercise {ref}`exer-ch11-full-schmidt` of §11.5.1.)

(1) Compute $\rho_A$ and $\rho_B$ and verify that they have the same eigenvalues.

(2) Compute the von Neumann entropy $S_{\text{ent}}(\theta) = -\cos^2\theta \log_2\cos^2\theta - \sin^2\theta\log_2\sin^2\theta$ (with the convention $0\log_2 0 = 0$), and discuss its values at $\theta = 0$ and $\theta = \pi/4$.

(3) Compute $\text{Tr}(\rho_A^2)$ in terms of $\theta$, and find the value of $\theta$ that makes $\rho_A$ most mixed.

(4) **Connection with §10.4**: if B is measured projectively in the basis $\{|0⟩_B, |1⟩_B\}$ and the **outcome is read**, what are the two possible conditional states (the post-collapse states of system A)? Compare them with $\rho_A$.


:::{solution} exer-ch11-reduced-symmetry
:class: dropdown

**(1) $\rho_A$, $\rho_B$, and their eigenvalues**

By the theorem of §11.5.2, the Schmidt coefficients are $\lambda_0 = \cos\theta$ and $\lambda_1 = \sin\theta$, so

$$\rho_A = \cos^2\theta\,|0⟩_A⟨  0| + \sin^2\theta\,|1⟩_A⟨  1| = \begin{pmatrix}\cos^2\theta & 0 \\ 0 & \sin^2\theta\end{pmatrix}$$

$$\rho_B = \cos^2\theta\,|0⟩_B⟨  0| + \sin^2\theta\,|1⟩_B⟨  1| = \begin{pmatrix}\cos^2\theta & 0 \\ 0 & \sin^2\theta\end{pmatrix}$$

The two have exactly the same form, and both have eigenvalues $\{\cos^2\theta,\, \sin^2\theta\}$.

**Note.** $\rho_A$ and $\rho_B$ have the same eigenvalues, but they live in different Hilbert spaces $\mathcal{H}_A$ and $\mathcal{H}_B$—this symmetry is a direct consequence of the Schmidt decomposition and has no classical analogue.

**(2) The von Neumann entropy**

- $\theta = 0$: $\rho_A = |0⟩⟨  0|$, $S_{\text{ent}} = -1\cdot\log_2 1 - 0 = 0$ (a pure state, no entanglement)
- $\theta = \pi/4$: $\rho_A = \frac{1}{2}\mathbf{I}_2$, $S_{\text{ent}} = -\frac{1}{2}\log_2\frac{1}{2} - \frac{1}{2}\log_2\frac{1}{2} = 1$ ebit (maximal entanglement)

The entropy increases monotonically on $[0,\pi/4]$, from zero to the maximum, corresponding to a continuous transition from completely separable to maximally entangled.

**(3) The purity $\text{Tr}(\rho_A^2)$**

$$\text{Tr}(\rho_A^2) = \cos^4\theta + \sin^4\theta = (\cos^2\theta + \sin^2\theta)^2 - 2\cos^2\theta\sin^2\theta = 1 - \frac{1}{2}\sin^2 2\theta$$

The purity attains its minimum $1/2$ at $\theta = \pi/4$, where $\sin 2\theta = 1$; that is where $\rho_A$ is most mixed. The purity decreases monotonically from $1$ (a pure state) at $\theta = 0$ to $1/2$ (maximally mixed) at $\theta = \pi/4$.

**(4) Connection with §10.4—conditional states vs. the average state**

Measure B in $\{|0⟩_B, |1⟩_B\}$:

Outcome $|0⟩_B$ (probability $\cos^2\theta$): A collapses to the **conditional state** $|0⟩_A$, which is the pure state $|0⟩_A⟨  0|$.

Outcome $|1⟩_B$ (probability $\sin^2\theta$): A collapses to the **conditional state** $|1⟩_A$, which is the pure state $|1⟩_A⟨  1|$.

If B is measured but the **outcome is not read**, the **average state** of A is

$$\rho_A^{\text{avg}} = \cos^2\theta\cdot|0⟩_A⟨  0| + \sin^2\theta\cdot|1⟩_A⟨  1| = \rho_A \quad \checkmark$$

This agrees completely with the result of the partial trace, confirming the statement at the beginning of this subsection: **partial trace = the average state of A after measuring system B and discarding the outcome**.
:::
::::

---

#### Purification

Purification is a profound concept: every mixed state can be regarded as the reduced density matrix of some larger pure state.

:::{prf:theorem} Purification Theorem
:label: thm-purification

Let $\rho_A$ be a mixed state of system A with spectral decomposition

$$\rho_A = \sum_{k=0}^{r-1} p_k |\phi_k⟩_A⟨ \phi_k|_A, \quad p_k > 0, \sum_k p_k = 1$$

Then there exist an auxiliary system B (of dimension $d_B \geq r$) and a pure state $|\psi⟩_{AB}$ such that

$$\rho_A = \text{Tr}_B(|\psi⟩_{AB}⟨ \psi|)$$

**Standard purification**:

$$|\psi⟩_{AB} = \sum_{k=0}^{r-1}\sqrt{p_k}|\phi_k⟩_A \otimes |k⟩_B$$

where $\{|k⟩_B\}$ is any orthonormal basis of system B.
:::

```{prf:proof}
Verify directly:

$$|\psi⟩_{AB}⟨ \psi| = \sum_{k,l}\sqrt{p_kp_l}|\phi_k⟩_A⟨ \phi_l|_A \otimes |k⟩_B⟨  l|_B$$

Take the partial trace:

$$\text{Tr}_B(|\psi⟩⟨ \psi|) = \sum_{k,l}\sqrt{p_kp_l}|\phi_k⟩_A⟨ \phi_l|_A \cdot ⟨  l|k⟩_B$$

$$= \sum_k p_k |\phi_k⟩_A⟨ \phi_k|_A = \rho_A \quad \checkmark$$

$\square$
```

---

::::{exercise} Non-Uniqueness of Purification
:label: exer-ch11-purification-nonunique

Consider the mixed state

$$\rho_A = \frac{1}{2}|0⟩⟨  0| + \frac{1}{2}|1⟩⟨  1| = \frac{1}{2}\mathbf{I}_2$$

(1) Write down the standard purification $|\psi_1⟩_{AB}$ (taking $|k⟩_B = |k⟩_B$ to be the computational basis).

(2) Verify that the following is also a purification of $\rho_A$:

$$|\psi_2⟩_{AB} = \frac{1}{\sqrt{2}}(|0⟩_A|+⟩_B + |1⟩_A|-⟩_B)$$

where $|+⟩_B = \frac{1}{\sqrt{2}}(|0⟩_B + |1⟩_B)$ and $|-⟩_B = \frac{1}{\sqrt{2}}(|0⟩_B - |1⟩_B)$.

(Hint: compute $\text{Tr}_B(|\psi_2⟩⟨ \psi_2|)$, using $\{|+⟩_B, |-⟩_B\}$ as the basis of system B for the partial trace.)

(3) Explain why purification is not unique—in which degree of freedom of system B does the arbitrariness lie?

(4) What are the Schmidt decompositions of these two purified states? Are their Schmidt coefficients the same?


:::{solution} exer-ch11-purification-nonunique
:class: dropdown

**(1) The standard purification**

The spectral decomposition of $\rho_A = \frac{1}{2}|0⟩⟨  0| + \frac{1}{2}|1⟩⟨  1|$: $p_0 = p_1 = \frac{1}{2}$, $|\phi_0⟩_A = |0⟩_A$, $|\phi_1⟩_A = |1⟩_A$.

The standard purification is

$$|\psi_1⟩_{AB} = \frac{1}{\sqrt{2}}|0⟩_A|0⟩_B + \frac{1}{\sqrt{2}}|1⟩_A|1⟩_B = |\Phi^+⟩$$

This is exactly the Bell state!

**(2) Verifying that $|\psi_2⟩_{AB}$ is also a purification**

Use $\{|+⟩_B, |-⟩_B\}$ as the basis for the partial trace:

$${}_B⟨ +|\psi_2⟩=\frac{|0⟩_A}{\sqrt{2}},\qquad {}_B⟨ -|\psi_2⟩=\frac{|1⟩_A}{\sqrt{2}}$$

Adding the two unnormalized outer products gives $\rho_A=\mathbf{I}_2/2$. One can also expand $|\psi_2⟩_{AB}$:

$$|\psi_2⟩_{AB} = \frac{1}{\sqrt{2}}|0⟩_A \cdot \frac{|0⟩_B + |1⟩_B}{\sqrt{2}} + \frac{1}{\sqrt{2}}|1⟩_A \cdot \frac{|0⟩_B - |1⟩_B}{\sqrt{2}}$$

$$= \frac{1}{2}|0⟩_A|0⟩_B + \frac{1}{2}|0⟩_A|1⟩_B + \frac{1}{2}|1⟩_A|0⟩_B - \frac{1}{2}|1⟩_A|1⟩_B$$

Take the partial trace in the computational basis of system B:

$$\rho_A = {}_B⟨  0|\psi_2⟩⟨ \psi_2|0⟩_B + {}_B⟨  1|\psi_2⟩⟨ \psi_2|1⟩_B$$

$${}_B⟨  0|\psi_2⟩ = \frac{1}{2}|0⟩_A + \frac{1}{2}|1⟩_A = \frac{1}{2}(|0⟩_A + |1⟩_A)$$

$${}_B⟨  1|\psi_2⟩ = \frac{1}{2}|0⟩_A - \frac{1}{2}|1⟩_A = \frac{1}{2}(|0⟩_A - |1⟩_A)$$

$$\rho_A = \frac{1}{4}(|0⟩_A + |1⟩_A)(⟨  0|_A + ⟨  1|_A) + \frac{1}{4}(|0⟩_A - |1⟩_A)(⟨  0|_A - ⟨  1|_A)$$

$$= \frac{1}{4}\begin{pmatrix}1&1\\1&1\end{pmatrix} + \frac{1}{4}\begin{pmatrix}1&-1\\-1&1\end{pmatrix} = \frac{1}{4}\begin{pmatrix}2&0\\0&2\end{pmatrix} = \frac{1}{2}\mathbf{I}_2 = \rho_A \quad \checkmark$$

**(3) The source of the arbitrariness**

Purification is free up to a **unitary transformation** on system B. If $|\psi⟩_{AB}$ is a purification of $\rho_A$, then $(\mathbf{I}_A \otimes \mathbf{W}_B)|\psi⟩_{AB}$ is also a purification of $\rho_A$ for any unitary matrix $\mathbf{W}_B$ (because $\mathbf{W}_B$ is "integrated out" in the partial trace).

Concretely, $|\psi_2⟩_{AB} = (\mathbf{I}_A \otimes \mathbf{H}_B)|\psi_1⟩_{AB}$, where $\mathbf{H}_B$ is the Hadamard matrix:

$$\mathbf{H} = \frac{1}{\sqrt{2}}\begin{pmatrix}1&1\\1&-1\end{pmatrix}, \quad \mathbf{H}|0⟩ = |+⟩, \quad \mathbf{H}|1⟩ = |-⟩$$

**(4) Comparing the Schmidt decompositions**

$|\psi_1⟩_{AB} = |\Phi^+⟩$ is already in Schmidt form: coefficients $\lambda_0 = \lambda_1 = \frac{1}{\sqrt{2}}$, Schmidt rank $= 2$.

The Schmidt decomposition of $|\psi_2⟩_{AB}$: note that it equals

$$|\psi_2⟩_{AB} = \frac{1}{\sqrt{2}}|0⟩_A|+⟩_B + \frac{1}{\sqrt{2}}|1⟩_A|-⟩_B$$

Since $\{|+⟩_B, |-⟩_B\}$ is an orthonormal basis of system B and $\{|0⟩_A, |1⟩_A\}$ is an orthonormal basis of system A, this is already in Schmidt form: the coefficients are again $\lambda_0 = \lambda_1 = \frac{1}{\sqrt{2}}$, and Schmidt rank $= 2$.

**Key conclusion.** Two different purified states (obtained by applying different unitary transformations on B) have **exactly the same Schmidt coefficients**, because the Schmidt coefficients are uniquely determined by the eigenvalues of $\rho_A$ and do not depend on the choice of basis for system B.
:::
::::

---

**Physical meaning**:

A mixed state can be regarded as the reduced state of a larger pure state; in this purified description, the mixedness comes from the entanglement between the system and the auxiliary system.

If an interaction builds up non-negligible entanglement between the system and its environment, then ignoring the environment may produce mixing and decoherence in the corresponding basis. Interaction by itself does not guarantee that this happens; the purification theorem is an existence result and does not specify the dynamics of any actual environment.

:::{prf:example} Purifying a Mixed State
:label: ex-purification-mixed-state

Consider a single-qubit mixed state:

$$\rho_A = \frac{2}{3}|0⟩⟨  0| + \frac{1}{3}|1⟩⟨  1|$$

**Step 1.** Identify the eigenvalues and eigenvectors

$p_0 = 2/3$, $p_1 = 1/3$; $|\phi_0⟩_A = |0⟩$, $|\phi_1⟩_A = |1⟩$

**Step 2.** Construct the purification

Introduce an auxiliary system B (also a qubit). The standard purification is

$$|\psi⟩_{AB} = \sqrt{\frac{2}{3}}|0⟩_A|0⟩_B + \sqrt{\frac{1}{3}}|1⟩_A|1⟩_B$$

**Step 3.** Verify (partial trace in the computational basis)

$${}_B⟨  0|\psi⟩ = \sqrt{\frac{2}{3}}|0⟩_A, \quad {}_B⟨  1|\psi⟩ = \sqrt{\frac{1}{3}}|1⟩_A$$

$$\rho_A = \frac{2}{3}|0⟩⟨  0| + \frac{1}{3}|1⟩⟨  1| \quad \checkmark$$

**Schmidt decomposition.** The Schmidt decomposition of this purified state is the state itself!

Schmidt coefficients: $\lambda_0 = \sqrt{2/3}$, $\lambda_1 = \sqrt{1/3}$

Entanglement entropy:

$$S_{\text{ent}} = -\frac{2}{3}\log_2\frac{2}{3} - \frac{1}{3}\log_2\frac{1}{3} \approx 0.918 \text{ bit}$$
:::

---

::::{exercise} Schmidt Rank of a Purification and Mixedness
:label: exer-ch11-purification-rank

Let $\rho_A$ be a mixed state of a $d_A$-dimensional system with rank $r$ (that is, with $r$ nonzero eigenvalues).

(1) What is the Schmidt rank of a purification $|\psi⟩_{AB}$? What is the minimum dimension of the auxiliary system B?

(2) If $\rho_A = \frac{1}{d}\mathbf{I}_d$ (the maximally mixed state), what is special about the Schmidt coefficients of its purification?

(3) "Thinking in reverse": given a purification $|\psi⟩_{AB}$ with Schmidt rank $r$, what is the rank of the reduced density matrix $\rho_A$?

(4) In what sense does this explain the relationship "**mixedness = complexity of entanglement**"?

:::{solution} exer-ch11-purification-rank
:class: dropdown

**(1) Schmidt rank and the dimension of the auxiliary system**

The standard purification $|\psi⟩_{AB} = \sum_{k=0}^{r-1}\sqrt{p_k}|\phi_k⟩_A|k⟩_B$ has exactly $r$ terms, each corresponding to linearly independent A states and B states, so the **Schmidt rank = $r$** (the rank of $\rho_A$).

The minimum dimension of the auxiliary system B is $d_B^{\min} = r$, because it must accommodate $r$ linearly independent Schmidt basis vectors.

If $d_B < r$, the Hilbert space of system B is too small to hold $r$ orthogonal vectors $|\beta_k⟩_B$, and the required purification cannot be constructed.

**(2) Purification of the maximally mixed state**

The eigenvalues of $\rho_A = \frac{1}{d}\mathbf{I}_d$ are all equal: $p_k = \frac{1}{d}$, $k=0,\ldots,d-1$.

Purification:

$$|\psi⟩_{AB} = \frac{1}{\sqrt{d}}\sum_{k=0}^{d-1}|\phi_k⟩_A|k⟩_B$$

The Schmidt coefficients are all equal: $\lambda_k = \frac{1}{\sqrt{d}}$. This is a maximally entangled state (in a $d\times d$ system), and the Schmidt entropy attains its maximum value $\log_2 d$.

**(3) Thinking in reverse**

If $|\psi⟩_{AB}$ has Schmidt rank $r$, then by the theorem,

$$\rho_A = \sum_{k=0}^{r-1}\lambda_k^2|\alpha_k⟩_A⟨ \alpha_k|$$

has $r$ nonzero eigenvalues ($\lambda_k^2 > 0$), so $\text{rank}(\rho_A) = r$.

**(4) Mixedness = complexity of entanglement**

Three quantities are equal at once:

$$\text{rank}(\rho_A) = \text{Schmidt rank} = d_B^{\min}$$

The rank determines the minimum auxiliary dimension needed for a purification; changing how evenly the eigenvalues are spread at a fixed rank can change the entropy, but not the Schmidt rank or the minimum auxiliary dimension.

This establishes a triangular correspondence: **the statistical complexity of the mixed state ↔ the quantum complexity of the entangled state ↔ the dimension of the minimal auxiliary system**.
:::
::::


**Core insights**:

1. **SVD = Schmidt decomposition**: the SVD of linear algebra is, in quantum mechanics, the Schmidt decomposition

2. **Singular values = a measure of entanglement**: the distribution of the singular values directly reflects the strength of entanglement

3. **Reduction = marginalization**: taking the partial trace amounts to "integrating out" a subsystem, corresponding to the average state of §10.4 (measuring and discarding the outcome)

4. **Purification = the inverse operation**: given a mixed state, purification constructs a "parent" pure state whose reduction is exactly the original mixed state; the unitary freedom on system B corresponds to the different ways of purifying

---

### 11.5.3 The Schmidt Structure of Multipartite Systems

#### Non-Uniqueness for Tripartite Entanglement

For a bipartite pure state, the Schmidt coefficients are uniquely determined, although the Schmidt bases need not be. Tripartite or larger systems differ in an essential way: given a tripartite pure state $|\psi⟩_{ABC}$, one can perform a Schmidt decomposition separately for different bipartitions—

$$|\psi⟩_{ABC} = \sum_i \lambda_i^{(1)} |\alpha_i⟩_A \otimes |\beta_i⟩_{BC} \quad (\text{A|BC bipartition}), \qquad |\psi⟩_{ABC} = \sum_j \lambda_j^{(2)} |\gamma_j⟩_{AB} \otimes |\delta_j⟩_C \quad (\text{AB|C bipartition})$$

The Schmidt coefficients for different bipartitions are in general different, but they must satisfy compatibility conditions on the reduced states; a single bipartition cannot fully describe multipartite entanglement.

#### The GHZ State and the W State

In the three-qubit state space there are two typical classes of genuinely tripartite entangled states:

$$|\text{GHZ}⟩ = \frac{1}{\sqrt{2}}(|000⟩ + |111⟩), \qquad |\text{W}⟩ = \frac{1}{\sqrt{3}}(|100⟩ + |010⟩ + |001⟩)$$

:::{prf:example} Schmidt Analysis of the GHZ State
:label: ex-ghz-schmidt

**The A|BC bipartition.** The coefficient matrix is

$$\mathbf{C}_{A|BC} = \frac{1}{\sqrt{2}}\begin{bmatrix}1&0&0&0\\0&0&0&1\end{bmatrix} \in \mathbb{C}^{2 \times 4}$$

This matrix has rank $2$, and the SVD gives directly

$$|\text{GHZ}⟩ = \frac{1}{\sqrt{2}}|0⟩_A \otimes |00⟩_{BC} + \frac{1}{\sqrt{2}}|1⟩_A \otimes |11⟩_{BC}$$

The Schmidt coefficients are $\lambda_0 = \lambda_1 = 1/\sqrt{2}$, and the von Neumann entropy is $S = 1$ (maximal entanglement).

**Reduced density matrices**:

$$\rho_A = \operatorname{Tr}_{BC}(|\text{GHZ}⟩⟨ \text{GHZ}|) = \frac{1}{2}\mathbf{I}_2$$

$$\rho_{AB} = \operatorname{Tr}_C(|\text{GHZ}⟩⟨ \text{GHZ}|) = \frac{1}{2}(|00⟩⟨ 00| + |11⟩⟨ 11|)$$

$\rho_{AB}$ is a mixed state with $S(\rho_{AB}) = 1$, so the AB subsystem described by $\rho_{AB}$ has no pure-state entanglement—the entanglement of the GHZ state is **global** and cannot be decomposed into a combination of bipartite entanglements.
:::

:::{prf:example} The Reduced State of the W State After Losing One Particle
:label: ex-w-partial-trace

Take the partial trace over system C. The component of $|\text{W}⟩$ with $C=0$ is $\frac{1}{\sqrt{3}}(|10⟩+|01⟩)$, and the component with $C=1$ is $\frac{1}{\sqrt{3}}|00⟩$, so

$$\rho_{AB} = \operatorname{Tr}_C(|\text{W}⟩⟨ \text{W}|) = \frac{1}{3}\bigl(|10⟩ + |01⟩\bigr)\bigl(⟨ 10| + ⟨ 01|\bigr) + \frac{1}{3}|00⟩⟨ 00| = \frac{2}{3}|\Psi^+⟩⟨ \Psi^+| + \frac{1}{3}|00⟩⟨ 00|$$

where $|\Psi^+⟩ = \frac{1}{\sqrt{2}}(|10⟩ + |01⟩)$ is a Bell state. The eigenvalues of $\rho_{AB}$ are $\{2/3,\ 1/3,\ 0,\ 0\}$—it is a **mixed state**, but it is **still entangled** (its concurrence is $2/3 > 0$, which the code at the end of this section verifies numerically). After one particle is lost, the remaining two parties of the W state still share usable entanglement; this is precisely the **robustness** of the W state.

This contrasts sharply with the GHZ state: taking the partial trace over C of the GHZ state gives $\rho_{AB} = \frac{1}{2}(|00⟩⟨ 00| + |11⟩⟨ 11|)$, a **separable** classical mixture, and the entanglement between A and B disappears completely with the loss of one particle.
:::

The difference between the GHZ state and the W state lies not only in the numerical results above; it also reflects the essential classification of tripartite entanglement.

#### SLOCC Equivalence Classes

**Definition** (SLOCC equivalence). If there exists an invertible local operator $\mathbf{A}_0 \otimes \mathbf{A}_1 \otimes \cdots \otimes \mathbf{A}_{n-1}$ such that

$$|\phi⟩ = (\mathbf{A}_0 \otimes \cdots \otimes \mathbf{A}_{n-1})|\psi⟩$$

then $|\psi⟩$ and $|\phi⟩$ are said to be equivalent under **stochastic local operations and classical communication** (SLOCC). SLOCC-equivalent states can be converted into each other in the sense of entanglement resources.

One can prove that GHZ and W do not belong to the same SLOCC equivalence class; that is, no invertible local operation converts one into the other. For $n$ qubits, the number of SLOCC equivalence classes grows with $n$ as follows: two classes for $n=2$, six classes for $n=3$, and already infinitely many for $n \geq 4$ (the infinitely many classes for $n=4$ can be organized into nine families).

#### Tensor Decompositions: Higher-Dimensional Generalizations of the Schmidt Decomposition

In the tripartite pure state

$$|\psi⟩_{ABC} = \sum_{i,j,k} c_{ijk}|i⟩_A|j⟩_B|k⟩_C$$

the coefficients $c_{ijk}$ form a third-order tensor $\mathcal{T} \in \mathbb{C}^{d_A \times d_B \times d_C}$. Two natural generalizations of the SVD are:

- **Tucker decomposition**: $\mathcal{T} \approx \mathcal{G} \times_0 \mathbf{U}_A \times_1 \mathbf{U}_B \times_2 \mathbf{U}_C$, where $\mathcal{G}$ is the core tensor and $\mathbf{U}_A, \mathbf{U}_B, \mathbf{U}_C$ are unitary matrices.
- **CP decomposition** (Canonical Polyadic): $\mathcal{T} = \sum_r \lambda_r \mathbf{a}_r \otimes \mathbf{b}_r \otimes \mathbf{c}_r$, a sum of pure tensor products of vectors.

Unlike the SVD of a matrix, computing the rank of a third-order tensor (the CP rank) is NP-hard, and a best low-rank approximation may not exist (the rank may not be attained). This is one of the roots of the difficulty of classifying entanglement in many-body quantum systems.

#### Entanglement as a Resource: The Operational Interpretation of the von Neumann Entropy

For a pure state $|\psi⟩_{AB}$, the von Neumann entropy $S(\rho_A)$ is not only a measure of entanglement; it also has a precise operational meaning:

:::{prf:theorem} Optimal Rate of Entanglement Distillation
:label: thm-entanglement-distillation

Let $|\psi⟩_{AB}$ have Schmidt decomposition $\sum_i \lambda_i |i⟩_A|i⟩_B$ and von Neumann entropy $S = -\sum_i \lambda_i^2 \log_2 \lambda_i^2$.

For $n$ copies $|\psi⟩^{\otimes n}$, a rate $R$ is called achievable if there exist LOCC protocols outputting $\omega_n$ and $m_n$ Bell pairs such that $\liminf m_n/n\geq R$ and $\|\omega_n-(|\Phi^+⟩⟨\Phi^+|)^{\otimes m_n}\|_1\to0$. In a heralded-success model, the success probability is also required to tend to 1. The supremum of all achievable rates is

$$E_D(|\psi⟩)=\sup\{R:R\text{ achievable}\}=S(\rho_A)$$

:::

This theorem is the quantum-information analogue of Shannon's theorem: the entanglement entropy is the "exchange rate" of entanglement as a resource. For pure states, the entanglement cost (the number of Bell states needed for preparation) and the distillable entanglement are equal, both being $S(\rho_A)$. For mixed states, entanglement conversion is in general irreversible, and the cost may exceed the distillable entanglement; among mixed states there are even **bound entangled states**, which are entangled but from which no entanglement can be distilled.

:::{prf:theorem} Schmidt Rank and Communication Complexity
:label: thm-schmidt-communication

Suppose Alice and Bob share no prior entanglement and, starting from a product initial state, prepare exactly a pure state $|\psi⟩_{AB}$ of Schmidt rank $r$. Then the total number of qubits that must be transmitted satisfies

$$Q \geq \log_2 r$$

Each transmitted qubit can at most double the Schmidt rank across the bipartition, which is why this lower bound holds. If prior shared entanglement is allowed, the resource model must be specified anew.
:::

Under the resource model above, the logarithm of the Schmidt rank gives a lower bound on the amount of quantum communication, describing the "intrinsic complexity" of a pure state.



In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

print_header("W State vs. GHZ State: Partial-Trace Check After Losing One Particle")

def partial_trace_C(psi):
    """Take Tr_C of a 3-qubit pure state |ψ⟩ (basis |abc⟩ in lexicographic order) and return the 4×4 ρ_AB"""
    rho = np.outer(psi, psi.conj())
    rho_AB = np.zeros((4, 4), dtype=complex)
    for ab in range(4):
        for ab2 in range(4):
            for c in range(2):
                rho_AB[ab, ab2] += rho[2 * ab + c, 2 * ab2 + c]
    return rho_AB

def concurrence_2qubit(rho):
    """Concurrence of a two-qubit mixed state (Wootters formula): C > 0 ⟺ entangled"""
    sy = np.array([[0, -1j], [1j, 0]])
    R = rho @ np.kron(sy, sy) @ rho.conj() @ np.kron(sy, sy)
    ev = np.sqrt(np.abs(np.sort(np.linalg.eigvals(R).real)[::-1]))
    return max(0.0, ev[0] - ev[1] - ev[2] - ev[3])

W = np.zeros(8)
W[[0b100, 0b010, 0b001]] = 1 / np.sqrt(3)
GHZ = np.zeros(8)
GHZ[[0b000, 0b111]] = 1 / np.sqrt(2)

print_step(1, "W state: spectrum and entanglement of ρ_AB")
rho_W = partial_trace_C(W)
print("  ρ_AB =\n", np.real_if_close(rho_W))   # display only: imaginary parts are dropped only within tolerance; ρ_AB itself is unchanged
print(f"  eigenvalues = {np.round(np.linalg.eigvalsh(rho_W)[::-1], 4)} (expected [2/3, 1/3, 0, 0])")
print(f"  Tr(ρ²) = {np.trace(rho_W @ rho_W).real:.4f} < 1  → mixed state")   # ρ is Hermitian, so Tr(ρ²) is real
print(f"  concurrence C(ρ_AB) = {concurrence_2qubit(rho_W.astype(complex)):.4f} (expected 2/3 > 0 → still entangled: the robustness of the W state)")

print_step(2, "GHZ state: spectrum and entanglement of ρ_AB")
rho_G = partial_trace_C(GHZ)
print("  ρ_AB =\n", np.real_if_close(rho_G))
print(f"  eigenvalues = {np.round(np.linalg.eigvalsh(rho_G)[::-1], 4)} (expected [1/2, 1/2, 0, 0])")
print(f"  concurrence C(ρ_AB) = {concurrence_2qubit(rho_G.astype(complex)):.4f} (expected 0 → the entanglement vanishes completely with the lost particle)")


:::{admonition} §11.5 Summary
:class: tip

This section answered one of the most central questions of quantum information theory: how can the entanglement structure of a composite quantum system be described precisely in mathematical language? The Schmidt decomposition theorem reveals that the coefficient matrix of a bipartite pure state $|\psi⟩_{AB}$ is exactly the stage for the SVD—the singular values become the Schmidt coefficients, the left and right singular vectors become the Schmidt bases of the two subsystems, and the Schmidt rank (the number of nonzero singular values) equals 1 if and only if the state is separable, that is, unentangled. This correspondence is not a metaphor but a literal identity: the SVD of linear algebra, in the context of quantum mechanics, is the Schmidt decomposition.

The strength of entanglement is measured precisely by the von Neumann entropy $S = -\sum_k \lambda_k^2 \log_2\lambda_k^2$: the more evenly the Schmidt coefficients are distributed, the larger the entropy and the stronger the entanglement—the Bell state attains the maximal entanglement entropy of 1 bit in a $2 \times 2$ system. Mixed states arise naturally through the partial trace: tracing out subsystem B of $|\psi⟩_{AB}$ gives the reduced density matrix $\rho_A$, whose eigenvalues are exactly the squares of the Schmidt coefficients; the purification theorem provides the construction in the reverse direction, showing that every mixed state is the marginal of some larger pure state. Tripartite and higher-order entanglement (the GHZ state, the W state) shows that the uniqueness of the Schmidt decomposition holds only for bipartitions, and the complete classification of multipartite entanglement remains an open problem to this day.

The entanglement distillation theorem gives the von Neumann entropy an operational meaning: it is the asymptotic rate at which Bell pairs can be extracted from a pure state, the "exchange rate" of quantum resource theory, deeply linking a purely algebraic quantity—the singular values—with quantum communication capacity.
:::


## 11.6 Chapter Summary

### Review of the Theoretical Thread

This chapter started from a natural and pressing question: when the spectral theorem for symmetric matrices meets the non-square data that are everywhere in the real world, can we keep the elegant geometric picture of "rotation—stretch—rotation"? §11.1 gave an affirmative answer. Through the bridge $\mathbf{A}^{*}\mathbf{A}$, every matrix $\mathbf{A} \in \mathbb{C}^{m \times n}$ has a decomposition $\mathbf{A} = \mathbf{U}\boldsymbol{\Sigma}\mathbf{V}^{*}$; the singular values uniquely describe the geometric structure by which the matrix stretches the unit sphere into an ellipsoid; the four fundamental subspaces each receive an orthonormal basis under the SVD; and the three viewpoints of algebra, subspaces, and geometry confirm one another, together laying the theoretical foundation of the whole chapter.

With singular values as "descriptors" in hand, §11.2 asked about their optimality: among all matrices of rank at most $k$, the one closest to $\mathbf{A}$ is exactly the truncated SVD $\mathbf{A}_k$, with the error given precisely by the discarded singular values. The Eckart–Young theorem is the theoretical crown of the chapter: it turns "best approximation," a problem that would otherwise require laborious optimization, into something read off directly from the SVD. The three matrix norms—the spectral norm, the Frobenius norm, and the nuclear norm—are all completely determined by the singular values, further highlighting the status of singular values as the essential invariants of a matrix.

This optimality immediately found its statistical incarnation in §11.3: principal component analysis (PCA) is essentially the truncated SVD of the centered data matrix; the right singular vectors give the directions of greatest variation in the data, and the squared singular values (after normalization) are the amounts of variation explained by the principal components. Taking the SVD of the data matrix directly, rather than finding the eigenvalues of the covariance matrix, avoids numerically the trap of squaring the condition number—a point textbooks seldom stress but one that is crucial in actual computation. The limitations of PCA—its helplessness in the face of nonlinear manifolds—also prompt readers to think about when they need to go beyond linear tools.

§11.4 turned "best approximation" from dimensionality reduction of data to the solution of equations: the Moore–Penrose pseudoinverse $\mathbf{A}^{+} = \mathbf{V}\boldsymbol{\Sigma}^{+}\mathbf{U}^{*}$, by taking reciprocals of the nonzero singular values, gracefully extends the concept of matrix inversion and gives the minimum-norm least-squares solution. The numerical reality that the reciprocals of small singular values amplify noise explosively led to ridge regression regularization: using the parameter $\lambda$ to smoothly suppress the contributions of small singular values, trading a controllable bias for the stability of the solution. The pseudoinverse, regularization, and singular value truncation are unified in the SVD framework as different choices along the same spectrum.

Finally, §11.5 carried the SVD into the territory of quantum physics: taking the SVD of the coefficient matrix of a bipartite pure state is precisely the Schmidt decomposition. The distribution of the Schmidt coefficients describes the entanglement structure precisely, the von Neumann entropy quantifies the strength of entanglement, and the entanglement distillation theorem gives the entropy an operational meaning. From condition numbers to entanglement entropy, the same mathematics of singular values speaks the same language in strikingly different fields.

### Connections to Other Chapters

This chapter follows directly from Chapter 9 (inner product spaces and symmetric matrices) and Chapter 10 (introduction to quantum mechanics). The spectral theorem of Chapter 9 is the cornerstone of the existence proof in §11.1: $\mathbf{A}^{*}\mathbf{A}$ is a Hermitian positive semidefinite matrix, and its orthogonal diagonalization guarantees the existence of the SVD; the language of positive definiteness established in Chapter 9 ($⟨ \mathbf{x}|\mathbf{A}^{*}\mathbf{A}|\mathbf{x}⟩ = \|\mathbf{A}\mathbf{x}\|^2 \geq 0$) is reused here directly. An earlier bridge comes from Chapter 5: the statistical context of covariance matrices and PCA is precisely activated in §11.3, and block-matrix thinking also runs through the account of how the three forms of the SVD (full, reduced, truncated) relate. The discussion of the solution structure of systems of linear equations in Chapter 6 finds its final, completed form in §11.4—the pseudoinverse unifies "generalized inversion" for the three cases of overdetermined, underdetermined, and rank-deficient systems.

The Dirac notation, tensor product spaces, and density matrices of Chapter 10 become the linguistic foundation of the Schmidt decomposition in §11.5; conversely, the Schmidt decomposition provides quantitative mathematical tools for the qualitative picture of entanglement described in Chapter 10—the entanglement entropy, the separability criterion, and the spectrum of the reduced density matrix all rely on the singular value structure of the SVD. Reading the two chapters side by side, the reader will see how linear algebra and quantum mechanics translate deeply into each other on the common platform of complex inner product spaces.

As the last chapter of the book, the SVD plays the role of the **terminal station**: it integrates all the main tools of the book—vector spaces, linear mappings, determinants, eigenvalues, inner products, positive definiteness, tensor products—and, over the broadest class of matrices, fulfills the core promise of linear algebra: "decompose in order to understand."

### The Role of This Chapter in the Book

The SVD solves an old and universal problem: given a linear mapping, how can we find an input basis and an output basis in which the mapping looks as "simple" as possible—diagonal, ordered, and geometrically intuitive? Symmetric matrices are only a special case of this problem; the SVD is its complete answer. It is both the end point of the theory and the starting point of computation: in numerical linear algebra, the SVD is the standard algorithm for the rank, the condition number, the pseudoinverse, and least squares, and almost every large-scale computation involving matrices ultimately comes back to the singular value decomposition.

In the broader scientific picture, the "correlation structure" revealed by the SVD is everywhere: spatial correlation in image compression, feature correlation in datasets, particle entanglement in quantum systems—mathematically they are all the same thing, the distribution of the singular values of a matrix (or tensor). After studying this chapter, whenever readers meet the words "dimensionality reduction," "regularization," or "entanglement measure," the image of singular values will come naturally to mind: a row of positive numbers arranged by size, each marking the "signal strength" in some direction, and truncating them means making a well-founded trade-off. This way of seeing is precisely the mark of linear algebra rising from a technical tool to a way of thinking.

### Concept Map
```{mermaid}
%%{init: {
  "theme": "base",
  "themeVariables": {
    "background":         "#FFFFFF",
    "primaryColor":       "#FFFFFF",
    "primaryTextColor":   "#57068C",
    "primaryBorderColor": "#57068C",
    "lineColor":          "#7F8C8D",
    "secondaryColor":     "#F1F2F6",
    "tertiaryColor":      "#FFFFFF",
    "clusterBkg":         "#F9F9FB",
    "clusterBorder":      "#DCDDE1",
    "fontSize":           "14px"
  }
}}%%
graph TD
    subgraph Sprev["<span style='color:#7F8C8D;font-weight:bold'>Foundations from Earlier Chapters</span>"]
        P1["Chapter 9<br/>spectral theorem and orthogonal diagonalization"]
        P2["Chapter 10<br/>quantum mechanics framework<br/>Dirac notation · tensor product"]
    end

    subgraph S1["<span style='color:#57068C;font-weight:bold'>§11.1 Existence of the SVD</span>"]
        N1["Hermitian positive<br/>semidefiniteness of A*A"]
        N2["Core SVD factorization<br/>A = UΣV*"]
        N3["Four fundamental subspaces<br/>orthonormal bases"]
        N4["Geometric interpretation<br/>rotation—stretch—rotation"]
    end

    subgraph S2["<span style='color:#57068C;font-weight:bold'>§11.2 The Eckart–Young Theorem</span>"]
        N5["Truncated SVD<br/>best low-rank approximation"]
        N6["Three matrix norms<br/>determined by singular values"]
    end

    subgraph S3["<span style='color:#006385;font-weight:bold'>§11.3 Principal Component Analysis</span>"]
        N7["PCA and the truncated SVD<br/>directions of greatest variation"]
    end

    subgraph S4["<span style='color:#006385;font-weight:bold'>§11.4 Pseudoinverse and Regularization</span>"]
        N8["Moore–Penrose pseudoinverse<br/>A⁺ = VΣ⁺U*"]
        N9["Ridge regression regularization<br/>suppressing small singular values"]
    end

    subgraph S5["<span style='color:#006385;font-weight:bold'>§11.5 Schmidt Decomposition and Quantum Entanglement</span>"]
        N10["Schmidt decomposition<br/>= SVD on quantum states"]
        N11["von Neumann entropy<br/>measure of entanglement strength"]
        N12["Classifying multipartite entanglement<br/>GHZ / W / SLOCC"]
    end

    P1 -.->|builds on| N1
    P2 -.->|builds on| N10
    N1 --> N2
    N2 --> N3
    N2 --> N4
    N2 --> N5
    N5 --> N6
    N5 --> N7
    N5 --> N8
    N8 --> N9
    N2 --> N10
    N10 --> N11
    N11 --> N12

    classDef main      fill:#F4EEF9,stroke:#57068C,stroke-width:2px,color:#57068C
    classDef structure fill:#E6F0F3,stroke:#006385,stroke-width:2px,color:#006385
    classDef accent    fill:#EAF9F8,stroke:#2AD2C9,stroke-width:2px,color:#087D77,font-weight:bold
    classDef neutral   fill:#FFFFFF,stroke:#DCDDE1,color:#2F3640
    classDef future    fill:#FFFFFF,stroke:#AB82C5,stroke-dasharray:5 5,color:#7F8C8D

    class P1 future
    class P2 future
    class N1 neutral
    class N2 main
    class N3 structure
    class N4 accent
    class N5 main
    class N6 neutral
    class N7 structure
    class N8 structure
    class N9 neutral
    class N10 main
    class N11 accent
    class N12 future
```